In [106]:
import importlib
import snowflake_loader

importlib.reload(snowflake_loader)

print("get_connection available:", hasattr(snowflake_loader, "get_connection"))

get_connection available: True


In [107]:
conn = snowflake_loader.get_connection()

print("Snowflake connection successful")

Snowflake connection successful


In [8]:
import pandas as pd

query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.CLEANED_HISTORICAL_YIELD
"""

historical_yield = pd.read_sql(query, conn)

print("Shape:", historical_yield.shape)
print("\nColumns:")
print(historical_yield.columns.tolist())

/tmp/ipykernel_75164/1842116511.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  historical_yield = pd.read_sql(query, conn)


Shape: (276, 7)

Columns:
['FIELD_ID', 'SEASON_YEAR', 'CROP_TYPE', 'ACTUAL_YIELD_BU_ACRE', 'HARVEST_DATE', 'YIELD_UNIT', 'YIELD_DATA_STATUS']


In [9]:
print("Rows:", len(historical_yield))

print("\nDuplicate FIELD_ID + SEASON_YEAR:")
duplicate_yield = (
    historical_yield
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .size()
    .reset_index(name="COUNT")
)

print(duplicate_yield[duplicate_yield["COUNT"] > 1])

print("\nMissing values:")
print(historical_yield.isna().sum())

print("\nSeason coverage:")
print(
    historical_yield["SEASON_YEAR"]
    .value_counts()
    .sort_index()
)

print("\nCrop coverage:")
print(
    historical_yield["CROP_TYPE"]
    .value_counts(dropna=False)
)

print("\nYield units:")
print(
    historical_yield["YIELD_UNIT"]
    .value_counts(dropna=False)
)

print("\nYield data status:")
print(
    historical_yield["YIELD_DATA_STATUS"]
    .value_counts(dropna=False)
)

Rows: 276

Duplicate FIELD_ID + SEASON_YEAR:
    FIELD_ID SEASON_YEAR  COUNT
12    FLD003        2021      3
17    FLD003        2026      3
18    FLD004        2021      2
19    FLD004        2022      2
20    FLD004        2023      2
21    FLD004        2024      2
22    FLD004        2025      2
23    FLD004        2026      2
30    FLD006        2022      2
31    FLD006        2023      2
32    FLD006        2024      2
33    FLD006        2025      2
70    FLD013        2021      2
71    FLD013        2022      2
72    FLD013        2023      2
73    FLD013        2024      2
74    FLD013        2025      2
75    FLD013        2026      2
166   FLD029        2021      2
167   FLD029        2022      2
168   FLD029        2023      2
169   FLD029        2024      2
170   FLD029        2025      2
171   FLD029        2026      2
184   FLD032        2021      2
185   FLD032        2022      2
186   FLD032        2023      2
187   FLD032        2024      2
188   FLD032        2025   

In [10]:
duplicate_keys = (
    historical_yield
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .size()
    .reset_index(name="COUNT")
)

duplicate_keys = duplicate_keys[duplicate_keys["COUNT"] > 1]

duplicate_records = historical_yield.merge(
    duplicate_keys[["FIELD_ID", "SEASON_YEAR"]],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

duplicate_records = duplicate_records.sort_values(
    ["FIELD_ID", "SEASON_YEAR", "HARVEST_DATE"]
)

print("Duplicate records shape:", duplicate_records.shape)
display(duplicate_records)

Duplicate records shape: (62, 7)


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS
0,FLD003,2021,Soybean,62,2021-09-19,bu/acre,Available
1,FLD003,2021,Soybean,62,2021-09-19,bu/acre,Available
2,FLD003,2021,Soybean,62,2021-09-19,bu/acre,Available
3,FLD003,2026,Soybean,66,2026-09-16,bu/acre,Available
4,FLD003,2026,Soybean,64,2026-09-16,bu/acre,Available
...,...,...,...,...,...,...,...
57,FLD032,2024,Corn,209,2024-10-07,bu/acre,Available
58,FLD032,2025,Corn,204,2025-10-07,bu/acre,Available
59,FLD032,2025,Corn,204,2025-10-07,bu/acre,Available
60,FLD032,2026,Corn,200,2026-10-10,bu/acre,Available


In [11]:
pd.set_option("display.max_rows", None)

print(duplicate_records.to_string(index=False))

FIELD_ID SEASON_YEAR CROP_TYPE ACTUAL_YIELD_BU_ACRE HARVEST_DATE YIELD_UNIT YIELD_DATA_STATUS
  FLD003        2021   Soybean                   62   2021-09-19    bu/acre         Available
  FLD003        2021   Soybean                   62   2021-09-19    bu/acre         Available
  FLD003        2021   Soybean                   62   2021-09-19    bu/acre         Available
  FLD003        2026   Soybean                   66   2026-09-16    bu/acre         Available
  FLD003        2026   Soybean                   64   2026-09-16    bu/acre         Available
  FLD003        2026   Soybean                   64   2026-09-16    bu/acre         Available
  FLD004        2021      Corn                  205   2021-10-01    bu/acre         Available
  FLD004        2021      Corn                  205   2021-10-01    bu/acre         Available
  FLD004        2022      Corn                  200   2022-10-01    bu/acre         Available
  FLD004        2022      Corn                  200   2022-1

In [12]:
duplicate_analysis = (
    historical_yield
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .agg(
        RECORD_COUNT=("ACTUAL_YIELD_BU_ACRE", "size"),
        UNIQUE_YIELD_VALUES=("ACTUAL_YIELD_BU_ACRE", "nunique"),
        UNIQUE_HARVEST_DATES=("HARVEST_DATE", "nunique"),
        UNIQUE_CROP_TYPES=("CROP_TYPE", "nunique")
    )
    .reset_index()
)

duplicate_analysis = duplicate_analysis[
    duplicate_analysis["RECORD_COUNT"] > 1
]

print("Duplicate groups:", len(duplicate_analysis))

print("\nExact duplicate groups:")
display(
    duplicate_analysis[
        (duplicate_analysis["UNIQUE_YIELD_VALUES"] == 1) &
        (duplicate_analysis["UNIQUE_HARVEST_DATES"] == 1) &
        (duplicate_analysis["UNIQUE_CROP_TYPES"] == 1)
    ]
)

print("\nConflicting duplicate groups:")
display(
    duplicate_analysis[
        (duplicate_analysis["UNIQUE_YIELD_VALUES"] > 1) |
        (duplicate_analysis["UNIQUE_HARVEST_DATES"] > 1) |
        (duplicate_analysis["UNIQUE_CROP_TYPES"] > 1)
    ]
)

Duplicate groups: 30

Exact duplicate groups:


,FIELD_ID,SEASON_YEAR,RECORD_COUNT,UNIQUE_YIELD_VALUES,UNIQUE_HARVEST_DATES,UNIQUE_CROP_TYPES
12,FLD003,2021,3,1,1,1
18,FLD004,2021,2,1,1,1
19,FLD004,2022,2,1,1,1
20,FLD004,2023,2,1,1,1
21,FLD004,2024,2,1,1,1
22,FLD004,2025,2,1,1,1
23,FLD004,2026,2,1,1,1
30,FLD006,2022,2,1,1,1
31,FLD006,2023,2,1,1,1
32,FLD006,2024,2,1,1,1



Conflicting duplicate groups:


,FIELD_ID,SEASON_YEAR,RECORD_COUNT,UNIQUE_YIELD_VALUES,UNIQUE_HARVEST_DATES,UNIQUE_CROP_TYPES
17,FLD003,2026,3,2,1,1


In [13]:
conflict_record = historical_yield[
    (historical_yield["FIELD_ID"] == "FLD003") &
    (historical_yield["SEASON_YEAR"] == 2026)
]

display(conflict_record)

,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS


In [14]:
print("Unique yield values:")
print(conflict_record["ACTUAL_YIELD_BU_ACRE"].unique())

print("\nYield value counts:")
print(conflict_record["ACTUAL_YIELD_BU_ACRE"].value_counts())

print("\nData status:")
print(conflict_record["YIELD_DATA_STATUS"].value_counts())

Unique yield values:
[]

Yield value counts:
Series([], Name: count, dtype: int64)

Data status:
Series([], Name: count, dtype: int64)


In [15]:
print("FIELD_ID dtype:", historical_yield["FIELD_ID"].dtype)
print("SEASON_YEAR dtype:", historical_yield["SEASON_YEAR"].dtype)

print("\nFLD003 records:")
display(
    historical_yield[
        historical_yield["FIELD_ID"].astype(str).str.strip().eq("FLD003")
    ]
)

FIELD_ID dtype: object
SEASON_YEAR dtype: object

FLD003 records:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS
12,FLD003,2021,Soybean,62,2021-09-19,bu/acre,Available
13,FLD003,2022,Soybean,63,2022-09-19,bu/acre,Available
14,FLD003,2023,Soybean,64,2023-10-01,bu/acre,Available
15,FLD003,2024,Soybean,66,2024-10-01,bu/acre,Available
16,FLD003,2025,Soybean,62.5,2025-10-01,bu/acre,Available
17,FLD003,2026,Soybean,66,2026-09-16,bu/acre,Available
30,FLD003,2021,Soybean,62,2021-09-19,bu/acre,Available
35,FLD003,2026,Soybean,64,2026-09-16,bu/acre,Available
240,FLD003,2021,Soybean,62,2021-09-19,bu/acre,Available
245,FLD003,2026,Soybean,64,2026-09-16,bu/acre,Available


In [16]:
yield_profile = historical_yield.copy()

# Normalize key fields for analysis
yield_profile["FIELD_ID"] = (
    yield_profile["FIELD_ID"]
    .astype("string")
    .str.strip()
)

yield_profile["SEASON_YEAR"] = (
    yield_profile["SEASON_YEAR"]
    .astype("string")
    .str.strip()
)

yield_profile["CROP_TYPE"] = (
    yield_profile["CROP_TYPE"]
    .astype("string")
    .str.strip()
)

yield_profile["HARVEST_DATE"] = pd.to_datetime(
    yield_profile["HARVEST_DATE"],
    errors="coerce"
)

print("Profiling copy created.")
print("Shape:", yield_profile.shape)
print("SEASON_YEAR dtype:", yield_profile["SEASON_YEAR"].dtype)
print("HARVEST_DATE dtype:", yield_profile["HARVEST_DATE"].dtype)

Profiling copy created.
Shape: (276, 7)
SEASON_YEAR dtype: string
HARVEST_DATE dtype: datetime64[ns]


/tmp/ipykernel_75164/2845969345.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  yield_profile["HARVEST_DATE"] = pd.to_datetime(


In [17]:
yield_dedup = yield_profile.drop_duplicates().copy()

print("Original rows:", len(yield_profile))
print("Rows after exact deduplication:", len(yield_dedup))
print("Exact duplicate rows removed:", len(yield_profile) - len(yield_dedup))

Original rows: 276
Rows after exact deduplication: 245
Exact duplicate rows removed: 31


In [18]:
remaining_duplicates = (
    yield_dedup
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .size()
    .reset_index(name="COUNT")
)

remaining_duplicates = remaining_duplicates[
    remaining_duplicates["COUNT"] > 1
]

print("Remaining duplicate FIELD_ID + SEASON_YEAR groups:")
display(remaining_duplicates)

Remaining duplicate FIELD_ID + SEASON_YEAR groups:


,FIELD_ID,SEASON_YEAR,COUNT
17,FLD003,2026,2


In [19]:
yield_conflict_check = (
    yield_dedup
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .agg(
        RECORD_COUNT=("ACTUAL_YIELD_BU_ACRE", "size"),
        UNIQUE_YIELD_VALUES=("ACTUAL_YIELD_BU_ACRE", "nunique")
    )
    .reset_index()
)

yield_conflict_check["YIELD_CONFLICT_FLAG"] = (
    yield_conflict_check["UNIQUE_YIELD_VALUES"] > 1
)

yield_conflicts = yield_conflict_check[
    yield_conflict_check["YIELD_CONFLICT_FLAG"]
]

print("Unresolved yield conflicts:")
display(yield_conflicts)

Unresolved yield conflicts:


,FIELD_ID,SEASON_YEAR,RECORD_COUNT,UNIQUE_YIELD_VALUES,YIELD_CONFLICT_FLAG
17,FLD003,2026,2,2,True


In [20]:
yield_conflict_check = (
    yield_dedup
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .agg(
        RECORD_COUNT=("ACTUAL_YIELD_BU_ACRE", "size"),
        UNIQUE_YIELD_VALUES=("ACTUAL_YIELD_BU_ACRE", "nunique")
    )
    .reset_index()
)

yield_conflict_check["YIELD_CONFLICT_FLAG"] = (
    yield_conflict_check["UNIQUE_YIELD_VALUES"] > 1
)

yield_conflicts = yield_conflict_check[
    yield_conflict_check["YIELD_CONFLICT_FLAG"]
]

print("Unresolved yield conflicts:")
display(yield_conflicts)

Unresolved yield conflicts:


,FIELD_ID,SEASON_YEAR,RECORD_COUNT,UNIQUE_YIELD_VALUES,YIELD_CONFLICT_FLAG
17,FLD003,2026,2,2,True


In [21]:
print("Number of unresolved yield conflicts:", len(yield_conflicts))

Number of unresolved yield conflicts: 1


In [22]:
missing_field_id = yield_dedup[
    yield_dedup["FIELD_ID"].isna()
]

print("Rows with missing FIELD_ID:", len(missing_field_id))

display(missing_field_id)

Rows with missing FIELD_ID: 6


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS
246,<NA>,2021,Corn,205,2021-10-24,bu/acre,Available
247,<NA>,2022,Corn,200,2022-10-24,bu/acre,Available
248,<NA>,2023,Corn,201,2023-10-01,bu/acre,Available
249,<NA>,2024,Corn,213,2024-10-01,bu/acre,Available
250,<NA>,2025,Corn,210,2025-10-01,bu/acre,Available
251,<NA>,2026,Corn,219,2026-09-16,bu/acre,Available


In [23]:
missing_field_id = yield_dedup[
    yield_dedup["FIELD_ID"].isna()
].copy()

print("Rows with missing FIELD_ID:", len(missing_field_id))

display(missing_field_id)

Rows with missing FIELD_ID: 6


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS
246,<NA>,2021,Corn,205,2021-10-24,bu/acre,Available
247,<NA>,2022,Corn,200,2022-10-24,bu/acre,Available
248,<NA>,2023,Corn,201,2023-10-01,bu/acre,Available
249,<NA>,2024,Corn,213,2024-10-01,bu/acre,Available
250,<NA>,2025,Corn,210,2025-10-01,bu/acre,Available
251,<NA>,2026,Corn,219,2026-09-16,bu/acre,Available


In [24]:
query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
LIMIT 10
"""

field_context_sample = pd.read_sql(query, conn)

print("Shape:", field_context_sample.shape)
print("Columns:")
print(field_context_sample.columns.tolist())

display(field_context_sample)

/tmp/ipykernel_75164/3372011488.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  field_context_sample = pd.read_sql(query, conn)


Shape: (10, 13)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'FARM_NAME', 'CROP_TYPE', 'CROP_VARIETY', 'ACREAGE', 'COUNTY_STATE', 'SOIL_TYPE', 'PLANTING_DATE', 'HARVEST_DATE', 'WINTER_SEVERITY', 'HISTORICAL_YIELD_BASELINE', 'SEASON_START_DATE']


,FIELD_ID,SEASON_YEAR,FARM_NAME,CROP_TYPE,CROP_VARIETY,ACREAGE,COUNTY_STATE,SOIL_TYPE,PLANTING_DATE,HARVEST_DATE,WINTER_SEVERITY,HISTORICAL_YIELD_BASELINE,SEASON_START_DATE
0,FLD001,2027,None,Soybean,"Soybean, MG 2.1",102.0,"Dane County, WI",None,2027-05-01,2027-09-18,Extreme,None,2027-05-01
1,FLD002,2026,Fry Farms,Corn,"Corn, 104-day RM",158.0,"Fulton County, OH",Loam,2026-04-27,2026-10-14,Mild,None,2026-04-27
2,FLD003,2026,Stafford Farms,Soybean,"Soybean, MG 3.2",103.0,"Champaign County, IL",Sandy Loam,2026-05-10,2026-09-27,Mild,None,2026-05-10
3,FLD004,2026,Miller Farms,Corn,"Corn, 105-day RM",165.0,"Story County, IA",Silty Clay Loam,2026-04-25,2026-10-10,Mild,None,2026-04-25
4,FLD005,2026,Vargas Farms,Corn,"Corn, 105-day RM",245.0,"Fulton County, OH",Clay Loam,2026-04-29,2026-10-13,Severe,None,2026-04-29
5,FLD006,2026,Henderson Farms,Soybean,"Soybean, MG 2.8",103.0,"Webster County, IA",Sandy Loam,2026-05-06,2026-10-06,Normal,None,2026-05-06
6,FLD007,2026,Moreno Farms,Corn,"Corn, 108-day RM",143.0,"McLean County, IL",Silt Loam,2026-05-02,2026-10-19,Normal,None,2026-05-02
7,FLD008,2026,None,Corn,"Corn, 111-day RM",118.0,"Webster County, IA",Silty Clay Loam,2026-05-03,2026-10-22,Normal,None,2026-05-03
8,FLD009,2026,Lee Farms,Soybean,"Soybean, MG 1.7",229.0,"Dane County, WI",Silty Clay Loam,2026-05-16,2026-09-20,Severe,None,2026-05-16
9,FLD010,2026,Wallace Farms,Soybean,"Soybean, MG 3.4",171.0,"Clinton County, IN",Loam,2026-05-06,2026-09-23,Mild,None,2026-05-06


In [25]:
# Step 16: Match historical yield records with missing FIELD_ID
# using FIELD_SEASON_CONTEXT

missing_field_yield = yield_dedup[yield_dedup["FIELD_ID"].isna()].copy()

print("Missing FIELD_ID rows:")
display(missing_field_yield[
    ["SEASON_YEAR", "CROP_TYPE", "ACTUAL_YIELD_BU_ACRE", "HARVEST_DATE"]
])

Missing FIELD_ID rows:


,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE
246,2021,Corn,205,2021-10-24
247,2022,Corn,200,2022-10-24
248,2023,Corn,201,2023-10-01
249,2024,Corn,213,2024-10-01
250,2025,Corn,210,2025-10-01
251,2026,Corn,219,2026-09-16


In [26]:
# Step 17: Match missing FIELD_ID yield records
# using SEASON_YEAR + CROP_TYPE + HARVEST_DATE

missing_field_yield = yield_dedup[
    yield_dedup["FIELD_ID"].isna()
].copy()

# Make sure matching columns have consistent types
missing_field_yield["SEASON_YEAR"] = (
    missing_field_yield["SEASON_YEAR"].astype(str).str.strip()
)
missing_field_yield["CROP_TYPE"] = (
    missing_field_yield["CROP_TYPE"].astype(str).str.strip()
)
missing_field_yield["HARVEST_DATE"] = pd.to_datetime(
    missing_field_yield["HARVEST_DATE"]
)

field_context["SEASON_YEAR"] = (
    field_context["SEASON_YEAR"].astype(str).str.strip()
)
field_context["CROP_TYPE"] = (
    field_context["CROP_TYPE"].astype(str).str.strip()
)
field_context["HARVEST_DATE"] = pd.to_datetime(
    field_context["HARVEST_DATE"]
)

# Match on the strongest available identifiers
matches = missing_field_yield.merge(
    field_context[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            "HARVEST_DATE"
        ]
    ],
    on=["SEASON_YEAR", "CROP_TYPE", "HARVEST_DATE"],
    how="left"
)

print("Matching results:")
display(matches)

NameError: name 'field_context' is not defined

In [27]:
# Step 17A: Load the complete FIELD_SEASON_CONTEXT table

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    CROP_TYPE,
    HARVEST_DATE
FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
"""

field_context = pd.read_sql(query, conn)

print("Shape:", field_context.shape)
print("Columns:", field_context.columns.tolist())

display(field_context.head())

Shape: (101, 4)
Columns: ['FIELD_ID', 'SEASON_YEAR', 'CROP_TYPE', 'HARVEST_DATE']


/tmp/ipykernel_75164/2884192168.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  field_context = pd.read_sql(query, conn)


,FIELD_ID,SEASON_YEAR,CROP_TYPE,HARVEST_DATE
0,FLD001,2027,Soybean,2027-09-18
1,FLD002,2026,Corn,2026-10-14
2,FLD003,2026,Soybean,2026-09-27
3,FLD004,2026,Corn,2026-10-10
4,FLD005,2026,Corn,2026-10-13


In [29]:
# Step 17B: Match missing FIELD_ID records

missing_field_yield = yield_dedup[
    yield_dedup["FIELD_ID"].isna()
].copy()

# Normalize matching columns
missing_field_yield["SEASON_YEAR"] = (
    missing_field_yield["SEASON_YEAR"].astype(str).str.strip()
)

missing_field_yield["CROP_TYPE"] = (
    missing_field_yield["CROP_TYPE"].astype(str).str.strip()
)

missing_field_yield["HARVEST_DATE"] = pd.to_datetime(
    missing_field_yield["HARVEST_DATE"]
)

field_context["SEASON_YEAR"] = (
    field_context["SEASON_YEAR"].astype(str).str.strip()
)

field_context["CROP_TYPE"] = (
    field_context["CROP_TYPE"].astype(str).str.strip()
)

field_context["HARVEST_DATE"] = pd.to_datetime(
    field_context["HARVEST_DATE"]
)

# Rename FIELD_ID from context so there is no column-name conflict
context_for_matching = field_context[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "CROP_TYPE",
        "HARVEST_DATE"
    ]
].rename(
    columns={"FIELD_ID": "MATCHED_FIELD_ID"}
)

# Perform the match
matches = missing_field_yield.merge(
    context_for_matching,
    on=["SEASON_YEAR", "CROP_TYPE", "HARVEST_DATE"],
    how="left"
)

print("Matching results:")

display(
    matches[
        [
            "SEASON_YEAR",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "HARVEST_DATE",
            "MATCHED_FIELD_ID"
        ]
    ]
)

Matching results:


,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,MATCHED_FIELD_ID
0,2021,Corn,205,2021-10-24,NaN
1,2022,Corn,200,2022-10-24,NaN
2,2023,Corn,201,2023-10-01,NaN
3,2024,Corn,213,2024-10-01,NaN
4,2025,Corn,210,2025-10-01,NaN
5,2026,Corn,219,2026-09-16,NaN


In [30]:
# Step 18: Inspect all available attributes for unresolved yield records

display(
    missing_field_yield[
        [
            "SEASON_YEAR",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "HARVEST_DATE",
            "YIELD_UNIT",
            "YIELD_DATA_STATUS"
        ]
    ]
)


,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS
246,2021,Corn,205,2021-10-24,bu/acre,Available
247,2022,Corn,200,2022-10-24,bu/acre,Available
248,2023,Corn,201,2023-10-01,bu/acre,Available
249,2024,Corn,213,2024-10-01,bu/acre,Available
250,2025,Corn,210,2025-10-01,bu/acre,Available
251,2026,Corn,219,2026-09-16,bu/acre,Available


In [31]:
print("FIELD_SEASON_CONTEXT columns:")
print(field_context.columns.tolist())

FIELD_SEASON_CONTEXT columns:
['FIELD_ID', 'SEASON_YEAR', 'CROP_TYPE', 'HARVEST_DATE']


In [32]:
# Step 19: Final historical yield coverage after exact deduplication

print("Original rows:", len(historical_yield))
print("Rows after exact deduplication:", len(yield_dedup))

print("\nMissing FIELD_ID:", yield_dedup["FIELD_ID"].isna().sum())

print("\nUnique field-season combinations:")
print(
    yield_dedup[
        ["FIELD_ID", "SEASON_YEAR"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\nDuplicate FIELD_ID + SEASON_YEAR groups remaining:")
duplicate_groups = (
    yield_dedup
    .groupby(["FIELD_ID", "SEASON_YEAR"], dropna=False)
    .size()
    .reset_index(name="COUNT")
)

display(
    duplicate_groups[
        duplicate_groups["COUNT"] > 1
    ]
)

print("\nYield availability:")
print(
    yield_dedup["YIELD_DATA_STATUS"]
    .value_counts(dropna=False)
)

Original rows: 276
Rows after exact deduplication: 245

Missing FIELD_ID: 6

Unique field-season combinations:
244

Duplicate FIELD_ID + SEASON_YEAR groups remaining:


,FIELD_ID,SEASON_YEAR,COUNT
17,FLD003,2026,2



Yield availability:
YIELD_DATA_STATUS
Available                           179
Missing - yield data unavailable     66
Name: count, dtype: int64


In [34]:
import numpy as np

In [35]:
# Step 20: Create the working historical-yield target table
# without fabricating or resolving ambiguous records

historical_yield_target = yield_dedup.copy()

# Flag records with missing FIELD_ID
historical_yield_target["FIELD_ID_RESOLUTION_STATUS"] = np.where(
    historical_yield_target["FIELD_ID"].isna(),
    "UNRESOLVED_MISSING_FIELD_ID",
    "RESOLVED"
)

# Flag the known conflicting FLD003 / 2026 target
historical_yield_target["YIELD_CONFLICT_FLAG"] = (
    historical_yield_target["FIELD_ID"].eq("FLD003")
    & historical_yield_target["SEASON_YEAR"].astype(str).eq("2026")
)

print("Shape:", historical_yield_target.shape)

print("\nFIELD_ID resolution status:")
print(
    historical_yield_target["FIELD_ID_RESOLUTION_STATUS"]
    .value_counts()
)

print("\nYield conflict records:")
display(
    historical_yield_target[
        historical_yield_target["YIELD_CONFLICT_FLAG"]
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "HARVEST_DATE",
            "YIELD_CONFLICT_FLAG"
        ]
    ]
)

Shape: (245, 9)

FIELD_ID resolution status:
FIELD_ID_RESOLUTION_STATUS
RESOLVED                       239
UNRESOLVED_MISSING_FIELD_ID      6
Name: count, dtype: int64

Yield conflict records:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_CONFLICT_FLAG
17,FLD003,2026,Soybean,66,2026-09-16,True
35,FLD003,2026,Soybean,64,2026-09-16,True


In [36]:
# Step 21: Inspect the complete FLD003 / 2026 yield conflict

fld003_2026 = historical_yield[
    (historical_yield["FIELD_ID"].astype(str).str.strip() == "FLD003")
    & (historical_yield["SEASON_YEAR"].astype(str).str.strip() == "2026")
].copy()

print("FLD003 / 2026 source records:", len(fld003_2026))

display(fld003_2026)

FLD003 / 2026 source records: 3


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,YIELD_UNIT,YIELD_DATA_STATUS
17,FLD003,2026,Soybean,66,2026-09-16,bu/acre,Available
35,FLD003,2026,Soybean,64,2026-09-16,bu/acre,Available
245,FLD003,2026,Soybean,64,2026-09-16,bu/acre,Available


In [37]:
# Step 22: Mark unresolved historical-yield conflicts

historical_yield_target = yield_dedup.copy()

# Missing FIELD_ID status
historical_yield_target["FIELD_ID_RESOLUTION_STATUS"] = np.where(
    historical_yield_target["FIELD_ID"].isna(),
    "UNRESOLVED_MISSING_FIELD_ID",
    "RESOLVED"
)

# Identify conflicting FLD003 / 2026 records
historical_yield_target["YIELD_CONFLICT_FLAG"] = (
    historical_yield_target["FIELD_ID"].eq("FLD003")
    & historical_yield_target["SEASON_YEAR"].astype(str).eq("2026")
)

# Identify records that should NOT be used as a definitive target
historical_yield_target["YIELD_TARGET_STATUS"] = np.where(
    historical_yield_target["YIELD_CONFLICT_FLAG"],
    "UNRESOLVED_CONFLICT",
    np.where(
        historical_yield_target["FIELD_ID_RESOLUTION_STATUS"]
        == "UNRESOLVED_MISSING_FIELD_ID",
        "UNRESOLVED_MISSING_FIELD_ID",
        "VALID"
    )
)

print("Yield target status:")
print(
    historical_yield_target["YIELD_TARGET_STATUS"]
    .value_counts()
)

print("\nUnresolved records:")
display(
    historical_yield_target[
        historical_yield_target["YIELD_TARGET_STATUS"] != "VALID"
    ]
)

TypeError: boolean value of NA is ambiguous

In [38]:
# Step 22: Mark unresolved historical-yield conflicts

historical_yield_target = yield_dedup.copy()

# Missing FIELD_ID status
historical_yield_target["FIELD_ID_RESOLUTION_STATUS"] = np.where(
    historical_yield_target["FIELD_ID"].isna(),
    "UNRESOLVED_MISSING_FIELD_ID",
    "RESOLVED"
)

# Identify FLD003 / 2026 conflict safely
historical_yield_target["YIELD_CONFLICT_FLAG"] = (
    historical_yield_target["FIELD_ID"].eq("FLD003")
    & historical_yield_target["SEASON_YEAR"].astype(str).eq("2026")
).fillna(False)

# Assign target status
historical_yield_target["YIELD_TARGET_STATUS"] = np.select(
    [
        historical_yield_target["YIELD_CONFLICT_FLAG"],
        historical_yield_target["FIELD_ID_RESOLUTION_STATUS"]
            .eq("UNRESOLVED_MISSING_FIELD_ID")
    ],
    [
        "UNRESOLVED_CONFLICT",
        "UNRESOLVED_MISSING_FIELD_ID"
    ],
    default="VALID"
)

print("Yield target status:")
print(
    historical_yield_target["YIELD_TARGET_STATUS"]
    .value_counts()
)

print("\nUnresolved records:")
display(
    historical_yield_target[
        historical_yield_target["YIELD_TARGET_STATUS"] != "VALID"
    ]
)


TypeError: invalid entry 0 in condlist: should be boolean ndarray

In [39]:
# Step 22: Mark unresolved historical-yield conflicts

historical_yield_target = yield_dedup.copy()

# 1. FIELD_ID resolution status
historical_yield_target["FIELD_ID_RESOLUTION_STATUS"] = (
    "RESOLVED"
)

historical_yield_target.loc[
    historical_yield_target["FIELD_ID"].isna(),
    "FIELD_ID_RESOLUTION_STATUS"
] = "UNRESOLVED_MISSING_FIELD_ID"


# 2. Identify the known FLD003 / 2026 conflict
historical_yield_target["YIELD_CONFLICT_FLAG"] = False

conflict_mask = (
    historical_yield_target["FIELD_ID"].fillna("").astype(str).str.strip().eq("FLD003")
    & historical_yield_target["SEASON_YEAR"].fillna("").astype(str).str.strip().eq("2026")
)

historical_yield_target.loc[
    conflict_mask,
    "YIELD_CONFLICT_FLAG"
] = True


# 3. Default every record to VALID
historical_yield_target["YIELD_TARGET_STATUS"] = "VALID"

# Mark missing FIELD_ID records
historical_yield_target.loc[
    historical_yield_target["FIELD_ID_RESOLUTION_STATUS"]
    == "UNRESOLVED_MISSING_FIELD_ID",
    "YIELD_TARGET_STATUS"
] = "UNRESOLVED_MISSING_FIELD_ID"

# Mark conflicting FLD003 / 2026 records
historical_yield_target.loc[
    historical_yield_target["YIELD_CONFLICT_FLAG"],
    "YIELD_TARGET_STATUS"
] = "UNRESOLVED_CONFLICT"


# 4. Display results
print("Yield target status:")
print(
    historical_yield_target["YIELD_TARGET_STATUS"].value_counts()
)

print("\nUnresolved records:")
display(
    historical_yield_target[
        historical_yield_target["YIELD_TARGET_STATUS"] != "VALID"
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "HARVEST_DATE",
            "FIELD_ID_RESOLUTION_STATUS",
            "YIELD_CONFLICT_FLAG",
            "YIELD_TARGET_STATUS"
        ]
    ]
)

Yield target status:
YIELD_TARGET_STATUS
VALID                          237
UNRESOLVED_MISSING_FIELD_ID      6
UNRESOLVED_CONFLICT              2
Name: count, dtype: int64

Unresolved records:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE,FIELD_ID_RESOLUTION_STATUS,YIELD_CONFLICT_FLAG,YIELD_TARGET_STATUS
17,FLD003,2026,Soybean,66,2026-09-16,RESOLVED,True,UNRESOLVED_CONFLICT
35,FLD003,2026,Soybean,64,2026-09-16,RESOLVED,True,UNRESOLVED_CONFLICT
246,<NA>,2021,Corn,205,2021-10-24,UNRESOLVED_MISSING_FIELD_ID,False,UNRESOLVED_MISSING_FIELD_ID
247,<NA>,2022,Corn,200,2022-10-24,UNRESOLVED_MISSING_FIELD_ID,False,UNRESOLVED_MISSING_FIELD_ID
248,<NA>,2023,Corn,201,2023-10-01,UNRESOLVED_MISSING_FIELD_ID,False,UNRESOLVED_MISSING_FIELD_ID
249,<NA>,2024,Corn,213,2024-10-01,UNRESOLVED_MISSING_FIELD_ID,False,UNRESOLVED_MISSING_FIELD_ID
250,<NA>,2025,Corn,210,2025-10-01,UNRESOLVED_MISSING_FIELD_ID,False,UNRESOLVED_MISSING_FIELD_ID
251,<NA>,2026,Corn,219,2026-09-16,UNRESOLVED_MISSING_FIELD_ID,False,UNRESOLVED_MISSING_FIELD_ID


In [40]:
# Step 23: Inspect available seasonal weeks
# from the Season Intelligence input

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    WEEK_INDEX
FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_INTELLIGENCE_INPUT
ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
"""

season_weeks = pd.read_sql(query, conn)

print("Shape:", season_weeks.shape)

print("\nAvailable WEEK_INDEX values:")
print(
    sorted(
        season_weeks["WEEK_INDEX"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nCheckpoint availability:")

for week in [4, 8, 12]:
    count = (
        season_weeks["WEEK_INDEX"] == week
    ).sum()

    print(f"Week {week}: {count} rows")

/tmp/ipykernel_75164/60902614.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_weeks = pd.read_sql(query, conn)


Shape: (111, 3)

Available WEEK_INDEX values:
[1, 3, 5, 7, 9, 11]

Checkpoint availability:
Week 4: 0 rows
Week 8: 0 rows
Week 12: 0 rows


In [41]:
# Step 24: Inspect the seasonal week structure in FIELD_SEASON_CONTEXT

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    SEASON_START_DATE,
    PLANTING_DATE,
    HARVEST_DATE
FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
ORDER BY FIELD_ID, SEASON_YEAR
"""

field_context_full = pd.read_sql(query, conn)

print("Shape:", field_context_full.shape)

display(field_context_full.head(10))

/tmp/ipykernel_75164/1992923801.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  field_context_full = pd.read_sql(query, conn)


Shape: (101, 5)


,FIELD_ID,SEASON_YEAR,SEASON_START_DATE,PLANTING_DATE,HARVEST_DATE
0,FLD001,2027,2027-05-01,2027-05-01,2027-09-18
1,FLD002,2026,2026-04-27,2026-04-27,2026-10-14
2,FLD003,2026,2026-05-10,2026-05-10,2026-09-27
3,FLD004,2023,2023-04-20,2023-04-20,2023-10-04
4,FLD004,2024,2024-05-01,2024-05-01,2024-10-04
5,FLD004,2025,2025-04-25,2025-04-25,2025-10-04
6,FLD004,2026,2026-04-25,2026-04-25,2026-10-10
7,FLD005,2026,2026-04-29,2026-04-29,2026-10-13
8,FLD006,2026,2026-05-06,2026-05-06,2026-10-06
9,FLD007,2023,2023-04-27,2023-04-27,2023-10-04


In [42]:
# Step 25: Inspect the weekly grain of Weather features

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    WEEK_INDEX
FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
"""

weather_weeks = pd.read_sql(query, conn)

print("Shape:", weather_weeks.shape)

print("\nAvailable WEEK_INDEX values:")
print(
    sorted(
        weather_weeks["WEEK_INDEX"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nWeek counts:")
print(
    weather_weeks["WEEK_INDEX"]
    .value_counts()
    .sort_index()
)

/tmp/ipykernel_75164/1469804826.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_weeks = pd.read_sql(query, conn)


DatabaseError: Execution failed on sql '
SELECT
    FIELD_ID,
    SEASON_YEAR,
    WEEK_INDEX
FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
': 000904 (42000): SQL compilation error: error line 4 at position 4
invalid identifier 'WEEK_INDEX'

In [43]:
# Step 25A: Inspect Weather Feature Snapshot schema

query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
LIMIT 5
"""

weather_sample = pd.read_sql(query, conn)

print("Shape:", weather_sample.shape)
print("\nColumns:")
print(weather_sample.columns.tolist())

display(weather_sample)

/tmp/ipykernel_75164/2442587959.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_sample = pd.read_sql(query, conn)


Shape: (5, 20)

Columns:
['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE', 'MEAN_TEMPERATURE', 'TEMPERATURE_7D', 'TEMPERATURE_14D', 'TEMPERATURE_DEVIATION', 'RAINFALL_7D', 'RAINFALL_14D', 'RAINFALL_DEVIATION', 'HUMIDITY_7D', 'HUMIDITY_14D', 'HUMIDITY_DEVIATION', 'LEAFWETNESS_7D', 'LEAFWETNESS_DEVIATION', 'CUMULATIVE_GDD', 'HISTORICAL_CUMULATIVE_GDD', 'GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'WEATHER_DATA_AGE']


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,MEAN_TEMPERATURE,TEMPERATURE_7D,TEMPERATURE_14D,TEMPERATURE_DEVIATION,RAINFALL_7D,RAINFALL_14D,RAINFALL_DEVIATION,HUMIDITY_7D,HUMIDITY_14D,HUMIDITY_DEVIATION,LEAFWETNESS_7D,LEAFWETNESS_DEVIATION,CUMULATIVE_GDD,HISTORICAL_CUMULATIVE_GDD,GDD_DEVIATION,GDD_DEVIATION_PCT,WEATHER_DATA_AGE
0,FLD001,2026,2026-05-04,62.5,62.5,62.50,-4.685714,0.68,0.68,-0.441667,55.9,55.90,-3.7,8.7,1.44,87.8,112.85,-25.05,-22.197607,84
1,FLD001,2026,2026-05-11,63.9,63.9,63.20,0.000000,0.00,0.34,0.000000,65.1,60.50,0.0,3.6,0.00,185.1,105.65,79.45,75.201136,77
2,FLD001,2026,2026-05-18,64.2,64.2,64.05,0.000000,0.00,0.00,0.000000,58.8,61.95,0.0,3.2,0.00,284.2,201.05,83.15,41.357871,70
3,FLD001,2026,2026-05-25,68.0,68.0,66.10,0.000000,2.34,1.17,0.000000,58.9,58.85,0.0,7.5,0.00,410.2,320.20,90.00,28.107433,63
4,FLD001,2026,2026-06-01,66.7,66.7,67.35,0.000000,1.78,2.06,0.000000,52.9,55.90,0.0,6.4,0.00,527.1,436.95,90.15,20.631651,56


In [44]:
# Step 25B: Derive WEEK_INDEX from SNAPSHOT_DATE
# using SEASON_START_DATE from FIELD_SEASON_CONTEXT

weather_check = weather_sample.copy()

# Prepare dates
weather_check["SNAPSHOT_DATE"] = pd.to_datetime(
    weather_check["SNAPSHOT_DATE"]
)

field_context_full["SEASON_START_DATE"] = pd.to_datetime(
    field_context_full["SEASON_START_DATE"]
)

# Add season start date
weather_check = weather_check.merge(
    field_context_full[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SEASON_START_DATE"
        ]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

# Calculate days from season start
weather_check["DAP_CALCULATED"] = (
    weather_check["SNAPSHOT_DATE"]
    - weather_check["SEASON_START_DATE"]
).dt.days

# Convert DAP to seasonal week
weather_check["WEEK_INDEX_CALCULATED"] = (
    weather_check["DAP_CALCULATED"] // 7
) + 1

print("Derived seasonal weeks:")

display(
    weather_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "DAP_CALCULATED",
            "WEEK_INDEX_CALCULATED"
        ]
    ]
)

Derived seasonal weeks:


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,DAP_CALCULATED,WEEK_INDEX_CALCULATED
0,FLD001,2026,2026-05-04,NaT,NaN,NaN
1,FLD001,2026,2026-05-11,NaT,NaN,NaN
2,FLD001,2026,2026-05-18,NaT,NaN,NaN
3,FLD001,2026,2026-05-25,NaT,NaN,NaN
4,FLD001,2026,2026-06-01,NaT,NaN,NaN


In [45]:
print("Weather key dtypes:")
print(weather_check[["FIELD_ID", "SEASON_YEAR"]].dtypes)

print("\nField context key dtypes:")
print(field_context_full[["FIELD_ID", "SEASON_YEAR"]].dtypes)

print("\nWeather keys:")
display(
    weather_check[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().head(10)
)

print("\nField context keys:")
display(
    field_context_full[
        ["FIELD_ID", "SEASON_YEAR", "SEASON_START_DATE"]
    ].drop_duplicates().head(10)
)

Weather key dtypes:
FIELD_ID       object
SEASON_YEAR     int64
dtype: object

Field context key dtypes:
FIELD_ID       object
SEASON_YEAR     int64
dtype: object

Weather keys:


,FIELD_ID,SEASON_YEAR
0,FLD001,2026



Field context keys:


,FIELD_ID,SEASON_YEAR,SEASON_START_DATE
0,FLD001,2027,2027-05-01
1,FLD002,2026,2026-04-27
2,FLD003,2026,2026-05-10
3,FLD004,2023,2023-04-20
4,FLD004,2024,2024-05-01
5,FLD004,2025,2025-04-25
6,FLD004,2026,2026-04-25
7,FLD005,2026,2026-04-29
8,FLD006,2026,2026-05-06
9,FLD007,2023,2023-04-27


In [46]:
# Step 25D: Check Weather Snapshot coverage against FIELD_SEASON_CONTEXT

weather_all = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SNAPSHOT_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
    """,
    conn
)

weather_all["FIELD_ID"] = weather_all["FIELD_ID"].astype(str).str.strip()
weather_all["SEASON_YEAR"] = pd.to_numeric(
    weather_all["SEASON_YEAR"],
    errors="coerce"
)

context_keys = field_context_full[
    ["FIELD_ID", "SEASON_YEAR"]
].drop_duplicates().copy()

weather_all = weather_all.merge(
    context_keys.assign(CONTEXT_MATCH=True),
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

print("Total weather snapshot rows:", len(weather_all))

print("\nRows with matching season context:")
print(weather_all["CONTEXT_MATCH"].eq(True).sum())

print("\nRows without matching season context:")
print(weather_all["CONTEXT_MATCH"].isna().sum())

print("\nUnmatched field-season combinations:")

display(
    weather_all.loc[
        weather_all["CONTEXT_MATCH"].isna(),
        ["FIELD_ID", "SEASON_YEAR"]
    ]
    .drop_duplicates()
    .sort_values(["FIELD_ID", "SEASON_YEAR"])
)

/tmp/ipykernel_75164/3291147275.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_all = pd.read_sql(


Total weather snapshot rows: 1412

Rows with matching season context:
1206

Rows without matching season context:
206

Unmatched field-season combinations:


,FIELD_ID,SEASON_YEAR
0,FLD001,2026
488,FLD016,2026
974,FLD031,2023
987,FLD031,2024
1000,FLD031,2025
1013,FLD031,2026
1025,FLD032,2023
1038,FLD032,2024
1051,FLD032,2025
1064,FLD032,2026


In [47]:
# Step 25E: Inspect FINAL_CLEANED_FIELDS for season-date coverage

query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.FINAL_CLEANED_FIELDS
LIMIT 5
"""

fields_sample = pd.read_sql(query, conn)

print("Shape:", fields_sample.shape)

print("\nColumns:")
print(fields_sample.columns.tolist())

display(fields_sample)

/tmp/ipykernel_75164/542122015.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fields_sample = pd.read_sql(query, conn)


Shape: (5, 12)

Columns:
['FIELD_ID', 'ACREAGE', 'HISTORICAL_AVG_YIELD_BU_ACRE', 'SEASON_YEAR', 'COUNTY_STATE', 'CROP_TYPE', 'CROP_VARIETY', 'FARM_NAME', 'PLANTING_DATE', 'HARVEST_DATE', 'SOIL_TYPE', 'WINTER_SEVERITY_INDEX']


,FIELD_ID,ACREAGE,HISTORICAL_AVG_YIELD_BU_ACRE,SEASON_YEAR,COUNTY_STATE,CROP_TYPE,CROP_VARIETY,FARM_NAME,PLANTING_DATE,HARVEST_DATE,SOIL_TYPE,WINTER_SEVERITY_INDEX
0,FLD001,102,56.9,2027,"Dane County, WI",Soybean,"Soybean, MG 2.1",TBD,2027-05-01,None,None,Extreme
1,FLD002,158,196.7,2026,"Fulton County, OH",Corn,"Corn, 104-day RM",Fry Farms,2026-04-27,None,Loam,Mild
2,FLD003,103,60.2,2026,"Champaign County, IL",Soybean,"Soybean, MG 3.2",Stafford Farms,2026-05-10,2026-09-27,Sandy Loam,Mild
3,FLD004,165 ac,181 bu-ac,2026,"Story County, IA",Corn,"Corn, 105-day RM",Miller Farms,2026-04-25,2026-10-10,Silty Clay Loam,Mild
4,FLD005,245 Acre,184.8,2026,"Fulton County, OH",Corn,"Corn, 105-day RM",Vargas Farms,2026-04-29,2026-10-13,Clay Loam,Severe


In [48]:
# Step 25F: Check unmatched weather field-seasons
# against FINAL_CLEANED_FIELDS

fields_all = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        CROP_TYPE,
        PLANTING_DATE,
        HARVEST_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.FINAL_CLEANED_FIELDS
    """,
    conn
)

fields_all["FIELD_ID"] = (
    fields_all["FIELD_ID"]
    .astype(str)
    .str.strip()
)

fields_all["SEASON_YEAR"] = pd.to_numeric(
    fields_all["SEASON_YEAR"],
    errors="coerce"
)

# Get unmatched weather field-season combinations
unmatched_weather = (
    weather_all.loc[
        weather_all["CONTEXT_MATCH"].isna(),
        ["FIELD_ID", "SEASON_YEAR"]
    ]
    .drop_duplicates()
)

# Match against FINAL_CLEANED_FIELDS
coverage_check = unmatched_weather.merge(
    fields_all,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    indicator=True
)

print("Unmatched weather field-season combinations:",
      len(unmatched_weather))

print("\nFound in FINAL_CLEANED_FIELDS:")
print((coverage_check["_merge"] == "both").sum())

print("\nStill missing:")
print((coverage_check["_merge"] == "left_only").sum())

display(
    coverage_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            "PLANTING_DATE",
            "HARVEST_DATE",
            "_merge"
        ]
    ]
)

/tmp/ipykernel_75164/4070367959.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fields_all = pd.read_sql(


Unmatched weather field-season combinations: 24

Found in FINAL_CLEANED_FIELDS:
2

Still missing:
22


,FIELD_ID,SEASON_YEAR,CROP_TYPE,PLANTING_DATE,HARVEST_DATE,_merge
0,FLD001,2026,NaN,NaN,NaN,left_only
1,FLD016,2026,Soybean,None,None,both
2,FLD031,2023,NaN,NaN,NaN,left_only
3,FLD031,2024,NaN,NaN,NaN,left_only
4,FLD031,2025,NaN,NaN,NaN,left_only
5,FLD031,2026,NaN,NaN,NaN,left_only
6,FLD032,2023,NaN,NaN,NaN,left_only
7,FLD032,2024,NaN,NaN,NaN,left_only
8,FLD032,2025,NaN,NaN,NaN,left_only
9,FLD032,2026,NaN,NaN,NaN,left_only


In [49]:
# Step 25G: Inspect SEASON_INTELLIGENCE_INPUT schema

query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_INTELLIGENCE_INPUT
LIMIT 5
"""

season_intel_sample = pd.read_sql(query, conn)

print("Shape:", season_intel_sample.shape)

print("\nColumns:")
print(season_intel_sample.columns.tolist())

display(season_intel_sample)

/tmp/ipykernel_75164/4142032201.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_intel_sample = pd.read_sql(query, conn)


Shape: (5, 39)

Columns:
['FIELD_ID', 'SEASON_YEAR', 'CROP_TYPE', 'WEEK_INDEX', 'SEASON_DAY', 'WEATHER_EXPECTED', 'GDD_EXPECTED', 'NDVI_EXPECTED', 'PEST_EXPECTED', 'DISEASE_EXPECTED', 'WEATHER_LOWER_BOUND', 'WEATHER_UPPER_BOUND', 'GDD_LOWER_BOUND', 'GDD_UPPER_BOUND', 'NDVI_LOWER_BOUND', 'NDVI_UPPER_BOUND', 'PEST_LOWER_BOUND', 'PEST_UPPER_BOUND', 'DISEASE_LOWER_BOUND', 'DISEASE_UPPER_BOUND', 'WEATHER_DEVIATION', 'GDD_DEVIATION', 'NDVI_DEVIATION', 'PEST_DEVIATION', 'DISEASE_DEVIATION', 'DEVIATION_STATE', 'DEVIATION_LEVEL', 'DEVIATION_DIRECTION', 'DOMINANT_DEVIATION_DRIVER', 'SIGNALS_ABNORMAL_COUNT', 'SIGNALS_CONFIRMING_COUNT', 'MULTI_SIGNAL_CONFIRMED', 'DEVIATION_CONFIDENCE', 'MISSING_SIGNAL_COUNT', 'PEST_CURRENT', 'DISEASE_CURRENT', 'NDVI_CURRENT', 'WEATHER_CURRENT', 'GDD_CURRENT']


,FIELD_ID,SEASON_YEAR,CROP_TYPE,WEEK_INDEX,SEASON_DAY,WEATHER_EXPECTED,GDD_EXPECTED,NDVI_EXPECTED,PEST_EXPECTED,DISEASE_EXPECTED,...,SIGNALS_ABNORMAL_COUNT,SIGNALS_CONFIRMING_COUNT,MULTI_SIGNAL_CONFIRMED,DEVIATION_CONFIDENCE,MISSING_SIGNAL_COUNT,PEST_CURRENT,DISEASE_CURRENT,NDVI_CURRENT,WEATHER_CURRENT,GDD_CURRENT
0,FLD007,2023,Corn,1,7,57.0,100.4,0.325,1.50,5.6,...,2,0,False,LOW,0,1.1,None,0.26,57.0,59.5
1,FLD007,2023,Corn,3,21,59.1,211.6,0.445,2.60,11.5,...,4,0,False,LOW,0,2.4,None,0.38,59.1,181.4
2,FLD007,2023,Corn,5,35,64.3,403.4,0.485,3.75,5.2,...,4,1,False,MEDIUM,0,3.8,None,0.40,63.7,343.1
3,FLD007,2023,Corn,7,49,68.2,614.4,0.580,8.45,8.8,...,3,2,True,MEDIUM,0,4.1,None,0.48,68.2,566.8
4,FLD007,2023,Corn,9,63,72.1,905.2,0.640,13.80,8.3,...,3,1,False,MEDIUM,0,15.9,None,0.57,69.9,833.5


In [50]:
# Step 25H: Inspect available snapshot dates for the main
# yield-forecast feature sources

tables_to_check = {
    "WEATHER": "WEATHER_FEATURE_SNAPSHOT",
    "NDVI": "NDVI_FEATURE_SNAPSHOT",
    "PEST": "GOLD_PEST_FEATURE_SNAPSHOT",
    "DISEASE": "GOLD_DISEASE_FEATURE_SNAPSHOT",
}

for name, table in tables_to_check.items():

    query = f"""
    SELECT *
    FROM YIELD_FORECAST_DB.POD_VERTEX.{table}
    LIMIT 3
    """

    df = pd.read_sql(query, conn)

    print(f"\n{'='*70}")
    print(f"{name} — {table}")
    print(f"Shape of sample: {df.shape}")
    print("Columns:")
    print(df.columns.tolist())

    # Show only first 3 rows
    display(df.head(3))


WEATHER — WEATHER_FEATURE_SNAPSHOT
Shape of sample: (3, 20)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE', 'MEAN_TEMPERATURE', 'TEMPERATURE_7D', 'TEMPERATURE_14D', 'TEMPERATURE_DEVIATION', 'RAINFALL_7D', 'RAINFALL_14D', 'RAINFALL_DEVIATION', 'HUMIDITY_7D', 'HUMIDITY_14D', 'HUMIDITY_DEVIATION', 'LEAFWETNESS_7D', 'LEAFWETNESS_DEVIATION', 'CUMULATIVE_GDD', 'HISTORICAL_CUMULATIVE_GDD', 'GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'WEATHER_DATA_AGE']


/tmp/ipykernel_75164/1840766849.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,MEAN_TEMPERATURE,TEMPERATURE_7D,TEMPERATURE_14D,TEMPERATURE_DEVIATION,RAINFALL_7D,RAINFALL_14D,RAINFALL_DEVIATION,HUMIDITY_7D,HUMIDITY_14D,HUMIDITY_DEVIATION,LEAFWETNESS_7D,LEAFWETNESS_DEVIATION,CUMULATIVE_GDD,HISTORICAL_CUMULATIVE_GDD,GDD_DEVIATION,GDD_DEVIATION_PCT,WEATHER_DATA_AGE
0,FLD001,2026,2026-05-04,62.5,62.5,62.50,-4.685714,0.68,0.68,-0.441667,55.9,55.90,-3.7,8.7,1.44,87.8,112.85,-25.05,-22.197607,84
1,FLD001,2026,2026-05-11,63.9,63.9,63.20,0.000000,0.00,0.34,0.000000,65.1,60.50,0.0,3.6,0.00,185.1,105.65,79.45,75.201136,77
2,FLD001,2026,2026-05-18,64.2,64.2,64.05,0.000000,0.00,0.00,0.000000,58.8,61.95,0.0,3.2,0.00,284.2,201.05,83.15,41.357871,70


/tmp/ipykernel_75164/1840766849.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



NDVI — NDVI_FEATURE_SNAPSHOT
Shape of sample: (3, 12)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'CROP_TYPE', 'WEEK_INDEX', 'SEASON_DAY', 'NDVI_VALUE', 'EXPECTED_NDVI', 'NDVI_DEVIATION', 'NDVI_CHANGE', 'NDVI_AVAILABLE', 'NDVI_RANGE_VALID', 'NDVI_CHANGE_VALID']


...


PEST — GOLD_PEST_FEATURE_SNAPSHOT
Shape of sample: (3, 12)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE', 'Max_Pest_Count', 'Max_Pest_Growth_Rate', 'Max_Pest_ET_Ratio', 'Any_Threshold_Exceeded', 'Dominant_Pest', 'Max_Pest_Count_Trend', 'Max_Pest_Acceleration', 'Cumulative_Pest_Pressure', 'Scouting_Data_Age']


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,Max_Pest_Count,Max_Pest_Growth_Rate,Max_Pest_ET_Ratio,Any_Threshold_Exceeded,Dominant_Pest,Max_Pest_Count_Trend,Max_Pest_Acceleration,Cumulative_Pest_Pressure,Scouting_Data_Age
0,FLD001,2026,1777852800000000,0.0,NaN,0.000,0.0,Bean Leaf Beetle,NaN,NaN,0.0,129
1,FLD001,2026,1779062400000000,2.9,NaN,NaN,0.0,Bean Leaf Beetle,2.9,NaN,2.9,115
2,FLD001,2026,1780272000000000,3.5,0.207,0.175,0.0,Bean Leaf Beetle,0.6,-2.3,6.4,101


/tmp/ipykernel_75164/1840766849.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



DISEASE — GOLD_DISEASE_FEATURE_SNAPSHOT
Shape of sample: (3, 9)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE', 'DOMINANT_DISEASE', 'MAX_DISEASE_RISK', 'MAX_DISEASE_RISK_TREND', 'MAX_DISEASE_WEATHER_SUITABILITY', 'DISEASE_TIMING_DEVIATION', 'DISEASE_DATA_AGE']


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,DOMINANT_DISEASE,MAX_DISEASE_RISK,MAX_DISEASE_RISK_TREND,MAX_DISEASE_WEATHER_SUITABILITY,DISEASE_TIMING_DEVIATION,DISEASE_DATA_AGE
0,FLD001,2026,2026-05-11,Sudden Death Syndrome (SDS),1,NaN,None,None,135
1,FLD001,2026,2026-05-25,White Mold (Sclerotinia),1,0.0,None,None,121
2,FLD001,2026,2026-06-08,White Mold (Sclerotinia),1,0.0,None,None,107


In [51]:
# Step 25I: Inspect the established NDVI seasonal-week structure

query = """
SELECT
    WEEK_INDEX,
    SEASON_DAY,
    COUNT(*) AS ROW_COUNT
FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
GROUP BY
    WEEK_INDEX,
    SEASON_DAY
ORDER BY
    WEEK_INDEX
"""

ndvi_week_structure = pd.read_sql(query, conn)

print("NDVI seasonal week structure:")
display(ndvi_week_structure)

/tmp/ipykernel_75164/2926784603.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ndvi_week_structure = pd.read_sql(query, conn)


NDVI seasonal week structure:


,WEEK_INDEX,SEASON_DAY,ROW_COUNT
0,0,3,111
1,1,10,111
2,2,17,107
3,3,24,113
4,4,31,111
5,5,38,111
6,6,45,105
7,7,52,113
8,8,59,113
9,9,66,111


In [52]:
# Step 25J: Inspect NDVI dates and seasonal alignment
# for one representative field-season

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    WEEK_INDEX,
    SEASON_DAY,
    NDVI_VALUE
FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
WHERE FIELD_ID = 'FLD001'
  AND SEASON_YEAR = 2026
ORDER BY WEEK_INDEX
"""

ndvi_fld001 = pd.read_sql(query, conn)

print("FLD001 / 2026 NDVI seasonal structure:")
display(ndvi_fld001)

/tmp/ipykernel_75164/3084230028.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ndvi_fld001 = pd.read_sql(query, conn)


FLD001 / 2026 NDVI seasonal structure:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,NDVI_VALUE
0,FLD001,2026,0,3,0.61
1,FLD001,2026,1,10,NaN
2,FLD001,2026,2,17,0.62
3,FLD001,2026,3,24,NaN
4,FLD001,2026,4,31,0.68
5,FLD001,2026,5,38,NaN
6,FLD001,2026,6,45,0.70
7,FLD001,2026,7,52,NaN
8,FLD001,2026,8,59,0.73
9,FLD001,2026,9,66,NaN


In [53]:
# Step 25K: Inspect the source NDVI records for FLD001 / 2026

query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.CLEANED_SATELLITE_NDVI_READINGS
WHERE FIELD_ID = 'FLD001'
  AND SEASON_YEAR = 2026
ORDER BY 1
"""

ndvi_source_fld001 = pd.read_sql(query, conn)

print("Shape:", ndvi_source_fld001.shape)

print("\nColumns:")
print(ndvi_source_fld001.columns.tolist())

display(ndvi_source_fld001)

/tmp/ipykernel_75164/1417486167.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ndvi_source_fld001 = pd.read_sql(query, conn)


Shape: (7, 9)

Columns:
['READING_ID', 'FIELD_ID', 'IMAGE_DATE', 'NDVI_VALUE', 'STRESS_FLAG', 'SEASON_YEAR', 'CROP_TYPE', 'NDVI_CHANGE', 'CLOUD_COVER_PCT']


,READING_ID,FIELD_ID,IMAGE_DATE,NDVI_VALUE,STRESS_FLAG,SEASON_YEAR,CROP_TYPE,NDVI_CHANGE,CLOUD_COVER_PCT
0,NDVI00001,FLD001,2026-05-04,0.61,N,2026,N/A,N/A,40.7
1,NDVI00002,FLD001,2026-05-18,0.62,N,2026,N/A,0.01,62.8
2,NDVI00003,FLD001,2026-06-01,0.68,N,2026,N/A,0.06,11.4
3,NDVI00004,FLD001,2026-06-15,0.7,N,2026,N/A,0.02,39.3
4,NDVI00005,FLD001,2026-06-29,0.73,Y,2026,N/A,0.03,56.4
5,NDVI00006,FLD001,2026-07-13,0.77,N,2026,N/A,0.04,6
6,NDVI00007,FLD001,2026-07-27,0.81,N,2026,N/A,0.04,45.3


In [54]:
# Step 25L: Validate NDVI seasonal-day calculation
# across all available NDVI observations

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    WEEK_INDEX,
    SEASON_DAY
FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
"""

ndvi_alignment = pd.read_sql(query, conn)

# Expected seasonal day from the established project rule
ndvi_alignment["EXPECTED_SEASON_DAY"] = (
    3 + (ndvi_alignment["WEEK_INDEX"] * 7)
)

ndvi_alignment["ALIGNMENT_MATCH"] = (
    ndvi_alignment["SEASON_DAY"]
    == ndvi_alignment["EXPECTED_SEASON_DAY"]
)

print("Total NDVI rows:", len(ndvi_alignment))

print(
    "Matching seasonal-day rows:",
    ndvi_alignment["ALIGNMENT_MATCH"].sum()
)

print(
    "Mismatching seasonal-day rows:",
    (~ndvi_alignment["ALIGNMENT_MATCH"]).sum()
)

print("\nMismatches:")
display(
    ndvi_alignment.loc[
        ~ndvi_alignment["ALIGNMENT_MATCH"]
    ].head(20)
)

Total NDVI rows: 1439
Matching seasonal-day rows: 1439
Mismatching seasonal-day rows: 0

Mismatches:


/tmp/ipykernel_75164/1845898964.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ndvi_alignment = pd.read_sql(query, conn)


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,EXPECTED_SEASON_DAY,ALIGNMENT_MATCH


In [55]:
# Step 25M: Compare Weather snapshot dates with
# the established NDVI seasonal checkpoints

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    SNAPSHOT_DATE
FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
WHERE FIELD_ID = 'FLD001'
  AND SEASON_YEAR = 2026
ORDER BY SNAPSHOT_DATE
"""

weather_fld001 = pd.read_sql(query, conn)

# Known NDVI seasonal checkpoints for FLD001 / 2026
ndvi_fld001_checkpoints = ndvi_source_fld001[
    ["IMAGE_DATE"]
].copy()

ndvi_fld001_checkpoints["IMAGE_DATE"] = pd.to_datetime(
    ndvi_fld001_checkpoints["IMAGE_DATE"]
)

weather_fld001["SNAPSHOT_DATE"] = pd.to_datetime(
    weather_fld001["SNAPSHOT_DATE"]
)

# Display all weather dates
print("Weather snapshot dates:")
display(weather_fld001)

print("\nNDVI observation dates:")
display(ndvi_fld001_checkpoints)


/tmp/ipykernel_75164/1080060570.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_fld001 = pd.read_sql(query, conn)


Weather snapshot dates:


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE
0,FLD001,2026,2026-05-04
1,FLD001,2026,2026-05-11
2,FLD001,2026,2026-05-18
3,FLD001,2026,2026-05-25
4,FLD001,2026,2026-06-01
5,FLD001,2026,2026-06-08
6,FLD001,2026,2026-06-15
7,FLD001,2026,2026-06-22
8,FLD001,2026,2026-06-29
9,FLD001,2026,2026-07-06



NDVI observation dates:


,IMAGE_DATE
0,2026-05-04
1,2026-05-18
2,2026-06-01
3,2026-06-15
4,2026-06-29
5,2026-07-13
6,2026-07-27


In [56]:
# Step 25N: Validate weekly Weather snapshot spacing
# across all field-seasons

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    SNAPSHOT_DATE
FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
ORDER BY FIELD_ID, SEASON_YEAR, SNAPSHOT_DATE
"""

weather_dates = pd.read_sql(query, conn)

weather_dates["SNAPSHOT_DATE"] = pd.to_datetime(
    weather_dates["SNAPSHOT_DATE"]
)

weather_dates["PREVIOUS_DATE"] = (
    weather_dates
    .groupby(["FIELD_ID", "SEASON_YEAR"])["SNAPSHOT_DATE"]
    .shift(1)
)

weather_dates["DAYS_FROM_PREVIOUS"] = (
    weather_dates["SNAPSHOT_DATE"]
    - weather_dates["PREVIOUS_DATE"]
).dt.days

# Ignore the first observation of each field-season
intervals = weather_dates.loc[
    weather_dates["DAYS_FROM_PREVIOUS"].notna(),
    "DAYS_FROM_PREVIOUS"
]

print("Total Weather rows:", len(weather_dates))
print("Intervals checked:", len(intervals))

print("\nInterval distribution:")
print(intervals.value_counts().sort_index())

print("\nNon-7-day intervals:")
display(
    weather_dates.loc[
        weather_dates["DAYS_FROM_PREVIOUS"].notna()
        & (weather_dates["DAYS_FROM_PREVIOUS"] != 7),
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "PREVIOUS_DATE",
            "SNAPSHOT_DATE",
            "DAYS_FROM_PREVIOUS"
        ]
    ].head(30)
)

Total Weather rows: 1412
Intervals checked: 1292

Interval distribution:
DAYS_FROM_PREVIOUS
2.0        3
5.0        3
7.0     1242
14.0      37
21.0       5
45.0       2
Name: count, dtype: int64

Non-7-day intervals:


/tmp/ipykernel_75164/2331104792.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_dates = pd.read_sql(query, conn)


,FIELD_ID,SEASON_YEAR,PREVIOUS_DATE,SNAPSHOT_DATE,DAYS_FROM_PREVIOUS
83,FLD004,2026,2026-06-08,2026-06-22,14.0
104,FLD006,2026,2026-05-11,2026-05-25,14.0
107,FLD006,2026,2026-06-08,2026-06-22,14.0
157,FLD007,2026,2026-06-01,2026-06-22,21.0
208,FLD008,2026,2026-06-08,2026-06-22,14.0
255,FLD009,2026,2026-05-11,2026-05-25,14.0
261,FLD009,2026,2026-06-29,2026-07-01,2.0
262,FLD009,2026,2026-07-01,2026-07-06,5.0
306,FLD010,2026,2026-05-11,2026-05-25,14.0
309,FLD010,2026,2026-06-08,2026-06-22,14.0


In [58]:
# Step 25O: Compare the first Weather observation
# with the first NDVI seasonal observation

query = """
WITH weather_first AS (
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        MIN(SNAPSHOT_DATE) AS FIRST_WEATHER_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
    GROUP BY FIELD_ID, SEASON_YEAR
),

ndvi_first AS (
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        MIN(WEEK_INDEX) AS FIRST_NDVI_WEEK,
        MIN(SEASON_DAY) AS FIRST_NDVI_SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
    GROUP BY FIELD_ID, SEASON_YEAR
)

SELECT
    w.FIELD_ID,
    w.SEASON_YEAR,
    w.FIRST_WEATHER_DATE,
    n.FIRST_NDVI_WEEK,
    n.FIRST_NDVI_SEASON_DAY
FROM weather_first w
INNER JOIN ndvi_first n
    ON w.FIELD_ID = n.FIELD_ID
    AND w.SEASON_YEAR = n.SEASON_YEAR
ORDER BY w.FIELD_ID, w.SEASON_YEAR
"""

first_observation_alignment = pd.read_sql(query, conn)

print(
    "Field-seasons with both Weather and NDVI:",
    len(first_observation_alignment)
)

print("\nFirst NDVI seasonal-week distribution:")
print(
    first_observation_alignment[
        "FIRST_NDVI_WEEK"
    ].value_counts().sort_index()
)

print("\nFirst NDVI seasonal-day distribution:")
print(
    first_observation_alignment[
        "FIRST_NDVI_SEASON_DAY"
    ].value_counts().sort_index()
)

print("\nSample:")
display(first_observation_alignment.head(30))

/tmp/ipykernel_75164/2971058852.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  first_observation_alignment = pd.read_sql(query, conn)


Field-seasons with both Weather and NDVI: 111

First NDVI seasonal-week distribution:
FIRST_NDVI_WEEK
0    108
1      3
Name: count, dtype: int64

First NDVI seasonal-day distribution:
FIRST_NDVI_SEASON_DAY
3     108
10      3
Name: count, dtype: int64

Sample:


,FIELD_ID,SEASON_YEAR,FIRST_WEATHER_DATE,FIRST_NDVI_WEEK,FIRST_NDVI_SEASON_DAY
0,FLD001,2026,2026-05-04,0,3
1,FLD002,2026,2026-05-04,0,3
2,FLD003,2026,2026-05-04,0,3
3,FLD004,2023,2023-05-04,0,3
4,FLD004,2024,2024-05-04,0,3
5,FLD004,2025,2025-05-04,0,3
6,FLD004,2026,2026-05-11,1,10
7,FLD005,2026,2026-05-04,0,3
8,FLD006,2026,2026-05-04,0,3
9,FLD007,2023,2023-05-04,0,3


In [59]:
# Step 25P: Check authoritative season-start coverage

query = """
SELECT
    COUNT(*) AS TOTAL_FIELD_SEASONS,
    COUNT(SEASON_START_DATE) AS SEASONS_WITH_START_DATE,
    COUNT(PLANTING_DATE) AS SEASONS_WITH_PLANTING_DATE
FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
"""

season_start_coverage = pd.read_sql(query, conn)

display(season_start_coverage)

/tmp/ipykernel_75164/560091421.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_start_coverage = pd.read_sql(query, conn)


,TOTAL_FIELD_SEASONS,SEASONS_WITH_START_DATE,SEASONS_WITH_PLANTING_DATE
0,101,101,101


In [60]:
# Show field-seasons where SEASON_START_DATE is missing

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    SEASON_START_DATE,
    PLANTING_DATE,
    HARVEST_DATE,
    CROP_TYPE
FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
WHERE SEASON_START_DATE IS NULL
ORDER BY FIELD_ID, SEASON_YEAR
"""

missing_season_start = pd.read_sql(query, conn)

print("Field-seasons missing SEASON_START_DATE:", len(missing_season_start))

display(missing_season_start)

/tmp/ipykernel_75164/2315181546.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  missing_season_start = pd.read_sql(query, conn)


Field-seasons missing SEASON_START_DATE: 0


,FIELD_ID,SEASON_YEAR,SEASON_START_DATE,PLANTING_DATE,HARVEST_DATE,CROP_TYPE


In [62]:
# Step 25Q: Attach authoritative season start
# to the validated NDVI seasonal structure

ndvi_season = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
    ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
    """,
    conn
)

season_context = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SEASON_START_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
    ORDER BY FIELD_ID, SEASON_YEAR
    """,
    conn
)

ndvi_season["SEASON_YEAR"] = ndvi_season["SEASON_YEAR"].astype(int)
season_context["SEASON_YEAR"] = season_context["SEASON_YEAR"].astype(int)

ndvi_season = ndvi_season.merge(
    season_context,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

ndvi_season["SEASON_START_DATE"] = pd.to_datetime(
    ndvi_season["SEASON_START_DATE"]
)

# Calculate the calendar date represented by each seasonal day.
ndvi_season["EXPECTED_DATE"] = (
    ndvi_season["SEASON_START_DATE"]
    + pd.to_timedelta(
        ndvi_season["SEASON_DAY"] - 1,
        unit="D"
    )
)

print("Total NDVI rows:", len(ndvi_season))
print(
    "Rows with missing season start:",
    ndvi_season["SEASON_START_DATE"].isna().sum()
)

display(ndvi_season.head(20))

/tmp/ipykernel_75164/2436509836.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ndvi_season = pd.read_sql(
/tmp/ipykernel_75164/2436509836.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_context = pd.read_sql(


Total NDVI rows: 1439
Rows with missing season start: 190


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,SEASON_START_DATE,EXPECTED_DATE
0,FLD001,2026,0,3,NaT,NaT
1,FLD001,2026,1,10,NaT,NaT
2,FLD001,2026,2,17,NaT,NaT
3,FLD001,2026,3,24,NaT,NaT
4,FLD001,2026,4,31,NaT,NaT
5,FLD001,2026,5,38,NaT,NaT
6,FLD001,2026,6,45,NaT,NaT
7,FLD001,2026,7,52,NaT,NaT
8,FLD001,2026,8,59,NaT,NaT
9,FLD001,2026,9,66,NaT,NaT


In [63]:
# Step 25R: Identify NDVI field-seasons without
# authoritative FIELD_SEASON_CONTEXT

ndvi_field_seasons = (
    ndvi_season[
        ["FIELD_ID", "SEASON_YEAR"]
    ]
    .drop_duplicates()
)

context_field_seasons = (
    season_context[
        ["FIELD_ID", "SEASON_YEAR"]
    ]
    .drop_duplicates()
)

unmatched_ndvi_seasons = ndvi_field_seasons.merge(
    context_field_seasons,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    indicator=True
)

unmatched_ndvi_seasons = unmatched_ndvi_seasons.loc[
    unmatched_ndvi_seasons["_merge"] == "left_only",
    ["FIELD_ID", "SEASON_YEAR"]
].sort_values(
    ["FIELD_ID", "SEASON_YEAR"]
)

print(
    "Unmatched NDVI field-seasons:",
    len(unmatched_ndvi_seasons)
)

display(unmatched_ndvi_seasons)

Unmatched NDVI field-seasons: 15


,FIELD_ID,SEASON_YEAR
0,FLD001,2026
39,FLD016,2026
81,FLD031,2023
82,FLD031,2024
83,FLD031,2025
84,FLD031,2026
85,FLD032,2023
86,FLD032,2024
87,FLD032,2025
88,FLD032,2026


In [64]:
# Step 25S: Check whether unmatched NDVI field-seasons
# have authoritative dates in FINAL_CLEANED_FIELDS

query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    PLANTING_DATE,
    HARVEST_DATE,
    CROP_TYPE
FROM YIELD_FORECAST_DB.POD_VERTEX.FINAL_CLEANED_FIELDS
WHERE
    (FIELD_ID = 'FLD001' AND SEASON_YEAR = 2026)
    OR (FIELD_ID = 'FLD016' AND SEASON_YEAR = 2026)
    OR (FIELD_ID = 'FLD031' AND SEASON_YEAR BETWEEN 2023 AND 2026)
    OR (FIELD_ID = 'FLD032' AND SEASON_YEAR BETWEEN 2023 AND 2026)
    OR (FIELD_ID = 'FLD033' AND SEASON_YEAR = 2026)
    OR (FIELD_ID = 'FLD035' AND SEASON_YEAR BETWEEN 2023 AND 2026)
ORDER BY FIELD_ID, SEASON_YEAR
"""

unmatched_field_dates = pd.read_sql(query, conn)

print("Rows found in FINAL_CLEANED_FIELDS:", len(unmatched_field_dates))

display(unmatched_field_dates)

/tmp/ipykernel_75164/2915447979.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  unmatched_field_dates = pd.read_sql(query, conn)


Rows found in FINAL_CLEANED_FIELDS: 2


,FIELD_ID,SEASON_YEAR,PLANTING_DATE,HARVEST_DATE,CROP_TYPE
0,FLD016,2026,None,None,Soybean
1,FLD033,2026,None,None,Corn


In [66]:
query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    PLANTING_DATE,
    HARVEST_DATE,
    CROP_TYPE
FROM YIELD_FORECAST_DB.POD_VERTEX.CLEANED_FIELDS
WHERE
    (FIELD_ID = 'FLD001' AND TRY_TO_NUMBER(SEASON_YEAR) = 2026)
    OR (FIELD_ID = 'FLD016' AND TRY_TO_NUMBER(SEASON_YEAR) = 2026)
    OR (FIELD_ID = 'FLD031' AND TRY_TO_NUMBER(SEASON_YEAR) BETWEEN 2023 AND 2026)
    OR (FIELD_ID = 'FLD032' AND TRY_TO_NUMBER(SEASON_YEAR) BETWEEN 2023 AND 2026)
    OR (FIELD_ID = 'FLD033' AND TRY_TO_NUMBER(SEASON_YEAR) = 2026)
    OR (FIELD_ID = 'FLD035' AND TRY_TO_NUMBER(SEASON_YEAR) BETWEEN 2023 AND 2026)
ORDER BY FIELD_ID, SEASON_YEAR
"""

cleaned_fields_check = pd.read_sql(query, conn)

print("Rows found in CLEANED_FIELDS:", len(cleaned_fields_check))
display(cleaned_fields_check)

/tmp/ipykernel_75164/4245859758.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cleaned_fields_check = pd.read_sql(query, conn)


Rows found in CLEANED_FIELDS: 15


,FIELD_ID,SEASON_YEAR,PLANTING_DATE,HARVEST_DATE,CROP_TYPE
0,FLD016,2026,N/A,N/A,Soybean
1,FLD031,2023,2023-04-23,2023-10-11,Corn
2,FLD031,2024,2024-05-04,2024-10-11,Corn
3,FLD031,2025,2025-04-28,2025-10-11,Corn
4,FLD031,2026,2026-04-28,2026-10-01,Corn
5,FLD032,2023,2023-04-23,2023-10-07,Corn
6,FLD032,2024,2024-05-04,2024-10-07,Corn
7,FLD032,2025,2025-04-28,2025-10-07,Corn
8,FLD032,2026,N/A,2026-10-04,Corn
9,FLD032,2026,N/A,2026-10-10,Corn


In [67]:
# Prepare planting dates from CLEANED_FIELDS
cleaned_fields_check["PLANTING_DATE"] = pd.to_datetime(
    cleaned_fields_check["PLANTING_DATE"],
    errors="coerce"
)

# Keep only rows where a planting date is available
planting_reference = (
    cleaned_fields_check[
        cleaned_fields_check["PLANTING_DATE"].notna()
    ][
        ["FIELD_ID", "SEASON_YEAR", "PLANTING_DATE", "CROP_TYPE"]
    ]
    .drop_duplicates()
)

# Merge with NDVI seasonal structure
ndvi_date_check = ndvi_alignment.merge(
    planting_reference,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

# Calculate expected date from planting date
ndvi_date_check["EXPECTED_DATE_FROM_PLANTING"] = (
    ndvi_date_check["PLANTING_DATE"]
    + pd.to_timedelta(
        ndvi_date_check["SEASON_DAY"] - 1,
        unit="D"
    )
)

# Show only field-seasons that were previously unmatched
unmatched_keys = ndvi_season[
    ndvi_season["SEASON_START_DATE"].isna()
][
    ["FIELD_ID", "SEASON_YEAR"]
].drop_duplicates()

ndvi_date_check_unmatched = ndvi_date_check.merge(
    unmatched_keys,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

print(
    "Unmatched NDVI rows with planting-date information:",
    ndvi_date_check_unmatched["PLANTING_DATE"].notna().sum()
)

display(
    ndvi_date_check_unmatched[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "PLANTING_DATE",
            "EXPECTED_DATE_FROM_PLANTING"
        ]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    )
)

/tmp/ipykernel_75164/1055844709.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned_fields_check["PLANTING_DATE"] = pd.to_datetime(


ValueError: You are trying to merge on int64 and object columns for key 'SEASON_YEAR'. If you wish to proceed you should use pd.concat

In [68]:
# Make merge keys consistent
ndvi_alignment["SEASON_YEAR"] = pd.to_numeric(
    ndvi_alignment["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

cleaned_fields_check["SEASON_YEAR"] = pd.to_numeric(
    cleaned_fields_check["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

# Parse planting dates safely
cleaned_fields_check["PLANTING_DATE"] = pd.to_datetime(
    cleaned_fields_check["PLANTING_DATE"],
    errors="coerce"
)

# Build planting-date reference
planting_reference = (
    cleaned_fields_check[
        cleaned_fields_check["PLANTING_DATE"].notna()
    ][
        ["FIELD_ID", "SEASON_YEAR", "PLANTING_DATE", "CROP_TYPE"]
    ]
    .drop_duplicates()
)

# Merge planting dates with NDVI seasonal structure
ndvi_date_check = ndvi_alignment.merge(
    planting_reference,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

# Calculate expected calendar date from planting date
ndvi_date_check["EXPECTED_DATE_FROM_PLANTING"] = (
    ndvi_date_check["PLANTING_DATE"]
    + pd.to_timedelta(
        ndvi_date_check["SEASON_DAY"] - 1,
        unit="D"
    )
)

# Previously unmatched NDVI field-seasons
unmatched_keys = ndvi_season[
    ndvi_season["SEASON_START_DATE"].isna()
][
    ["FIELD_ID", "SEASON_YEAR"]
].drop_duplicates()

# Keep only those unmatched field-seasons
ndvi_date_check_unmatched = ndvi_date_check.merge(
    unmatched_keys,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

print(
    "Unmatched NDVI rows with planting-date information:",
    ndvi_date_check_unmatched["PLANTING_DATE"].notna().sum()
)

display(
    ndvi_date_check_unmatched[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "PLANTING_DATE",
            "EXPECTED_DATE_FROM_PLANTING"
        ]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    )
)

Unmatched NDVI rows with planting-date information: 143


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,PLANTING_DATE,EXPECTED_DATE_FROM_PLANTING
0,FLD001,2026,0,3,NaT,NaT
1,FLD001,2026,1,10,NaT,NaT
2,FLD001,2026,2,17,NaT,NaT
3,FLD001,2026,3,24,NaT,NaT
4,FLD001,2026,4,31,NaT,NaT
5,FLD001,2026,5,38,NaT,NaT
6,FLD001,2026,6,45,NaT,NaT
7,FLD001,2026,7,52,NaT,NaT
8,FLD001,2026,8,59,NaT,NaT
9,FLD001,2026,9,66,NaT,NaT


In [69]:
# Load raw NDVI observations
query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    IMAGE_DATE,
    NDVI_VALUE
FROM YIELD_FORECAST_DB.POD_VERTEX.CLEANED_SATELLITE_NDVI_READINGS
"""

raw_ndvi = pd.read_sql(query, conn)

# Normalize types
raw_ndvi["SEASON_YEAR"] = pd.to_numeric(
    raw_ndvi["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

raw_ndvi["IMAGE_DATE"] = pd.to_datetime(
    raw_ndvi["IMAGE_DATE"],
    errors="coerce"
)

# Add actual NDVI date to the seasonal NDVI structure
ndvi_actual_check = ndvi_alignment.merge(
    raw_ndvi,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

# Keep only previously unmatched field-seasons
ndvi_actual_check = ndvi_actual_check.merge(
    unmatched_keys,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

# Add planting-date reference
ndvi_actual_check = ndvi_actual_check.merge(
    planting_reference[
        ["FIELD_ID", "SEASON_YEAR", "PLANTING_DATE"]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

# Calculate expected date from planting date
ndvi_actual_check["EXPECTED_DATE_FROM_PLANTING"] = (
    ndvi_actual_check["PLANTING_DATE"]
    + pd.to_timedelta(
        ndvi_actual_check["SEASON_DAY"] - 1,
        unit="D"
    )
)

# Compare actual observation date to expected date
ndvi_actual_check["DATE_DIFFERENCE_DAYS"] = (
    ndvi_actual_check["IMAGE_DATE"]
    - ndvi_actual_check["EXPECTED_DATE_FROM_PLANTING"]
).dt.days

# Display only rows where an actual NDVI observation exists
validation = ndvi_actual_check[
    ndvi_actual_check["IMAGE_DATE"].notna()
][
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY",
        "PLANTING_DATE",
        "EXPECTED_DATE_FROM_PLANTING",
        "IMAGE_DATE",
        "DATE_DIFFERENCE_DAYS",
        "NDVI_VALUE"
    ]
].sort_values(
    ["FIELD_ID", "SEASON_YEAR", "IMAGE_DATE"]
)

print("Actual NDVI observations available for validation:", len(validation))

display(validation)

/tmp/ipykernel_75164/3788104420.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  raw_ndvi = pd.read_sql(query, conn)


Actual NDVI observations available for validation: 1225


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,PLANTING_DATE,EXPECTED_DATE_FROM_PLANTING,IMAGE_DATE,DATE_DIFFERENCE_DAYS,NDVI_VALUE
0,FLD001,2026,0,3,NaT,NaT,2026-05-04,NaN,0.61
7,FLD001,2026,1,10,NaT,NaT,2026-05-04,NaN,0.61
14,FLD001,2026,2,17,NaT,NaT,2026-05-04,NaN,0.61
21,FLD001,2026,3,24,NaT,NaT,2026-05-04,NaN,0.61
28,FLD001,2026,4,31,NaT,NaT,2026-05-04,NaN,0.61
35,FLD001,2026,5,38,NaT,NaT,2026-05-04,NaN,0.61
42,FLD001,2026,6,45,NaT,NaT,2026-05-04,NaN,0.61
49,FLD001,2026,7,52,NaT,NaT,2026-05-04,NaN,0.61
56,FLD001,2026,8,59,NaT,NaT,2026-05-04,NaN,0.61
63,FLD001,2026,9,66,NaT,NaT,2026-05-04,NaN,0.61


In [70]:
query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    IMAGE_DATE,
    NDVI_VALUE
FROM YIELD_FORECAST_DB.POD_VERTEX.CLEANED_SATELLITE_NDVI_READINGS
"""

raw_ndvi = pd.read_sql(query, conn)

raw_ndvi["SEASON_YEAR"] = pd.to_numeric(
    raw_ndvi["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

raw_ndvi["IMAGE_DATE"] = pd.to_datetime(
    raw_ndvi["IMAGE_DATE"],
    errors="coerce"
)

ndvi_actual_check = ndvi_alignment.merge(
    raw_ndvi,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

ndvi_actual_check = ndvi_actual_check.merge(
    unmatched_keys,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

ndvi_actual_check = ndvi_actual_check.merge(
    planting_reference[
        ["FIELD_ID", "SEASON_YEAR", "PLANTING_DATE"]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

ndvi_actual_check["EXPECTED_DATE_FROM_PLANTING"] = (
    ndvi_actual_check["PLANTING_DATE"]
    + pd.to_timedelta(
        ndvi_actual_check["SEASON_DAY"] - 1,
        unit="D"
    )
)

ndvi_actual_check["DATE_DIFFERENCE_DAYS"] = (
    ndvi_actual_check["IMAGE_DATE"]
    - ndvi_actual_check["EXPECTED_DATE_FROM_PLANTING"]
).dt.days

validation = ndvi_actual_check[
    ndvi_actual_check["IMAGE_DATE"].notna()
][
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY",
        "PLANTING_DATE",
        "EXPECTED_DATE_FROM_PLANTING",
        "IMAGE_DATE",
        "DATE_DIFFERENCE_DAYS",
        "NDVI_VALUE"
    ]
].sort_values(
    ["FIELD_ID", "SEASON_YEAR", "IMAGE_DATE"]
)

print(
    "Actual NDVI observations available for validation:",
    len(validation)
)

display(validation)

/tmp/ipykernel_75164/2889255854.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  raw_ndvi = pd.read_sql(query, conn)


Actual NDVI observations available for validation: 1225


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,PLANTING_DATE,EXPECTED_DATE_FROM_PLANTING,IMAGE_DATE,DATE_DIFFERENCE_DAYS,NDVI_VALUE
0,FLD001,2026,0,3,NaT,NaT,2026-05-04,NaN,0.61
7,FLD001,2026,1,10,NaT,NaT,2026-05-04,NaN,0.61
14,FLD001,2026,2,17,NaT,NaT,2026-05-04,NaN,0.61
21,FLD001,2026,3,24,NaT,NaT,2026-05-04,NaN,0.61
28,FLD001,2026,4,31,NaT,NaT,2026-05-04,NaN,0.61
35,FLD001,2026,5,38,NaT,NaT,2026-05-04,NaN,0.61
42,FLD001,2026,6,45,NaT,NaT,2026-05-04,NaN,0.61
49,FLD001,2026,7,52,NaT,NaT,2026-05-04,NaN,0.61
56,FLD001,2026,8,59,NaT,NaT,2026-05-04,NaN,0.61
63,FLD001,2026,9,66,NaT,NaT,2026-05-04,NaN,0.61


In [71]:
print("Date difference summary:")
print(validation["DATE_DIFFERENCE_DAYS"].describe())

print("\nUnique date differences:")
print(
    validation["DATE_DIFFERENCE_DAYS"]
    .value_counts(dropna=False)
    .sort_index()
)

Date difference summary:
count    962.000000
mean      -1.864865
std       40.919408
min     -104.000000
25%      -30.750000
50%       -2.000000
75%       26.000000
max      133.000000
Name: DATE_DIFFERENCE_DAYS, dtype: float64

Unique date differences:
DATE_DIFFERENCE_DAYS
-104.0      1
-98.0       2
-97.0       1
-93.0       1
-91.0       2
-90.0       2
-86.0       3
-84.0       4
-83.0       2
-80.0       3
-79.0       4
-77.0       4
-76.0       3
-75.0       2
-73.0       3
-72.0       6
-70.0       5
-69.0       3
-68.0       2
-66.0       6
-65.0       7
-63.0       5
-62.0       4
-61.0       4
-59.0       6
-58.0       9
-56.0       7
-55.0       4
-54.0       4
-52.0       8
-51.0      10
-49.0       7
-48.0       5
-47.0       6
-45.0       8
-44.0      12
-42.0       9
-41.0       5
-40.0       6
-38.0      10
-37.0      13
-35.0       9
-34.0       6
-33.0       8
-31.0      10
-30.0      15
-28.0      11
-27.0       6
-26.0       8
-24.0      13
-23.0      16
-21.0      

In [72]:
print("NDVI temporal alignment status:")

print(
    ndvi_season["SEASON_START_DATE"]
    .notna()
    .value_counts()
    .rename({
        True: "AUTHORITATIVE_SEASON_START_AVAILABLE",
        False: "SEASON_START_UNRESOLVED"
    })
)

print("\nUnique field-seasons:")
print(
    ndvi_season[
        ["FIELD_ID", "SEASON_YEAR", "SEASON_START_DATE"]
    ]
    .drop_duplicates()["SEASON_START_DATE"]
    .notna()
    .value_counts()
    .rename({
        True: "AUTHORITATIVE_SEASON_START_AVAILABLE",
        False: "SEASON_START_UNRESOLVED"
    })
)

NDVI temporal alignment status:
SEASON_START_DATE
AUTHORITATIVE_SEASON_START_AVAILABLE    1249
SEASON_START_UNRESOLVED                  190
Name: count, dtype: int64

Unique field-seasons:
SEASON_START_DATE
AUTHORITATIVE_SEASON_START_AVAILABLE    99
SEASON_START_UNRESOLVED                 15
Name: count, dtype: int64


In [73]:
query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    WEEK_INDEX
FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_INTELLIGENCE_INPUT
ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
"""

forecast_grain = pd.read_sql(query, conn)

forecast_grain["SEASON_YEAR"] = pd.to_numeric(
    forecast_grain["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

forecast_grain["WEEK_INDEX"] = pd.to_numeric(
    forecast_grain["WEEK_INDEX"],
    errors="coerce"
).astype("Int64")

print("Forecast grain shape:", forecast_grain.shape)

print(
    "\nUnique WEEK_INDEX values:",
    sorted(forecast_grain["WEEK_INDEX"].dropna().unique())
)

print(
    "\nUnique field-season combinations:",
    forecast_grain[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

display(forecast_grain.head(20))

/tmp/ipykernel_75164/1133345993.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  forecast_grain = pd.read_sql(query, conn)


Forecast grain shape: (111, 3)

Unique WEEK_INDEX values: [1, 3, 5, 7, 9, 11]

Unique field-season combinations: 19


,FIELD_ID,SEASON_YEAR,WEEK_INDEX
0,FLD007,2023,1
1,FLD007,2023,3
2,FLD007,2023,5
3,FLD007,2023,7
4,FLD007,2023,9
5,FLD007,2023,11
6,FLD007,2024,1
7,FLD007,2024,3
8,FLD007,2024,5
9,FLD007,2024,7


In [74]:
feature_tables = [
    "WEATHER_FEATURE_SNAPSHOT",
    "NDVI_FEATURE_SNAPSHOT",
    "GOLD_PEST_FEATURE_SNAPSHOT",
    "GOLD_DISEASE_FEATURE_SNAPSHOT",
    "SEASON_DEVIATION_RESULT",
    "FIELD_RISK_CONFIDENCE_SCORECARD_PB"
]

for table in feature_tables:
    query = f"""
    SELECT *
    FROM YIELD_FORECAST_DB.POD_VERTEX.{table}
    LIMIT 3
    """

    df = pd.read_sql(query, conn)

    print(f"\n{'='*70}")
    print(f"{table}")
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

/tmp/ipykernel_75164/1460351578.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



WEATHER_FEATURE_SNAPSHOT
Shape: (3, 20)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE', 'MEAN_TEMPERATURE', 'TEMPERATURE_7D', 'TEMPERATURE_14D', 'TEMPERATURE_DEVIATION', 'RAINFALL_7D', 'RAINFALL_14D', 'RAINFALL_DEVIATION', 'HUMIDITY_7D', 'HUMIDITY_14D', 'HUMIDITY_DEVIATION', 'LEAFWETNESS_7D', 'LEAFWETNESS_DEVIATION', 'CUMULATIVE_GDD', 'HISTORICAL_CUMULATIVE_GDD', 'GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'WEATHER_DATA_AGE']

NDVI_FEATURE_SNAPSHOT
Shape: (3, 12)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'CROP_TYPE', 'WEEK_INDEX', 'SEASON_DAY', 'NDVI_VALUE', 'EXPECTED_NDVI', 'NDVI_DEVIATION', 'NDVI_CHANGE', 'NDVI_AVAILABLE', 'NDVI_RANGE_VALID', 'NDVI_CHANGE_VALID']

GOLD_PEST_FEATURE_SNAPSHOT
Shape: (3, 12)
Columns:
['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE', 'Max_Pest_Count', 'Max_Pest_Growth_Rate', 'Max_Pest_ET_Ratio', 'Any_Threshold_Exceeded', 'Dominant_Pest', 'Max_Pest_Count_Trend', 'Max_Pest_Acceleration', 'Cumulative_Pest_Pressure', 'Scouting_Data_Age']

GOLD_DISEASE_FEATURE_SNAPSHO

In [75]:
date_tables = [
    "WEATHER_FEATURE_SNAPSHOT",
    "GOLD_PEST_FEATURE_SNAPSHOT",
    "GOLD_DISEASE_FEATURE_SNAPSHOT"
]

for table in date_tables:
    query = f"""
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SNAPSHOT_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.{table}
    ORDER BY FIELD_ID, SEASON_YEAR, SNAPSHOT_DATE
    """

    df = pd.read_sql(query, conn)

    df["SEASON_YEAR"] = pd.to_numeric(
        df["SEASON_YEAR"],
        errors="coerce"
    ).astype("Int64")

    df["SNAPSHOT_DATE"] = pd.to_datetime(
        df["SNAPSHOT_DATE"],
        errors="coerce"
    )

    print(f"\n{'='*70}")
    print(table)
    print("Rows:", len(df))
    print("Unique field-seasons:",
          df[["FIELD_ID", "SEASON_YEAR"]].drop_duplicates().shape[0])
    print("Date range:",
          df["SNAPSHOT_DATE"].min(),
          "to",
          df["SNAPSHOT_DATE"].max())
    print("Missing snapshot dates:",
          df["SNAPSHOT_DATE"].isna().sum())

    print("\nSample:")
    display(df.head(10))


WEATHER_FEATURE_SNAPSHOT
Rows: 1412
Unique field-seasons: 120
Date range: 2023-05-04 00:00:00 to 2026-09-10 00:00:00
Missing snapshot dates: 0

Sample:


/tmp/ipykernel_75164/316949818.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE
0,FLD001,2026,2026-05-04
1,FLD001,2026,2026-05-11
2,FLD001,2026,2026-05-18
3,FLD001,2026,2026-05-25
4,FLD001,2026,2026-06-01
5,FLD001,2026,2026-06-08
6,FLD001,2026,2026-06-15
7,FLD001,2026,2026-06-22
8,FLD001,2026,2026-06-29
9,FLD001,2026,2026-07-06


/tmp/ipykernel_75164/316949818.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



GOLD_PEST_FEATURE_SNAPSHOT
Rows: 762
Unique field-seasons: 120
Date range: 1970-01-20 11:32:38.400000 to 1970-01-21 16:56:38.400000
Missing snapshot dates: 0

Sample:


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE
0,FLD001,2026,1970-01-21 13:50:52.800
1,FLD001,2026,1970-01-21 14:11:02.400
2,FLD001,2026,1970-01-21 14:31:12.000
3,FLD001,2026,1970-01-21 15:11:31.200
4,FLD001,2026,1970-01-21 15:51:50.400
5,FLD002,2026,1970-01-21 13:50:52.800
6,FLD002,2026,1970-01-21 14:11:02.400
7,FLD002,2026,1970-01-21 15:11:31.200
8,FLD002,2026,1970-01-21 15:31:40.800
9,FLD002,2026,1970-01-21 16:56:38.400


/tmp/ipykernel_75164/316949818.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



GOLD_DISEASE_FEATURE_SNAPSHOT
Rows: 232
Unique field-seasons: 40
Date range: 2026-05-11 00:00:00 to 2026-07-20 00:00:00
Missing snapshot dates: 0

Sample:


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE
0,FLD001,2026,2026-05-11
1,FLD001,2026,2026-05-25
2,FLD001,2026,2026-06-08
3,FLD001,2026,2026-06-22
4,FLD001,2026,2026-07-06
5,FLD001,2026,2026-07-20
6,FLD002,2026,2026-05-11
7,FLD002,2026,2026-05-25
8,FLD002,2026,2026-06-08
9,FLD002,2026,2026-06-22


In [76]:
query = """
SELECT
    FIELD_ID,
    SEASON_YEAR,
    SNAPSHOT_DATE,
    TYPEOF(SNAPSHOT_DATE) AS SNAPSHOT_DATE_TYPE
FROM YIELD_FORECAST_DB.POD_VERTEX.GOLD_PEST_FEATURE_SNAPSHOT
LIMIT 20
"""

pest_date_raw = pd.read_sql(query, conn)

print("Pest SNAPSHOT_DATE source:")
display(pest_date_raw)

/tmp/ipykernel_75164/610031978.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pest_date_raw = pd.read_sql(query, conn)


Pest SNAPSHOT_DATE source:


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SNAPSHOT_DATE_TYPE
0,FLD001,2026,1777852800000000,INTEGER
1,FLD001,2026,1779062400000000,INTEGER
2,FLD001,2026,1780272000000000,INTEGER
3,FLD001,2026,1782691200000000,INTEGER
4,FLD001,2026,1785110400000000,INTEGER
5,FLD002,2026,1777852800000000,INTEGER
6,FLD002,2026,1779062400000000,INTEGER
7,FLD002,2026,1782691200000000,INTEGER
8,FLD002,2026,1783900800000000,INTEGER
9,FLD002,2026,1788998400000000,INTEGER


In [77]:
pest_date_check = pest_date_raw.copy()

pest_date_check["SNAPSHOT_DATE_CONVERTED"] = pd.to_datetime(
    pest_date_check["SNAPSHOT_DATE"],
    unit="us",
    errors="coerce"
)

print("Converted Pest snapshot dates:")

display(
    pest_date_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SNAPSHOT_DATE_CONVERTED"
        ]
    ]
)

Converted Pest snapshot dates:


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SNAPSHOT_DATE_CONVERTED
0,FLD001,2026,1777852800000000,2026-05-04
1,FLD001,2026,1779062400000000,2026-05-18
2,FLD001,2026,1780272000000000,2026-06-01
3,FLD001,2026,1782691200000000,2026-06-29
4,FLD001,2026,1785110400000000,2026-07-27
5,FLD002,2026,1777852800000000,2026-05-04
6,FLD002,2026,1779062400000000,2026-05-18
7,FLD002,2026,1782691200000000,2026-06-29
8,FLD002,2026,1783900800000000,2026-07-13
9,FLD002,2026,1788998400000000,2026-09-10


In [78]:
pest_dates = pest_date_check[
    ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE_CONVERTED"]
].copy()

pest_dates["SEASON_YEAR"] = pd.to_numeric(
    pest_dates["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

pest_dates["SNAPSHOT_DATE_CONVERTED"] = pd.to_datetime(
    pest_dates["SNAPSHOT_DATE_CONVERTED"],
    errors="coerce"
)

pest_dates["SEASON_DAY_FROM_FIRST_OBS"] = (
    pest_dates.groupby(
        ["FIELD_ID", "SEASON_YEAR"]
    )["SNAPSHOT_DATE_CONVERTED"]
    .transform(
        lambda x: (x - x.min()).dt.days + 1
    )
)

print("Pest temporal coverage:")
print("Rows:", len(pest_dates))

print(
    "Unique field-seasons:",
    pest_dates[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print(
    "\nObservation interval distribution:"
)

print(
    pest_dates
    .sort_values(
        ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE_CONVERTED"]
    )
    .groupby(
        ["FIELD_ID", "SEASON_YEAR"]
    )["SNAPSHOT_DATE_CONVERTED"]
    .diff()
    .dt.days
    .value_counts()
    .sort_index()
)

display(
    pest_dates.sort_values(
        ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE_CONVERTED"]
    ).head(30)
)

Pest temporal coverage:
Rows: 20
Unique field-seasons: 4

Observation interval distribution:
SNAPSHOT_DATE_CONVERTED
14.0    12
28.0     2
42.0     1
59.0     1
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,SEASON_DAY_FROM_FIRST_OBS
0,FLD001,2026,2026-05-04,1
1,FLD001,2026,2026-05-18,15
2,FLD001,2026,2026-06-01,29
3,FLD001,2026,2026-06-29,57
4,FLD001,2026,2026-07-27,85
5,FLD002,2026,2026-05-04,1
6,FLD002,2026,2026-05-18,15
7,FLD002,2026,2026-06-29,57
8,FLD002,2026,2026-07-13,71
9,FLD002,2026,2026-09-10,130


In [79]:
weather_dates = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SNAPSHOT_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
    """,
    conn
)

weather_dates["SEASON_YEAR"] = pd.to_numeric(
    weather_dates["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

weather_dates["SNAPSHOT_DATE"] = pd.to_datetime(
    weather_dates["SNAPSHOT_DATE"],
    errors="coerce"
)

pest_weather_match = pest_dates.merge(
    weather_dates,
    left_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "SNAPSHOT_DATE_CONVERTED"
    ],
    right_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "SNAPSHOT_DATE"
    ],
    how="left",
    indicator=True
)

print("Total Pest observations:", len(pest_weather_match))

print("\nExact Weather-date matches:")
print(
    pest_weather_match["_merge"]
    .value_counts()
)

display(
    pest_weather_match[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "SNAPSHOT_DATE",
            "_merge"
        ]
    ]
)

Total Pest observations: 20

Exact Weather-date matches:
_merge
both          19
left_only      1
right_only     0
Name: count, dtype: int64


/tmp/ipykernel_75164/1643319332.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_dates = pd.read_sql(


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,SNAPSHOT_DATE,_merge
0,FLD001,2026,2026-05-04,2026-05-04,both
1,FLD001,2026,2026-05-18,2026-05-18,both
2,FLD001,2026,2026-06-01,2026-06-01,both
3,FLD001,2026,2026-06-29,2026-06-29,both
4,FLD001,2026,2026-07-27,2026-07-27,both
5,FLD002,2026,2026-05-04,2026-05-04,both
6,FLD002,2026,2026-05-18,2026-05-18,both
7,FLD002,2026,2026-06-29,2026-06-29,both
8,FLD002,2026,2026-07-13,2026-07-13,both
9,FLD002,2026,2026-09-10,NaT,left_only


In [80]:
# Load the authoritative seasonal timeline
season_timeline = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
    """,
    conn
)

season_timeline["SEASON_YEAR"] = pd.to_numeric(
    season_timeline["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

season_timeline["WEEK_INDEX"] = pd.to_numeric(
    season_timeline["WEEK_INDEX"],
    errors="coerce"
).astype("Int64")

# Keep one seasonal row per field-season-week
season_timeline = season_timeline[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY"
    ]
].drop_duplicates()

# Attach the authoritative seasonal week to Pest by exact calendar date
pest_week_check = pest_dates.merge(
    season_timeline,
    left_on=[
        "FIELD_ID",
        "SEASON_YEAR"
    ],
    right_on=[
        "FIELD_ID",
        "SEASON_YEAR"
    ],
    how="left"
)

print("Pest rows:", len(pest_week_check))
print(
    "Pest rows with seasonal timeline available:",
    pest_week_check["WEEK_INDEX"].notna().sum()
)

display(
    pest_week_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "WEEK_INDEX",
            "SEASON_DAY"
        ]
    ].head(30)
)

/tmp/ipykernel_75164/681208235.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_timeline = pd.read_sql(


Pest rows: 260
Pest rows with seasonal timeline available: 260


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,WEEK_INDEX,SEASON_DAY
0,FLD001,2026,2026-05-04,0,3
1,FLD001,2026,2026-05-04,1,10
2,FLD001,2026,2026-05-04,10,73
3,FLD001,2026,2026-05-04,11,80
4,FLD001,2026,2026-05-04,12,87
5,FLD001,2026,2026-05-04,2,17
6,FLD001,2026,2026-05-04,3,24
7,FLD001,2026,2026-05-04,4,31
8,FLD001,2026,2026-05-04,5,38
9,FLD001,2026,2026-05-04,6,45


In [81]:
# Load authoritative season start dates
season_start = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SEASON_START_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
    """,
    conn
)

season_start["SEASON_YEAR"] = pd.to_numeric(
    season_start["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

season_start["SEASON_START_DATE"] = pd.to_datetime(
    season_start["SEASON_START_DATE"],
    errors="coerce"
)

# Build authoritative seasonal calendar
season_calendar = season_timeline.merge(
    season_start,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

season_calendar["EXPECTED_DATE"] = (
    season_calendar["SEASON_START_DATE"]
    + pd.to_timedelta(
        season_calendar["SEASON_DAY"] - 1,
        unit="D"
    )
)

# Match Pest observations to exact seasonal calendar dates
pest_week_check = pest_dates.merge(
    season_calendar[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_DATE"
        ]
    ],
    left_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "SNAPSHOT_DATE_CONVERTED"
    ],
    right_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "EXPECTED_DATE"
    ],
    how="left"
)

print("Pest observations:", len(pest_week_check))

print(
    "Pest observations mapped to WEEK_INDEX:",
    pest_week_check["WEEK_INDEX"].notna().sum()
)

print(
    "Pest observations without WEEK_INDEX:",
    pest_week_check["WEEK_INDEX"].isna().sum()
)

display(
    pest_week_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_DATE"
        ]
    ]
)

/tmp/ipykernel_75164/2038944006.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_start = pd.read_sql(


Pest observations: 20
Pest observations mapped to WEEK_INDEX: 0
Pest observations without WEEK_INDEX: 20


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,WEEK_INDEX,SEASON_DAY,EXPECTED_DATE
0,FLD001,2026,2026-05-04,<NA>,NaN,NaT
1,FLD001,2026,2026-05-18,<NA>,NaN,NaT
2,FLD001,2026,2026-06-01,<NA>,NaN,NaT
3,FLD001,2026,2026-06-29,<NA>,NaN,NaT
4,FLD001,2026,2026-07-27,<NA>,NaN,NaT
5,FLD002,2026,2026-05-04,<NA>,NaN,NaT
6,FLD002,2026,2026-05-18,<NA>,NaN,NaT
7,FLD002,2026,2026-06-29,<NA>,NaN,NaT
8,FLD002,2026,2026-07-13,<NA>,NaN,NaT
9,FLD002,2026,2026-09-10,<NA>,NaN,NaT


In [82]:
pest_field_seasons = pest_dates[
    ["FIELD_ID", "SEASON_YEAR"]
].drop_duplicates()

pest_context_check = pest_field_seasons.merge(
    season_start,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

print("Unique Pest field-seasons:", len(pest_field_seasons))
print(
    "Field-seasons with SEASON_START_DATE:",
    pest_context_check["SEASON_START_DATE"].notna().sum()
)
print(
    "Field-seasons without SEASON_START_DATE:",
    pest_context_check["SEASON_START_DATE"].isna().sum()
)

display(
    pest_context_check.sort_values(
        ["FIELD_ID", "SEASON_YEAR"]
    )
)

Unique Pest field-seasons: 4
Field-seasons with SEASON_START_DATE: 3
Field-seasons without SEASON_START_DATE: 1


,FIELD_ID,SEASON_YEAR,SEASON_START_DATE
0,FLD001,2026,NaT
1,FLD002,2026,2026-04-27
2,FLD003,2026,2026-05-10
3,FLD004,2023,2023-04-20


In [83]:
# Keep only Pest field-seasons with an authoritative season start
valid_pest_context = pest_context_check[
    pest_context_check["SEASON_START_DATE"].notna()
].copy()

# Build seasonal calendar only for those field-seasons
pest_season_calendar = season_timeline.merge(
    valid_pest_context,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

pest_season_calendar["EXPECTED_DATE"] = (
    pest_season_calendar["SEASON_START_DATE"]
    + pd.to_timedelta(
        pest_season_calendar["SEASON_DAY"] - 1,
        unit="D"
    )
)

# Map Pest observations to the exact expected seasonal date
pest_week_check_valid = pest_dates.merge(
    pest_season_calendar[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_DATE"
        ]
    ],
    left_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "SNAPSHOT_DATE_CONVERTED"
    ],
    right_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "EXPECTED_DATE"
    ],
    how="left"
)

print("Pest observations from valid field-seasons:",
      len(pest_week_check_valid))

print(
    "Mapped to WEEK_INDEX:",
    pest_week_check_valid["WEEK_INDEX"].notna().sum()
)

print(
    "Unmapped:",
    pest_week_check_valid["WEEK_INDEX"].isna().sum()
)

display(
    pest_week_check_valid[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_DATE"
        ]
    ]
)

Pest observations from valid field-seasons: 20
Mapped to WEEK_INDEX: 0
Unmapped: 20


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,WEEK_INDEX,SEASON_DAY,EXPECTED_DATE
0,FLD001,2026,2026-05-04,<NA>,NaN,NaT
1,FLD001,2026,2026-05-18,<NA>,NaN,NaT
2,FLD001,2026,2026-06-01,<NA>,NaN,NaT
3,FLD001,2026,2026-06-29,<NA>,NaN,NaT
4,FLD001,2026,2026-07-27,<NA>,NaN,NaT
5,FLD002,2026,2026-05-04,<NA>,NaN,NaT
6,FLD002,2026,2026-05-18,<NA>,NaN,NaT
7,FLD002,2026,2026-06-29,<NA>,NaN,NaT
8,FLD002,2026,2026-07-13,<NA>,NaN,NaT
9,FLD002,2026,2026-09-10,<NA>,NaN,NaT


In [84]:
pest_dap_check = pest_dates.merge(
    valid_pest_context[
        ["FIELD_ID", "SEASON_YEAR", "SEASON_START_DATE"]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

pest_dap_check["PEST_DAP"] = (
    pest_dap_check["SNAPSHOT_DATE_CONVERTED"]
    - pest_dap_check["SEASON_START_DATE"]
).dt.days + 1

display(
    pest_dap_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "SEASON_START_DATE",
            "PEST_DAP"
        ]
    ]
)

,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,SEASON_START_DATE,PEST_DAP
0,FLD001,2026,2026-05-04,NaT,NaN
1,FLD001,2026,2026-05-18,NaT,NaN
2,FLD001,2026,2026-06-01,NaT,NaN
3,FLD001,2026,2026-06-29,NaT,NaN
4,FLD001,2026,2026-07-27,NaT,NaN
5,FLD002,2026,2026-05-04,2026-04-27,8.0
6,FLD002,2026,2026-05-18,2026-04-27,22.0
7,FLD002,2026,2026-06-29,2026-04-27,64.0
8,FLD002,2026,2026-07-13,2026-04-27,78.0
9,FLD002,2026,2026-09-10,2026-04-27,137.0


In [85]:
# Create a unique seasonal-day reference
season_day_reference = (
    season_timeline[
        ["WEEK_INDEX", "SEASON_DAY"]
    ]
    .drop_duplicates()
    .sort_values("SEASON_DAY")
)

# Keep only observations where DAP can be calculated
pest_alignment = pest_dap_check[
    pest_dap_check["PEST_DAP"].notna()
].copy()

def map_dap_to_week(dap):
    if pd.isna(dap):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    distances = (
        season_day_reference["SEASON_DAY"] - dap
    ).abs()

    nearest_idx = distances.idxmin()

    return pd.Series([
        season_day_reference.loc[nearest_idx, "WEEK_INDEX"],
        season_day_reference.loc[nearest_idx, "SEASON_DAY"],
        distances.loc[nearest_idx]
    ])

pest_alignment[
    ["MAPPED_WEEK_INDEX", "MAPPED_SEASON_DAY", "DAP_DISTANCE"]
] = pest_alignment["PEST_DAP"].apply(map_dap_to_week)

display(
    pest_alignment[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "SEASON_START_DATE",
            "PEST_DAP",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY",
            "DAP_DISTANCE"
        ]
    ]
)

,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,SEASON_START_DATE,PEST_DAP,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY,DAP_DISTANCE
5,FLD002,2026,2026-05-04,2026-04-27,8.0,1.0,10.0,2.0
6,FLD002,2026,2026-05-18,2026-04-27,22.0,3.0,24.0,2.0
7,FLD002,2026,2026-06-29,2026-04-27,64.0,9.0,66.0,2.0
8,FLD002,2026,2026-07-13,2026-04-27,78.0,11.0,80.0,2.0
9,FLD002,2026,2026-09-10,2026-04-27,137.0,12.0,87.0,50.0
10,FLD003,2026,2026-05-04,2026-05-10,-5.0,0.0,3.0,8.0
11,FLD003,2026,2026-05-18,2026-05-10,9.0,1.0,10.0,1.0
12,FLD003,2026,2026-06-01,2026-05-10,23.0,3.0,24.0,1.0
13,FLD003,2026,2026-06-15,2026-05-10,37.0,5.0,38.0,1.0
14,FLD003,2026,2026-06-29,2026-05-10,51.0,7.0,52.0,1.0


In [86]:
print("Total Pest observations with season start:",
      len(pest_alignment))

print("\nDAP distance distribution:")
print(
    pest_alignment["DAP_DISTANCE"]
    .value_counts()
    .sort_index()
)

print("\nObservations by alignment quality:")

pest_alignment["ALIGNMENT_STATUS"] = np.select(
    [
        pest_alignment["PEST_DAP"] < 1,
        pest_alignment["DAP_DISTANCE"] <= 2,
        pest_alignment["DAP_DISTANCE"] <= 7,
        pest_alignment["DAP_DISTANCE"] > 7
    ],
    [
        "BEFORE_SEASON_START",
        "CLOSE_MATCH",
        "ACCEPTABLE_WINDOW",
        "OUTSIDE_ALIGNMENT_WINDOW"
    ],
    default="UNRESOLVED"
)

print(
    pest_alignment["ALIGNMENT_STATUS"]
    .value_counts()
)

display(
    pest_alignment[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE_CONVERTED",
            "PEST_DAP",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY",
            "DAP_DISTANCE",
            "ALIGNMENT_STATUS"
        ]
    ]
)

Total Pest observations with season start: 15

DAP distance distribution:
DAP_DISTANCE
1.0     5
2.0     8
8.0     1
50.0    1
Name: count, dtype: int64

Observations by alignment quality:
ALIGNMENT_STATUS
CLOSE_MATCH                 13
OUTSIDE_ALIGNMENT_WINDOW     1
BEFORE_SEASON_START          1
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE_CONVERTED,PEST_DAP,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY,DAP_DISTANCE,ALIGNMENT_STATUS
5,FLD002,2026,2026-05-04,8.0,1.0,10.0,2.0,CLOSE_MATCH
6,FLD002,2026,2026-05-18,22.0,3.0,24.0,2.0,CLOSE_MATCH
7,FLD002,2026,2026-06-29,64.0,9.0,66.0,2.0,CLOSE_MATCH
8,FLD002,2026,2026-07-13,78.0,11.0,80.0,2.0,CLOSE_MATCH
9,FLD002,2026,2026-09-10,137.0,12.0,87.0,50.0,OUTSIDE_ALIGNMENT_WINDOW
10,FLD003,2026,2026-05-04,-5.0,0.0,3.0,8.0,BEFORE_SEASON_START
11,FLD003,2026,2026-05-18,9.0,1.0,10.0,1.0,CLOSE_MATCH
12,FLD003,2026,2026-06-01,23.0,3.0,24.0,1.0,CLOSE_MATCH
13,FLD003,2026,2026-06-15,37.0,5.0,38.0,1.0,CLOSE_MATCH
14,FLD003,2026,2026-06-29,51.0,7.0,52.0,1.0,CLOSE_MATCH


In [87]:
disease_dates = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SNAPSHOT_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.GOLD_DISEASE_FEATURE_SNAPSHOT
    ORDER BY FIELD_ID, SEASON_YEAR, SNAPSHOT_DATE
    """,
    conn
)

print("Disease rows:", len(disease_dates))
print("Disease columns:", disease_dates.columns.tolist())
print("SNAPSHOT_DATE dtype:", disease_dates["SNAPSHOT_DATE"].dtype)

display(disease_dates.head(20))

/tmp/ipykernel_75164/3571915765.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  disease_dates = pd.read_sql(


Disease rows: 232
Disease columns: ['FIELD_ID', 'SEASON_YEAR', 'SNAPSHOT_DATE']
SNAPSHOT_DATE dtype: object


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE
0,FLD001,2026,2026-05-11
1,FLD001,2026,2026-05-25
2,FLD001,2026,2026-06-08
3,FLD001,2026,2026-06-22
4,FLD001,2026,2026-07-06
5,FLD001,2026,2026-07-20
6,FLD002,2026,2026-05-11
7,FLD002,2026,2026-05-25
8,FLD002,2026,2026-06-08
9,FLD002,2026,2026-06-22


In [88]:
disease_dates["SEASON_YEAR"] = pd.to_numeric(
    disease_dates["SEASON_YEAR"],
    errors="coerce"
).astype("Int64")

disease_dates["SNAPSHOT_DATE"] = pd.to_datetime(
    disease_dates["SNAPSHOT_DATE"],
    errors="coerce"
)

disease_dap_check = disease_dates.merge(
    season_start[
        ["FIELD_ID", "SEASON_YEAR", "SEASON_START_DATE"]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left"
)

disease_dap_check["DISEASE_DAP"] = (
    disease_dap_check["SNAPSHOT_DATE"]
    - disease_dap_check["SEASON_START_DATE"]
).dt.days + 1

print("Disease observations:", len(disease_dap_check))

print(
    "Observations with season start:",
    disease_dap_check["SEASON_START_DATE"].notna().sum()
)

print(
    "Observations without season start:",
    disease_dap_check["SEASON_START_DATE"].isna().sum()
)

display(
    disease_dap_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "DISEASE_DAP"
        ]
    ].head(30)
)

Disease observations: 232
Observations with season start: 192
Observations without season start: 40


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,DISEASE_DAP
0,FLD001,2026,2026-05-11,NaT,NaN
1,FLD001,2026,2026-05-25,NaT,NaN
2,FLD001,2026,2026-06-08,NaT,NaN
3,FLD001,2026,2026-06-22,NaT,NaN
4,FLD001,2026,2026-07-06,NaT,NaN
5,FLD001,2026,2026-07-20,NaT,NaN
6,FLD002,2026,2026-05-11,2026-04-27,15.0
7,FLD002,2026,2026-05-25,2026-04-27,29.0
8,FLD002,2026,2026-06-08,2026-04-27,43.0
9,FLD002,2026,2026-06-22,2026-04-27,57.0


In [89]:
# Use only Disease observations with an authoritative season start
disease_alignment = disease_dap_check[
    disease_dap_check["DISEASE_DAP"].notna()
].copy()

# Reuse the established seasonal-day reference
season_day_reference = (
    season_timeline[
        ["WEEK_INDEX", "SEASON_DAY"]
    ]
    .drop_duplicates()
    .sort_values("SEASON_DAY")
)

def map_disease_dap_to_week(dap):
    if pd.isna(dap):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    distances = (
        season_day_reference["SEASON_DAY"] - dap
    ).abs()

    nearest_idx = distances.idxmin()

    return pd.Series([
        season_day_reference.loc[nearest_idx, "WEEK_INDEX"],
        season_day_reference.loc[nearest_idx, "SEASON_DAY"],
        distances.loc[nearest_idx]
    ])

disease_alignment[
    ["MAPPED_WEEK_INDEX", "MAPPED_SEASON_DAY", "DAP_DISTANCE"]
] = disease_alignment["DISEASE_DAP"].apply(
    map_disease_dap_to_week
)

print("Disease observations with season start:",
      len(disease_alignment))

print("\nDAP distance distribution:")
print(
    disease_alignment["DAP_DISTANCE"]
    .value_counts()
    .sort_index()
)

display(
    disease_alignment[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "DISEASE_DAP",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY",
            "DAP_DISTANCE"
        ]
    ].head(50)
)

Disease observations with season start: 192

DAP distance distribution:
DAP_DISTANCE
0.0     51
1.0     36
2.0     49
3.0     51
4.0      1
7.0      1
9.0      2
12.0     1
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,DISEASE_DAP,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY,DAP_DISTANCE
6,FLD002,2026,2026-05-11,2026-04-27,15.0,2.0,17.0,2.0
7,FLD002,2026,2026-05-25,2026-04-27,29.0,4.0,31.0,2.0
8,FLD002,2026,2026-06-08,2026-04-27,43.0,6.0,45.0,2.0
9,FLD002,2026,2026-06-22,2026-04-27,57.0,8.0,59.0,2.0
10,FLD002,2026,2026-07-06,2026-04-27,71.0,10.0,73.0,2.0
11,FLD002,2026,2026-07-20,2026-04-27,85.0,12.0,87.0,2.0
12,FLD003,2026,2026-05-11,2026-05-10,2.0,0.0,3.0,1.0
13,FLD003,2026,2026-05-25,2026-05-10,16.0,2.0,17.0,1.0
14,FLD003,2026,2026-06-08,2026-05-10,30.0,4.0,31.0,1.0
15,FLD003,2026,2026-06-22,2026-05-10,44.0,6.0,45.0,1.0


In [90]:
disease_alignment["ALIGNMENT_STATUS"] = np.select(
    [
        disease_alignment["DISEASE_DAP"] < 1,
        disease_alignment["DAP_DISTANCE"] <= 3,
        disease_alignment["DAP_DISTANCE"] > 3
    ],
    [
        "BEFORE_SEASON_START",
        "CLOSE_MATCH",
        "OUTSIDE_ALIGNMENT_WINDOW"
    ],
    default="UNRESOLVED"
)

# Add the 40 observations with no authoritative season start
disease_full_alignment = disease_dap_check.copy()

disease_full_alignment["ALIGNMENT_STATUS"] = "NO_SEASON_START"

disease_full_alignment.loc[
    disease_full_alignment["DISEASE_DAP"].notna(),
    "ALIGNMENT_STATUS"
] = disease_alignment["ALIGNMENT_STATUS"].values

print("All Disease observations:", len(disease_full_alignment))

print("\nDisease alignment status:")
print(
    disease_full_alignment["ALIGNMENT_STATUS"]
    .value_counts()
)

display(
    disease_full_alignment[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "DISEASE_DAP",
            "ALIGNMENT_STATUS"
        ]
    ]
)

All Disease observations: 232

Disease alignment status:
ALIGNMENT_STATUS
CLOSE_MATCH                 186
NO_SEASON_START              40
BEFORE_SEASON_START           4
OUTSIDE_ALIGNMENT_WINDOW      2
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,DISEASE_DAP,ALIGNMENT_STATUS
0,FLD001,2026,2026-05-11,NaT,NaN,NO_SEASON_START
1,FLD001,2026,2026-05-25,NaT,NaN,NO_SEASON_START
2,FLD001,2026,2026-06-08,NaT,NaN,NO_SEASON_START
3,FLD001,2026,2026-06-22,NaT,NaN,NO_SEASON_START
4,FLD001,2026,2026-07-06,NaT,NaN,NO_SEASON_START
5,FLD001,2026,2026-07-20,NaT,NaN,NO_SEASON_START
6,FLD002,2026,2026-05-11,2026-04-27,15.0,CLOSE_MATCH
7,FLD002,2026,2026-05-25,2026-04-27,29.0,CLOSE_MATCH
8,FLD002,2026,2026-06-08,2026-04-27,43.0,CLOSE_MATCH
9,FLD002,2026,2026-06-22,2026-04-27,57.0,CLOSE_MATCH


In [91]:
# Map Disease observations to the nearest established seasonal WEEK_INDEX

disease_timeline = pd.read_sql(
    """
    SELECT DISTINCT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
    ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
    """,
    conn
)

# Keep only observations that passed the Disease alignment rule
disease_valid = disease_full_alignment[
    disease_full_alignment["ALIGNMENT_STATUS"] == "CLOSE_MATCH"
].copy()

# For each disease observation, find the nearest established seasonal day
disease_valid["DAP_DISTANCE"] = np.nan
disease_valid["MAPPED_WEEK_INDEX"] = np.nan
disease_valid["MAPPED_SEASON_DAY"] = np.nan

for idx, row in disease_valid.iterrows():

    candidates = disease_timeline[
        (disease_timeline["FIELD_ID"] == row["FIELD_ID"]) &
        (disease_timeline["SEASON_YEAR"] == row["SEASON_YEAR"])
    ].copy()

    if candidates.empty:
        continue

    candidates["DISTANCE"] = (
        candidates["SEASON_DAY"] - row["DISEASE_DAP"]
    ).abs()

    nearest = candidates.loc[candidates["DISTANCE"].idxmin()]

    disease_valid.loc[idx, "DAP_DISTANCE"] = nearest["DISTANCE"]
    disease_valid.loc[idx, "MAPPED_WEEK_INDEX"] = nearest["WEEK_INDEX"]
    disease_valid.loc[idx, "MAPPED_SEASON_DAY"] = nearest["SEASON_DAY"]

print("Valid Disease observations:", len(disease_valid))
print("\nMapped WEEK_INDEX distribution:")
print(disease_valid["MAPPED_WEEK_INDEX"].value_counts().sort_index())

display(
    disease_valid[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "DISEASE_DAP",
            "DAP_DISTANCE",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY"
        ]
    ].head(30)
)


/tmp/ipykernel_75164/3901864142.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  disease_timeline = pd.read_sql(


Valid Disease observations: 186

Mapped WEEK_INDEX distribution:
MAPPED_WEEK_INDEX
0.0     10
1.0     16
2.0     14
3.0     18
4.0     17
5.0     14
6.0     16
7.0     15
8.0     19
9.0     11
10.0    19
11.0     9
12.0     8
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,DISEASE_DAP,DAP_DISTANCE,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY
6,FLD002,2026,2026-05-11,15.0,2.0,2.0,17.0
7,FLD002,2026,2026-05-25,29.0,2.0,4.0,31.0
8,FLD002,2026,2026-06-08,43.0,2.0,6.0,45.0
9,FLD002,2026,2026-06-22,57.0,2.0,8.0,59.0
10,FLD002,2026,2026-07-06,71.0,2.0,10.0,73.0
11,FLD002,2026,2026-07-20,85.0,2.0,12.0,87.0
12,FLD003,2026,2026-05-11,2.0,1.0,0.0,3.0
13,FLD003,2026,2026-05-25,16.0,1.0,2.0,17.0
14,FLD003,2026,2026-06-08,30.0,1.0,4.0,31.0
15,FLD003,2026,2026-06-22,44.0,1.0,6.0,45.0


In [92]:
# Correct Disease mapping using the canonical seasonal calendar
# WEEK_INDEX 0–12 -> SEASON_DAY 3, 10, 17, ..., 87

canonical_seasonal_calendar = pd.DataFrame({
    "WEEK_INDEX": range(13)
})

canonical_seasonal_calendar["SEASON_DAY"] = (
    3 + canonical_seasonal_calendar["WEEK_INDEX"] * 7
)

disease_valid = disease_full_alignment[
    disease_full_alignment["ALIGNMENT_STATUS"] == "CLOSE_MATCH"
].copy()

# Find nearest canonical seasonal day
disease_valid["MAPPED_WEEK_INDEX"] = np.nan
disease_valid["MAPPED_SEASON_DAY"] = np.nan
disease_valid["DAP_DISTANCE"] = np.nan

for idx, row in disease_valid.iterrows():

    candidates = canonical_seasonal_calendar.copy()

    candidates["DISTANCE"] = (
        candidates["SEASON_DAY"] - row["DISEASE_DAP"]
    ).abs()

    nearest = candidates.loc[candidates["DISTANCE"].idxmin()]

    disease_valid.loc[idx, "MAPPED_WEEK_INDEX"] = nearest["WEEK_INDEX"]
    disease_valid.loc[idx, "MAPPED_SEASON_DAY"] = nearest["SEASON_DAY"]
    disease_valid.loc[idx, "DAP_DISTANCE"] = nearest["DISTANCE"]

print("Valid Disease observations:", len(disease_valid))

print("\nMapped WEEK_INDEX distribution:")
print(
    disease_valid["MAPPED_WEEK_INDEX"]
    .value_counts()
    .sort_index()
)

print("\nMaximum mapping distance:")
print(disease_valid["DAP_DISTANCE"].max())

display(
    disease_valid[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "DISEASE_DAP",
            "DAP_DISTANCE",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY"
        ]
    ].head(30)
)

Valid Disease observations: 186

Mapped WEEK_INDEX distribution:
MAPPED_WEEK_INDEX
0.0     11
1.0     13
2.0     18
3.0     13
4.0     18
5.0     14
6.0     19
7.0     14
8.0     18
9.0     12
10.0    19
11.0    10
12.0     7
Name: count, dtype: int64

Maximum mapping distance:
3.0


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,DISEASE_DAP,DAP_DISTANCE,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY
6,FLD002,2026,2026-05-11,15.0,2.0,2.0,17.0
7,FLD002,2026,2026-05-25,29.0,2.0,4.0,31.0
8,FLD002,2026,2026-06-08,43.0,2.0,6.0,45.0
9,FLD002,2026,2026-06-22,57.0,2.0,8.0,59.0
10,FLD002,2026,2026-07-06,71.0,2.0,10.0,73.0
11,FLD002,2026,2026-07-20,85.0,2.0,12.0,87.0
12,FLD003,2026,2026-05-11,2.0,1.0,0.0,3.0
13,FLD003,2026,2026-05-25,16.0,1.0,2.0,17.0
14,FLD003,2026,2026-06-08,30.0,1.0,4.0,31.0
15,FLD003,2026,2026-06-22,44.0,1.0,6.0,45.0


In [94]:
# Reload the original Disease feature values
disease_features = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SNAPSHOT_DATE,
        DOMINANT_DISEASE,
        MAX_DISEASE_RISK,
        MAX_DISEASE_RISK_TREND,
        MAX_DISEASE_WEATHER_SUITABILITY,
        DISEASE_TIMING_DEVIATION,
        DISEASE_DATA_AGE
    FROM YIELD_FORECAST_DB.POD_VERTEX.GOLD_DISEASE_FEATURE_SNAPSHOT
    """,
    conn
)

# Make sure dates have the same type
disease_features["SNAPSHOT_DATE"] = pd.to_datetime(
    disease_features["SNAPSHOT_DATE"],
    errors="coerce"
)

disease_valid["SNAPSHOT_DATE"] = pd.to_datetime(
    disease_valid["SNAPSHOT_DATE"],
    errors="coerce"
)

# Join the actual Disease feature values back to the validated alignment
disease_aligned = disease_valid.merge(
    disease_features,
    on=["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"],
    how="left",
    validate="one_to_one"
)

print("Disease aligned rows:", len(disease_aligned))

print("\nMissing Disease feature rows after join:")
print(
    disease_aligned[
        [
            "DOMINANT_DISEASE",
            "MAX_DISEASE_RISK",
            "MAX_DISEASE_RISK_TREND",
            "MAX_DISEASE_WEATHER_SUITABILITY",
            "DISEASE_TIMING_DEVIATION",
            "DISEASE_DATA_AGE"
        ]
    ].isna().all(axis=1).sum()
)

display(
    disease_aligned[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "DISEASE_DAP",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY",
            "DAP_DISTANCE",
            "DOMINANT_DISEASE",
            "MAX_DISEASE_RISK",
            "MAX_DISEASE_RISK_TREND",
            "MAX_DISEASE_WEATHER_SUITABILITY",
            "DISEASE_TIMING_DEVIATION",
            "DISEASE_DATA_AGE"
        ]
    ].head(20)
)

/tmp/ipykernel_75164/2869556262.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  disease_features = pd.read_sql(


Disease aligned rows: 186

Missing Disease feature rows after join:
0


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,DISEASE_DAP,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY,DAP_DISTANCE,DOMINANT_DISEASE,MAX_DISEASE_RISK,MAX_DISEASE_RISK_TREND,MAX_DISEASE_WEATHER_SUITABILITY,DISEASE_TIMING_DEVIATION,DISEASE_DATA_AGE
0,FLD002,2026,2026-05-11,15.0,2.0,17.0,2.0,Tar Spot,1,NaN,None,-27.0,135
1,FLD002,2026,2026-05-25,29.0,4.0,31.0,2.0,Gray Leaf Spot,1,0.0,None,-13.0,121
2,FLD002,2026,2026-06-08,43.0,6.0,45.0,2.0,Gray Leaf Spot,1,0.0,None,1.0,107
3,FLD002,2026,2026-06-22,57.0,8.0,59.0,2.0,Tar Spot,1,0.0,None,15.0,93
4,FLD002,2026,2026-07-06,71.0,10.0,73.0,2.0,Tar Spot,1,0.0,None,29.0,79
5,FLD002,2026,2026-07-20,85.0,12.0,87.0,2.0,Tar Spot,1,0.0,None,43.0,65
6,FLD003,2026,2026-05-11,2.0,0.0,3.0,1.0,White Mold (Sclerotinia),1,NaN,None,-40.0,135
7,FLD003,2026,2026-05-25,16.0,2.0,17.0,1.0,Sudden Death Syndrome (SDS),1,0.0,None,-26.0,121
8,FLD003,2026,2026-06-08,30.0,4.0,31.0,1.0,White Mold (Sclerotinia),1,0.0,None,-12.0,107
9,FLD003,2026,2026-06-22,44.0,6.0,45.0,1.0,Sudden Death Syndrome (SDS),1,0.0,None,2.0,93


In [95]:
# Finalize Disease dataset for forecast integration

disease_aligned = disease_aligned[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "SNAPSHOT_DATE",
        "DISEASE_DAP",
        "MAPPED_WEEK_INDEX",
        "MAPPED_SEASON_DAY",
        "DAP_DISTANCE",
        "DOMINANT_DISEASE",
        "MAX_DISEASE_RISK",
        "MAX_DISEASE_RISK_TREND",
        "MAX_DISEASE_WEATHER_SUITABILITY",
        "DISEASE_TIMING_DEVIATION",
        "DISEASE_DATA_AGE"
    ]
].copy()

disease_aligned = disease_aligned.rename(
    columns={
        "MAPPED_WEEK_INDEX": "WEEK_INDEX",
        "MAPPED_SEASON_DAY": "SEASON_DAY"
    }
)

# Enforce semantic types
disease_aligned["SEASON_YEAR"] = disease_aligned["SEASON_YEAR"].astype(int)
disease_aligned["WEEK_INDEX"] = disease_aligned["WEEK_INDEX"].astype(int)
disease_aligned["SEASON_DAY"] = disease_aligned["SEASON_DAY"].astype(int)

print("Disease aligned rows:", len(disease_aligned))

print(
    "Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX combinations:",
    disease_aligned[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    ].drop_duplicates().shape[0]
)

print("\nMissing WEEK_INDEX:")
print(disease_aligned["WEEK_INDEX"].isna().sum())

print("\nMaximum DAP alignment distance:")
print(disease_aligned["DAP_DISTANCE"].max())

print("\nDisease observation count by WEEK_INDEX:")
print(
    disease_aligned["WEEK_INDEX"]
    .value_counts()
    .sort_index()
)

display(disease_aligned.head(20))

Disease aligned rows: 186
Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX combinations: 186

Missing WEEK_INDEX:
0

Maximum DAP alignment distance:
3.0

Disease observation count by WEEK_INDEX:
WEEK_INDEX
0     11
1     13
2     18
3     13
4     18
5     14
6     19
7     14
8     18
9     12
10    19
11    10
12     7
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,DISEASE_DAP,WEEK_INDEX,SEASON_DAY,DAP_DISTANCE,DOMINANT_DISEASE,MAX_DISEASE_RISK,MAX_DISEASE_RISK_TREND,MAX_DISEASE_WEATHER_SUITABILITY,DISEASE_TIMING_DEVIATION,DISEASE_DATA_AGE
0,FLD002,2026,2026-05-11,15.0,2,17,2.0,Tar Spot,1,NaN,None,-27.0,135
1,FLD002,2026,2026-05-25,29.0,4,31,2.0,Gray Leaf Spot,1,0.0,None,-13.0,121
2,FLD002,2026,2026-06-08,43.0,6,45,2.0,Gray Leaf Spot,1,0.0,None,1.0,107
3,FLD002,2026,2026-06-22,57.0,8,59,2.0,Tar Spot,1,0.0,None,15.0,93
4,FLD002,2026,2026-07-06,71.0,10,73,2.0,Tar Spot,1,0.0,None,29.0,79
5,FLD002,2026,2026-07-20,85.0,12,87,2.0,Tar Spot,1,0.0,None,43.0,65
6,FLD003,2026,2026-05-11,2.0,0,3,1.0,White Mold (Sclerotinia),1,NaN,None,-40.0,135
7,FLD003,2026,2026-05-25,16.0,2,17,1.0,Sudden Death Syndrome (SDS),1,0.0,None,-26.0,121
8,FLD003,2026,2026-06-08,30.0,4,31,1.0,White Mold (Sclerotinia),1,0.0,None,-12.0,107
9,FLD003,2026,2026-06-22,44.0,6,45,1.0,Sudden Death Syndrome (SDS),1,0.0,None,2.0,93


In [96]:
# Load Weather snapshots for temporal alignment

weather_features = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SNAPSHOT_DATE,
        MEAN_TEMPERATURE,
        TEMPERATURE_7D,
        TEMPERATURE_14D,
        TEMPERATURE_DEVIATION,
        RAINFALL_7D,
        RAINFALL_14D,
        RAINFALL_DEVIATION,
        HUMIDITY_7D,
        HUMIDITY_14D,
        HUMIDITY_DEVIATION,
        LEAFWETNESS_7D,
        LEAFWETNESS_DEVIATION,
        CUMULATIVE_GDD,
        HISTORICAL_CUMULATIVE_GDD,
        GDD_DEVIATION,
        GDD_DEVIATION_PCT,
        WEATHER_DATA_AGE
    FROM YIELD_FORECAST_DB.POD_VERTEX.WEATHER_FEATURE_SNAPSHOT
    """,
    conn
)

weather_features["SNAPSHOT_DATE"] = pd.to_datetime(
    weather_features["SNAPSHOT_DATE"],
    errors="coerce"
)

weather_features["SEASON_YEAR"] = weather_features["SEASON_YEAR"].astype(int)

print("Weather rows:", len(weather_features))

print(
    "Unique FIELD_ID + SEASON_YEAR combinations:",
    weather_features[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print("\nMissing SNAPSHOT_DATE:")
print(weather_features["SNAPSHOT_DATE"].isna().sum())

display(weather_features.head(10))

/tmp/ipykernel_75164/3868159608.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_features = pd.read_sql(


Weather rows: 1412
Unique FIELD_ID + SEASON_YEAR combinations: 120

Missing SNAPSHOT_DATE:
0


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,MEAN_TEMPERATURE,TEMPERATURE_7D,TEMPERATURE_14D,TEMPERATURE_DEVIATION,RAINFALL_7D,RAINFALL_14D,RAINFALL_DEVIATION,HUMIDITY_7D,HUMIDITY_14D,HUMIDITY_DEVIATION,LEAFWETNESS_7D,LEAFWETNESS_DEVIATION,CUMULATIVE_GDD,HISTORICAL_CUMULATIVE_GDD,GDD_DEVIATION,GDD_DEVIATION_PCT,WEATHER_DATA_AGE
0,FLD001,2026,2026-05-04,62.5,62.5,62.50,-4.685714,0.68,0.680,-0.441667,55.9,55.90,-3.7,8.7,1.44,87.8,112.85,-25.05,-22.197607,84
1,FLD001,2026,2026-05-11,63.9,63.9,63.20,0.000000,0.00,0.340,0.000000,65.1,60.50,0.0,3.6,0.00,185.1,105.65,79.45,75.201136,77
2,FLD001,2026,2026-05-18,64.2,64.2,64.05,0.000000,0.00,0.000,0.000000,58.8,61.95,0.0,3.2,0.00,284.2,201.05,83.15,41.357871,70
3,FLD001,2026,2026-05-25,68.0,68.0,66.10,0.000000,2.34,1.170,0.000000,58.9,58.85,0.0,7.5,0.00,410.2,320.20,90.00,28.107433,63
4,FLD001,2026,2026-06-01,66.7,66.7,67.35,0.000000,1.78,2.060,0.000000,52.9,55.90,0.0,6.4,0.00,527.1,436.95,90.15,20.631651,56
5,FLD001,2026,2026-06-08,70.2,70.2,68.45,0.000000,0.83,1.305,0.000000,63.3,58.10,0.0,4.6,0.00,668.9,572.95,95.95,16.746662,49
6,FLD001,2026,2026-06-15,67.2,67.2,68.70,0.000000,0.38,0.605,0.000000,53.0,58.15,0.0,4.7,0.00,789.3,688.60,100.70,14.623875,42
7,FLD001,2026,2026-06-22,72.5,72.5,69.85,0.000000,0.71,0.545,0.000000,53.1,53.05,0.0,6.2,0.00,946.5,837.40,109.10,13.028421,35
8,FLD001,2026,2026-06-29,73.6,73.6,73.05,3.808333,1.10,0.905,0.000000,61.9,57.50,0.0,3.3,0.00,970.1,936.90,33.20,3.543601,28
9,FLD001,2026,2026-07-06,74.7,74.7,74.15,-5.800000,1.65,1.375,1.650000,57.6,59.75,-7.0,5.8,1.60,1143.0,1115.40,27.60,2.474449,21


In [98]:
# Reload FIELD_SEASON_CONTEXT with the authoritative season start date

field_context_with_start = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        SEASON_START_DATE,
        PLANTING_DATE,
        HARVEST_DATE
    FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_SEASON_CONTEXT
    """,
    conn
)

field_context_with_start["SEASON_YEAR"] = (
    field_context_with_start["SEASON_YEAR"].astype(int)
)

field_context_with_start["SEASON_START_DATE"] = pd.to_datetime(
    field_context_with_start["SEASON_START_DATE"],
    errors="coerce"
)

print("FIELD_SEASON_CONTEXT rows:", len(field_context_with_start))

print(
    "Missing SEASON_START_DATE:",
    field_context_with_start["SEASON_START_DATE"].isna().sum()
)

display(field_context_with_start.head(10))

/tmp/ipykernel_75164/3523775714.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  field_context_with_start = pd.read_sql(


FIELD_SEASON_CONTEXT rows: 101
Missing SEASON_START_DATE: 0


,FIELD_ID,SEASON_YEAR,SEASON_START_DATE,PLANTING_DATE,HARVEST_DATE
0,FLD001,2027,2027-05-01,2027-05-01,2027-09-18
1,FLD002,2026,2026-04-27,2026-04-27,2026-10-14
2,FLD003,2026,2026-05-10,2026-05-10,2026-09-27
3,FLD004,2026,2026-04-25,2026-04-25,2026-10-10
4,FLD005,2026,2026-04-29,2026-04-29,2026-10-13
5,FLD006,2026,2026-05-06,2026-05-06,2026-10-06
6,FLD007,2026,2026-05-02,2026-05-02,2026-10-19
7,FLD008,2026,2026-05-03,2026-05-03,2026-10-22
8,FLD009,2026,2026-05-16,2026-05-16,2026-09-20
9,FLD010,2026,2026-05-06,2026-05-06,2026-09-23


In [99]:
# Attach authoritative season start date to Weather observations

weather_alignment = weather_features.merge(
    field_context_with_start[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SEASON_START_DATE"
        ]
    ].drop_duplicates(
        subset=["FIELD_ID", "SEASON_YEAR"]
    ),
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    validate="many_to_one"
)

weather_alignment["WEATHER_DAP"] = (
    weather_alignment["SNAPSHOT_DATE"]
    - weather_alignment["SEASON_START_DATE"]
).dt.days

print("Weather rows:", len(weather_alignment))

print(
    "Rows with authoritative season start:",
    weather_alignment["SEASON_START_DATE"].notna().sum()
)

print(
    "Rows without authoritative season start:",
    weather_alignment["SEASON_START_DATE"].isna().sum()
)

print("\nWeather DAP summary:")
print(weather_alignment["WEATHER_DAP"].describe())

display(
    weather_alignment[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "WEATHER_DAP"
        ]
    ].head(20)
)

Weather rows: 1412
Rows with authoritative season start: 1206
Rows without authoritative season start: 206

Weather DAP summary:
count    1206.000000
mean       41.820896
std        27.563421
min       -20.000000
25%        19.000000
50%        42.000000
75%        64.000000
max       129.000000
Name: WEATHER_DAP, dtype: float64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,WEATHER_DAP
0,FLD001,2026,2026-05-04,NaT,NaN
1,FLD001,2026,2026-05-11,NaT,NaN
2,FLD001,2026,2026-05-18,NaT,NaN
3,FLD001,2026,2026-05-25,NaT,NaN
4,FLD001,2026,2026-06-01,NaT,NaN
5,FLD001,2026,2026-06-08,NaT,NaN
6,FLD001,2026,2026-06-15,NaT,NaN
7,FLD001,2026,2026-06-22,NaT,NaN
8,FLD001,2026,2026-06-29,NaT,NaN
9,FLD001,2026,2026-07-06,NaT,NaN


In [100]:
# Classify Weather observations based on authoritative season start

weather_alignment["WEATHER_ALIGNMENT_STATUS"] = "NO_SEASON_START"

weather_alignment.loc[
    weather_alignment["WEATHER_DAP"].notna()
    & (weather_alignment["WEATHER_DAP"] < 0),
    "WEATHER_ALIGNMENT_STATUS"
] = "BEFORE_SEASON_START"

weather_alignment.loc[
    weather_alignment["WEATHER_DAP"].notna()
    & (weather_alignment["WEATHER_DAP"] >= 0),
    "WEATHER_ALIGNMENT_STATUS"
] = "VALID_DAP"

print("Weather alignment status:")
print(
    weather_alignment["WEATHER_ALIGNMENT_STATUS"]
    .value_counts()
)

print("\nWeather observations before season start:")
print(
    (weather_alignment["WEATHER_DAP"] < 0).sum()
)

print("\nWeather observations with valid DAP:")
print(
    (weather_alignment["WEATHER_DAP"] >= 0).sum()
)

print("\nWeather observations without season start:")
print(
    weather_alignment["WEATHER_DAP"].isna().sum()
)

Weather alignment status:
WEATHER_ALIGNMENT_STATUS
VALID_DAP              1142
NO_SEASON_START         206
BEFORE_SEASON_START      64
Name: count, dtype: int64

Weather observations before season start:
64

Weather observations with valid DAP:
1142

Weather observations without season start:
206


In [101]:
# Map valid Weather observations to the canonical seasonal calendar

canonical_seasonal_calendar = pd.DataFrame({
    "WEEK_INDEX": range(13)
})

canonical_seasonal_calendar["SEASON_DAY"] = (
    3 + canonical_seasonal_calendar["WEEK_INDEX"] * 7
)

weather_valid = weather_alignment[
    weather_alignment["WEATHER_ALIGNMENT_STATUS"] == "VALID_DAP"
].copy()

weather_valid["MAPPED_WEEK_INDEX"] = np.nan
weather_valid["MAPPED_SEASON_DAY"] = np.nan
weather_valid["DAP_DISTANCE"] = np.nan

for idx, row in weather_valid.iterrows():

    candidates = canonical_seasonal_calendar.copy()

    candidates["DISTANCE"] = (
        candidates["SEASON_DAY"] - row["WEATHER_DAP"]
    ).abs()

    nearest = candidates.loc[
        candidates["DISTANCE"].idxmin()
    ]

    weather_valid.loc[idx, "MAPPED_WEEK_INDEX"] = (
        nearest["WEEK_INDEX"]
    )

    weather_valid.loc[idx, "MAPPED_SEASON_DAY"] = (
        nearest["SEASON_DAY"]
    )

    weather_valid.loc[idx, "DAP_DISTANCE"] = (
        nearest["DISTANCE"]
    )

print("Valid Weather observations:", len(weather_valid))

print("\nMaximum Weather DAP mapping distance:")
print(weather_valid["DAP_DISTANCE"].max())

print("\nWeather DAP mapping distance distribution:")
print(
    weather_valid["DAP_DISTANCE"]
    .value_counts()
    .sort_index()
)

print("\nMapped WEEK_INDEX distribution:")
print(
    weather_valid["MAPPED_WEEK_INDEX"]
    .value_counts()
    .sort_index()
)

display(
    weather_valid[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "WEATHER_DAP",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY",
            "DAP_DISTANCE"
        ]
    ].head(20)
)

Valid Weather observations: 1142

Maximum Weather DAP mapping distance:
42.0

Weather DAP mapping distance distribution:
DAP_DISTANCE
0.0     182
1.0     250
2.0     293
3.0     392
4.0       7
5.0       1
6.0       5
7.0       1
8.0       3
9.0       1
10.0      1
11.0      3
18.0      1
34.0      1
42.0      1
Name: count, dtype: int64

Mapped WEEK_INDEX distribution:
MAPPED_WEEK_INDEX
0.0     74
1.0     88
2.0     93
3.0     91
4.0     96
5.0     88
6.0     92
7.0     95
8.0     93
9.0     95
10.0    89
11.0    74
12.0    74
Name: count, dtype: int64


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,WEATHER_DAP,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY,DAP_DISTANCE
13,FLD002,2026,2026-05-04,2026-04-27,7.0,1.0,10.0,3.0
14,FLD002,2026,2026-05-11,2026-04-27,14.0,2.0,17.0,3.0
15,FLD002,2026,2026-05-18,2026-04-27,21.0,3.0,24.0,3.0
16,FLD002,2026,2026-05-25,2026-04-27,28.0,4.0,31.0,3.0
17,FLD002,2026,2026-06-01,2026-04-27,35.0,5.0,38.0,3.0
18,FLD002,2026,2026-06-08,2026-04-27,42.0,6.0,45.0,3.0
19,FLD002,2026,2026-06-15,2026-04-27,49.0,7.0,52.0,3.0
20,FLD002,2026,2026-06-22,2026-04-27,56.0,8.0,59.0,3.0
21,FLD002,2026,2026-06-29,2026-04-27,63.0,9.0,66.0,3.0
22,FLD002,2026,2026-07-06,2026-04-27,70.0,10.0,73.0,3.0


In [102]:
# Inspect Weather observations with unusually large
# distance from the canonical seasonal calendar

weather_large_distance = weather_valid[
    weather_valid["DAP_DISTANCE"] > 7
].copy()

print(
    "Weather observations with DAP_DISTANCE > 7:",
    len(weather_large_distance)
)

display(
    weather_large_distance[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "WEATHER_DAP",
            "MAPPED_WEEK_INDEX",
            "MAPPED_SEASON_DAY",
            "DAP_DISTANCE"
        ]
    ].sort_values(
        ["DAP_DISTANCE", "FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]
    )
)

Weather observations with DAP_DISTANCE > 7: 11


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,SEASON_START_DATE,WEATHER_DAP,MAPPED_WEEK_INDEX,MAPPED_SEASON_DAY,DAP_DISTANCE
338,FLD012,2023,2023-07-27,2023-04-23,95.0,12.0,87.0,8.0
808,FLD026,2023,2023-07-27,2023-04-23,95.0,12.0,87.0,8.0
1365,FLD040,2023,2023-07-27,2023-04-23,95.0,12.0,87.0,8.0
1098,FLD034,2023,2023-07-27,2023-04-22,96.0,12.0,87.0,9.0
723,FLD022,2026,2026-07-27,2026-04-21,97.0,12.0,87.0,10.0
51,FLD004,2023,2023-07-27,2023-04-20,98.0,12.0,87.0,11.0
375,FLD012,2026,2026-07-20,2026-04-13,98.0,12.0,87.0,11.0
510,FLD017,2023,2023-07-27,2023-04-20,98.0,12.0,87.0,11.0
376,FLD012,2026,2026-07-27,2026-04-13,105.0,12.0,87.0,18.0
1291,FLD037,2026,2026-09-10,2026-05-12,121.0,12.0,87.0,34.0


In [103]:
print("Canonical seasonal calendar:")
display(canonical_seasonal_calendar)

print("\nWeather DAP summary:")
display(
    weather_valid["WEATHER_DAP"]
    .describe()
    .to_frame("WEATHER_DAP")
)

print("\nMaximum Weather DAP by field-season:")
weather_coverage = (
    weather_valid
    .groupby(["FIELD_ID", "SEASON_YEAR"], as_index=False)
    .agg(
        MIN_WEATHER_DAP=("WEATHER_DAP", "min"),
        MAX_WEATHER_DAP=("WEATHER_DAP", "max"),
        WEATHER_OBSERVATION_COUNT=("WEATHER_DAP", "count")
    )
)

display(
    weather_coverage
    .sort_values("MAX_WEATHER_DAP", ascending=False)
    .head(20)
)

Canonical seasonal calendar:


,WEEK_INDEX,SEASON_DAY
0,0,3
1,1,10
2,2,17
3,3,24
4,4,31
5,5,38
6,6,45
7,7,52
8,8,59
9,9,66



Weather DAP summary:


,WEATHER_DAP
count,1142.000000
mean,44.529772
std,25.744489
min,0.000000
25%,23.000000
50%,44.000000
75%,66.000000
max,129.000000



Maximum Weather DAP by field-season:


,FIELD_ID,SEASON_YEAR,MIN_WEATHER_DAP,MAX_WEATHER_DAP,WEATHER_OBSERVATION_COUNT
32,FLD013,2026,0.0,129.0,11
87,FLD037,2026,13.0,121.0,11
28,FLD012,2026,21.0,105.0,12
2,FLD004,2023,14.0,98.0,13
38,FLD017,2023,14.0,98.0,13
55,FLD022,2026,13.0,97.0,10
76,FLD034,2023,12.0,96.0,13
25,FLD012,2023,11.0,95.0,13
92,FLD040,2023,11.0,95.0,13
62,FLD026,2023,11.0,95.0,13


In [104]:
print("Season Intelligence coverage:")
display(
    season_intelligence_input[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    ).head(20)
)

print("\nSeason Intelligence maximum SEASON_DAY:")
print(
    season_intelligence_input["SEASON_DAY"].min(),
    "to",
    season_intelligence_input["SEASON_DAY"].max()
)

print("\nWeather observations beyond canonical Day 87:")
display(
    weather_valid[
        weather_valid["WEATHER_DAP"] > 87
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "SEASON_START_DATE",
            "WEATHER_DAP"
        ]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]
    )
)


Season Intelligence coverage:


NameError: name 'season_intelligence_input' is not defined

In [105]:
season_intelligence_coverage = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_INTELLIGENCE_INPUT
    ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
    """,
    conn
)

print("Season Intelligence rows:", len(season_intelligence_coverage))

print("\nSeason Intelligence coverage:")
display(
    season_intelligence_coverage.head(20)
)

print("\nWEEK_INDEX values:")
print(
    sorted(
        season_intelligence_coverage["WEEK_INDEX"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nSEASON_DAY range:")
print(
    season_intelligence_coverage["SEASON_DAY"].min(),
    "to",
    season_intelligence_coverage["SEASON_DAY"].max()
)

/tmp/ipykernel_75164/2981017823.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_intelligence_coverage = pd.read_sql(


DatabaseError: Execution failed on sql: 
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_INTELLIGENCE_INPUT
    ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
    
390114 (08001): Authentication token has expired.  The user must authenticate again.
unable to rollback

In [108]:
import importlib
import snowflake_loader

importlib.reload(snowflake_loader)

conn = snowflake_loader.get_connection()

print("Snowflake connection refreshed successfully")

Snowflake connection refreshed successfully


In [109]:
season_intelligence_coverage = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY
    FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_INTELLIGENCE_INPUT
    ORDER BY FIELD_ID, SEASON_YEAR, WEEK_INDEX
    """,
    conn
)

print("Season Intelligence rows:", len(season_intelligence_coverage))

print("\nSeason Intelligence coverage:")
display(
    season_intelligence_coverage.head(20)
)

print("\nWEEK_INDEX values:")
print(
    sorted(
        season_intelligence_coverage["WEEK_INDEX"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nSEASON_DAY range:")
print(
    season_intelligence_coverage["SEASON_DAY"].min(),
    "to",
    season_intelligence_coverage["SEASON_DAY"].max()
)

/tmp/ipykernel_75164/2981017823.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  season_intelligence_coverage = pd.read_sql(


Season Intelligence rows: 111

Season Intelligence coverage:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY
0,FLD007,2023,1,7
1,FLD007,2023,3,21
2,FLD007,2023,5,35
3,FLD007,2023,7,49
4,FLD007,2023,9,63
5,FLD007,2023,11,77
6,FLD007,2024,1,10
7,FLD007,2024,3,24
8,FLD007,2024,5,38
9,FLD007,2024,7,52



WEEK_INDEX values:
[1, 3, 5, 7, 9, 11]

SEASON_DAY range:
7 to 83


In [110]:
forecast_timeline = season_intelligence_coverage[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY"
    ]
].copy()

print("Forecast timeline rows:", len(forecast_timeline))
print(
    "Unique field-seasons:",
    forecast_timeline[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print("\nSeason-day distribution by WEEK_INDEX:")
display(
    forecast_timeline
    .groupby("WEEK_INDEX")["SEASON_DAY"]
    .agg(["count", "min", "max", "mean"])
)

print("\nSample field-season timelines:")
display(
    forecast_timeline
    .groupby(["FIELD_ID", "SEASON_YEAR"])
    .head(6)
    .head(30)
)

Forecast timeline rows: 111
Unique field-seasons: 19

Season-day distribution by WEEK_INDEX:


,count,min,max,mean
WEEK_INDEX,,,,
1,19,7,13,9.578947
3,18,21,27,23.722222
5,18,35,41,37.611111
7,19,49,55,51.578947
9,19,63,69,65.578947
11,18,77,83,79.666667



Sample field-season timelines:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY
0,FLD007,2023,1,7
1,FLD007,2023,3,21
2,FLD007,2023,5,35
3,FLD007,2023,7,49
4,FLD007,2023,9,63
5,FLD007,2023,11,77
6,FLD007,2024,1,10
7,FLD007,2024,3,24
8,FLD007,2024,5,38
9,FLD007,2024,7,52


In [111]:
forecast_timeline_with_dates = forecast_timeline.merge(
    field_context_with_start[
        ["FIELD_ID", "SEASON_YEAR", "SEASON_START_DATE"]
    ].drop_duplicates(
        subset=["FIELD_ID", "SEASON_YEAR"]
    ),
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    validate="many_to_one"
)

forecast_timeline_with_dates["EXPECTED_SNAPSHOT_DATE"] = (
    forecast_timeline_with_dates["SEASON_START_DATE"]
    + pd.to_timedelta(
        forecast_timeline_with_dates["SEASON_DAY"] - 1,
        unit="D"
    )
)

print("Forecast timeline rows:", len(forecast_timeline_with_dates))

print(
    "Missing season start:",
    forecast_timeline_with_dates["SEASON_START_DATE"].isna().sum()
)

print(
    "Missing expected snapshot date:",
    forecast_timeline_with_dates["EXPECTED_SNAPSHOT_DATE"].isna().sum()
)

display(
    forecast_timeline_with_dates.head(20)
)

Forecast timeline rows: 111
Missing season start: 0
Missing expected snapshot date: 0


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,SEASON_START_DATE,EXPECTED_SNAPSHOT_DATE
0,FLD007,2023,1,7,2023-04-27,2023-05-03
1,FLD007,2023,3,21,2023-04-27,2023-05-17
2,FLD007,2023,5,35,2023-04-27,2023-05-31
3,FLD007,2023,7,49,2023-04-27,2023-06-14
4,FLD007,2023,9,63,2023-04-27,2023-06-28
5,FLD007,2023,11,77,2023-04-27,2023-07-12
6,FLD007,2024,1,10,2024-05-08,2024-05-17
7,FLD007,2024,3,24,2024-05-08,2024-05-31
8,FLD007,2024,5,38,2024-05-08,2024-06-14
9,FLD007,2024,7,52,2024-05-08,2024-06-28


In [112]:
weather_checkpoint_alignment = weather_features.merge(
    forecast_timeline_with_dates[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_SNAPSHOT_DATE"
        ]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="inner"
)

weather_checkpoint_alignment["CHECKPOINT_DATE_DISTANCE"] = (
    weather_checkpoint_alignment["SNAPSHOT_DATE"]
    - weather_checkpoint_alignment["EXPECTED_SNAPSHOT_DATE"]
).abs().dt.days

print(
    "Weather rows after checkpoint alignment:",
    len(weather_checkpoint_alignment)
)

print(
    "Unique Weather observations:",
    weather_checkpoint_alignment[
        ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]
    ].drop_duplicates().shape[0]
)

print("\nCheckpoint distance distribution:")
display(
    weather_checkpoint_alignment[
        "CHECKPOINT_DATE_DISTANCE"
    ].describe()
)

Weather rows after checkpoint alignment: 1405
Unique Weather observations: 240

Checkpoint distance distribution:


count    1405.000000
mean       29.287544
std        20.782825
min         1.000000
25%        13.000000
50%        27.000000
75%        43.000000
max        85.000000
Name: CHECKPOINT_DATE_DISTANCE, dtype: float64

In [113]:
weather_checkpoint_nearest = (
    weather_checkpoint_alignment
    .sort_values(
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "CHECKPOINT_DATE_DISTANCE"
        ]
    )
    .drop_duplicates(
        subset=[
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE"
        ],
        keep="first"
    )
    .copy()
)

print(
    "Unique Weather observations:",
    weather_checkpoint_nearest[
        ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]
    ].drop_duplicates().shape[0]
)

print(
    "Nearest-checkpoint rows:",
    len(weather_checkpoint_nearest)
)

print("\nNearest checkpoint distance distribution:")
display(
    weather_checkpoint_nearest[
        "CHECKPOINT_DATE_DISTANCE"
    ].describe()
)

print("\nNearest checkpoint distance frequencies:")
display(
    weather_checkpoint_nearest[
        "CHECKPOINT_DATE_DISTANCE"
    ]
    .value_counts()
    .sort_index()
    .head(20)
)

Unique Weather observations: 240
Nearest-checkpoint rows: 240

Nearest checkpoint distance distribution:


count    240.000000
mean       4.254167
std        3.568514
min        1.000000
25%        1.000000
50%        6.000000
75%        6.000000
max       20.000000
Name: CHECKPOINT_DATE_DISTANCE, dtype: float64


Nearest checkpoint distance frequencies:


CHECKPOINT_DATE_DISTANCE
1     111
6     102
8      14
13      7
15      5
20      1
Name: count, dtype: int64

In [114]:
weather_unmatched_checkpoint = weather_checkpoint_nearest[
    weather_checkpoint_nearest["CHECKPOINT_DATE_DISTANCE"] > 6
].copy()

print(
    "Weather observations > 6 days from nearest checkpoint:",
    len(weather_unmatched_checkpoint)
)

display(
    weather_unmatched_checkpoint[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_SNAPSHOT_DATE",
            "CHECKPOINT_DATE_DISTANCE"
        ]
    ].sort_values(
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "SNAPSHOT_DATE"
        ]
    )
)

Weather observations > 6 days from nearest checkpoint: 27


,FIELD_ID,SEASON_YEAR,SNAPSHOT_DATE,WEEK_INDEX,SEASON_DAY,EXPECTED_SNAPSHOT_DATE,CHECKPOINT_DATE_DISTANCE
71,FLD007,2023,2023-07-20,11,77,2023-07-12,8
77,FLD007,2023,2023-07-27,11,77,2023-07-12,15
78,FLD007,2024,2024-05-04,1,10,2024-05-17,13
233,FLD007,2025,2025-07-27,11,79,2025-07-19,8
255,FLD007,2026,2026-06-01,3,23,2026-05-24,8
288,FLD007,2026,2026-07-27,11,79,2026-07-19,8
289,FLD014,2023,2023-05-04,1,12,2023-05-17,13
367,FLD014,2024,2024-05-04,1,8,2024-05-24,20
372,FLD014,2024,2024-05-11,1,8,2024-05-24,13
431,FLD014,2024,2024-07-27,9,64,2024-07-19,8


In [115]:
weather_for_asof = weather_features[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "SNAPSHOT_DATE",
        "MEAN_TEMPERATURE",
        "TEMPERATURE_7D",
        "TEMPERATURE_14D",
        "TEMPERATURE_DEVIATION",
        "RAINFALL_7D",
        "RAINFALL_14D",
        "RAINFALL_DEVIATION",
        "HUMIDITY_7D",
        "HUMIDITY_14D",
        "HUMIDITY_DEVIATION",
        "LEAFWETNESS_7D",
        "LEAFWETNESS_DEVIATION",
        "CUMULATIVE_GDD",
        "HISTORICAL_CUMULATIVE_GDD",
        "GDD_DEVIATION",
        "GDD_DEVIATION_PCT",
        "WEATHER_DATA_AGE"
    ]
].copy()

forecast_for_asof = forecast_timeline_with_dates[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY",
        "EXPECTED_SNAPSHOT_DATE"
    ]
].copy()

weather_for_asof = weather_for_asof.sort_values(
    ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]
)

forecast_for_asof = forecast_for_asof.sort_values(
    ["FIELD_ID", "SEASON_YEAR", "EXPECTED_SNAPSHOT_DATE"]
)

weather_asof_aligned = pd.merge_asof(
    forecast_for_asof,
    weather_for_asof,
    left_on="EXPECTED_SNAPSHOT_DATE",
    right_on="SNAPSHOT_DATE",
    by=["FIELD_ID", "SEASON_YEAR"],
    direction="backward"
)

weather_asof_aligned["WEATHER_DATA_LAG_DAYS"] = (
    weather_asof_aligned["EXPECTED_SNAPSHOT_DATE"]
    - weather_asof_aligned["SNAPSHOT_DATE"]
).dt.days

print("Forecast checkpoint rows:", len(weather_asof_aligned))

print(
    "Checkpoints with Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].notna().sum()
)

print(
    "Checkpoints without Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].isna().sum()
)

print("\nWeather lag distribution:")
display(
    weather_asof_aligned["WEATHER_DATA_LAG_DAYS"]
    .describe()
)

ValueError: left keys must be sorted

In [116]:
weather_for_asof = weather_for_asof.sort_values(
    ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]
).reset_index(drop=True)

forecast_for_asof = forecast_for_asof.sort_values(
    ["FIELD_ID", "SEASON_YEAR", "EXPECTED_SNAPSHOT_DATE"]
).reset_index(drop=True)

weather_asof_aligned = pd.merge_asof(
    forecast_for_asof,
    weather_for_asof,
    left_on="EXPECTED_SNAPSHOT_DATE",
    right_on="SNAPSHOT_DATE",
    by=["FIELD_ID", "SEASON_YEAR"],
    direction="backward"
)

weather_asof_aligned["WEATHER_DATA_LAG_DAYS"] = (
    weather_asof_aligned["EXPECTED_SNAPSHOT_DATE"]
    - weather_asof_aligned["SNAPSHOT_DATE"]
).dt.days

print("Forecast checkpoint rows:", len(weather_asof_aligned))

print(
    "Checkpoints with Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].notna().sum()
)

print(
    "Checkpoints without Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].isna().sum()
)

print("\nWeather lag distribution:")
display(
    weather_asof_aligned["WEATHER_DATA_LAG_DAYS"]
    .describe()
)

ValueError: left keys must be sorted

In [117]:
def align_weather_asof(group):
    field_id = group["FIELD_ID"].iloc[0]
    season_year = group["SEASON_YEAR"].iloc[0]

    forecast_rows = (
        forecast_for_asof[
            (forecast_for_asof["FIELD_ID"] == field_id) &
            (forecast_for_asof["SEASON_YEAR"] == season_year)
        ]
        .sort_values("EXPECTED_SNAPSHOT_DATE")
        .copy()
    )

    weather_rows = (
        weather_for_asof[
            (weather_for_asof["FIELD_ID"] == field_id) &
            (weather_for_asof["SEASON_YEAR"] == season_year)
        ]
        .sort_values("SNAPSHOT_DATE")
        .copy()
    )

    results = []

    for _, forecast_row in forecast_rows.iterrows():

        eligible_weather = weather_rows[
            weather_rows["SNAPSHOT_DATE"]
            <= forecast_row["EXPECTED_SNAPSHOT_DATE"]
        ]

        if len(eligible_weather) == 0:
            selected_weather = None
        else:
            selected_weather = eligible_weather.iloc[-1]

        result = forecast_row.to_dict()

        if selected_weather is None:
            result["SNAPSHOT_DATE"] = pd.NaT

            for column in weather_for_asof.columns:
                if column not in ["FIELD_ID", "SEASON_YEAR", "SNAPSHOT_DATE"]:
                    result[column] = np.nan

        else:
            for column in weather_for_asof.columns:
                if column not in ["FIELD_ID", "SEASON_YEAR"]:
                    result[column] = selected_weather[column]

        results.append(result)

    return pd.DataFrame(results)


weather_asof_aligned = (
    forecast_for_asof[
        ["FIELD_ID", "SEASON_YEAR"]
    ]
    .drop_duplicates()
    .groupby(
        ["FIELD_ID", "SEASON_YEAR"],
        group_keys=False
    )
    .apply(
        align_weather_asof,
        include_groups=False
    )
    .reset_index(drop=True)
)

weather_asof_aligned["WEATHER_DATA_LAG_DAYS"] = (
    weather_asof_aligned["EXPECTED_SNAPSHOT_DATE"]
    - weather_asof_aligned["SNAPSHOT_DATE"]
).dt.days

print("Forecast checkpoint rows:", len(weather_asof_aligned))

print(
    "Checkpoints with Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].notna().sum()
)

print(
    "Checkpoints without Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].isna().sum()
)

print("\nWeather lag distribution:")
display(
    weather_asof_aligned["WEATHER_DATA_LAG_DAYS"]
    .describe()
)

TypeError: align_weather_asof() got an unexpected keyword argument 'include_groups'

In [118]:
weather_asof_results = []

field_seasons = forecast_for_asof[
    ["FIELD_ID", "SEASON_YEAR"]
].drop_duplicates()

for _, fs in field_seasons.iterrows():

    field_id = fs["FIELD_ID"]
    season_year = fs["SEASON_YEAR"]

    forecast_rows = forecast_for_asof[
        (forecast_for_asof["FIELD_ID"] == field_id) &
        (forecast_for_asof["SEASON_YEAR"] == season_year)
    ].sort_values("EXPECTED_SNAPSHOT_DATE")

    weather_rows = weather_for_asof[
        (weather_for_asof["FIELD_ID"] == field_id) &
        (weather_for_asof["SEASON_YEAR"] == season_year)
    ].sort_values("SNAPSHOT_DATE")

    for _, forecast_row in forecast_rows.iterrows():

        eligible_weather = weather_rows[
            weather_rows["SNAPSHOT_DATE"]
            <= forecast_row["EXPECTED_SNAPSHOT_DATE"]
        ]

        result = forecast_row.to_dict()

        if eligible_weather.empty:

            result["SNAPSHOT_DATE"] = pd.NaT

            for column in weather_for_asof.columns:
                if column not in [
                    "FIELD_ID",
                    "SEASON_YEAR",
                    "SNAPSHOT_DATE"
                ]:
                    result[column] = np.nan

        else:

            selected_weather = eligible_weather.iloc[-1]

            for column in weather_for_asof.columns:
                if column not in [
                    "FIELD_ID",
                    "SEASON_YEAR"
                ]:
                    result[column] = selected_weather[column]

        weather_asof_results.append(result)


weather_asof_aligned = pd.DataFrame(weather_asof_results)

weather_asof_aligned["WEATHER_DATA_LAG_DAYS"] = (
    weather_asof_aligned["EXPECTED_SNAPSHOT_DATE"]
    - weather_asof_aligned["SNAPSHOT_DATE"]
).dt.days

print("Forecast checkpoint rows:", len(weather_asof_aligned))

print(
    "Checkpoints with Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].notna().sum()
)

print(
    "Checkpoints without Weather:",
    weather_asof_aligned["SNAPSHOT_DATE"].isna().sum()
)

print("\nWeather lag distribution:")
display(
    weather_asof_aligned["WEATHER_DATA_LAG_DAYS"]
    .describe()
)

Forecast checkpoint rows: 111
Checkpoints with Weather: 106
Checkpoints without Weather: 5

Weather lag distribution:


count    106.000000
mean       6.330189
std        1.776683
min        6.000000
25%        6.000000
50%        6.000000
75%        6.000000
max       20.000000
Name: WEATHER_DATA_LAG_DAYS, dtype: float64

In [119]:
weather_missing_checkpoints = weather_asof_aligned[
    weather_asof_aligned["SNAPSHOT_DATE"].isna()
].copy()

display(
    weather_missing_checkpoints[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_SNAPSHOT_DATE"
        ]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    )
)

,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,EXPECTED_SNAPSHOT_DATE
0,FLD007,2023,1,7,2023-05-03
45,FLD029,2023,1,10,2023-05-03
69,FLD034,2023,1,12,2023-05-03
81,FLD034,2025,1,7,2025-05-03
87,FLD036,2023,1,8,2023-05-03


In [120]:
missing_weather_fs = weather_asof_aligned[
    weather_asof_aligned["SNAPSHOT_DATE"].isna()
][
    ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX",
     "SEASON_DAY", "EXPECTED_SNAPSHOT_DATE"]
].copy()

for _, row in missing_weather_fs.iterrows():

    field_id = row["FIELD_ID"]
    season_year = row["SEASON_YEAR"]

    print("\n" + "=" * 70)
    print(
        f"FIELD_ID={field_id}, "
        f"SEASON_YEAR={season_year}, "
        f"WEEK_INDEX={row['WEEK_INDEX']}, "
        f"SEASON_DAY={row['SEASON_DAY']}, "
        f"CHECKPOINT={row['EXPECTED_SNAPSHOT_DATE']}"
    )

    available_weather = weather_features[
        (weather_features["FIELD_ID"] == field_id) &
        (weather_features["SEASON_YEAR"] == season_year)
    ][
        ["SNAPSHOT_DATE"]
    ].sort_values("SNAPSHOT_DATE")

    if available_weather.empty:
        print("No Weather observations available for this field-season.")
    else:
        print("Available Weather observations:")
        display(available_weather)


FIELD_ID=FLD007, SEASON_YEAR=2023, WEEK_INDEX=1, SEASON_DAY=7, CHECKPOINT=2023-05-03 00:00:00
Available Weather observations:


,SNAPSHOT_DATE
113,2023-05-04
114,2023-05-11
115,2023-05-18
116,2023-05-25
117,2023-06-01
118,2023-06-08
119,2023-06-15
120,2023-06-22
121,2023-06-29
122,2023-07-06



FIELD_ID=FLD029, SEASON_YEAR=2023, WEEK_INDEX=1, SEASON_DAY=10, CHECKPOINT=2023-05-03 00:00:00
Available Weather observations:


,SNAPSHOT_DATE
911,2023-05-04
912,2023-05-11
913,2023-05-18
914,2023-05-25
915,2023-06-01
916,2023-06-08
917,2023-06-15
918,2023-06-22
919,2023-06-29
920,2023-07-06



FIELD_ID=FLD034, SEASON_YEAR=2023, WEEK_INDEX=1, SEASON_DAY=12, CHECKPOINT=2023-05-03 00:00:00
Available Weather observations:


,SNAPSHOT_DATE
1086,2023-05-04
1087,2023-05-11
1088,2023-05-18
1089,2023-05-25
1090,2023-06-01
1091,2023-06-08
1092,2023-06-15
1093,2023-06-22
1094,2023-06-29
1095,2023-07-06



FIELD_ID=FLD034, SEASON_YEAR=2025, WEEK_INDEX=1, SEASON_DAY=7, CHECKPOINT=2025-05-03 00:00:00
Available Weather observations:


,SNAPSHOT_DATE
1112,2025-05-04
1113,2025-05-11
1114,2025-05-18
1115,2025-05-25
1116,2025-06-01
1117,2025-06-08
1118,2025-06-15
1119,2025-06-22
1120,2025-06-29
1121,2025-07-06



FIELD_ID=FLD036, SEASON_YEAR=2023, WEEK_INDEX=1, SEASON_DAY=8, CHECKPOINT=2023-05-03 00:00:00
Available Weather observations:


,SNAPSHOT_DATE
1190,2023-05-04
1191,2023-05-11
1192,2023-05-18
1193,2023-05-25
1194,2023-06-01
1195,2023-06-08
1196,2023-06-15
1197,2023-06-22
1198,2023-06-29
1199,2023-07-06


In [121]:
weather_asof_aligned["WEATHER_ALIGNMENT_STATUS"] = np.where(
    weather_asof_aligned["SNAPSHOT_DATE"].isna(),
    "NO_PRIOR_WEATHER",
    "AS_OF_ALIGNED"
)

print("Weather alignment status:")
display(
    weather_asof_aligned["WEATHER_ALIGNMENT_STATUS"]
    .value_counts()
)

print("\nMissing Weather checkpoints:")
display(
    weather_asof_aligned[
        weather_asof_aligned["WEATHER_ALIGNMENT_STATUS"]
        == "NO_PRIOR_WEATHER"
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "EXPECTED_SNAPSHOT_DATE",
            "WEATHER_ALIGNMENT_STATUS"
        ]
    ]
)

Weather alignment status:


WEATHER_ALIGNMENT_STATUS
AS_OF_ALIGNED       106
NO_PRIOR_WEATHER      5
Name: count, dtype: int64


Missing Weather checkpoints:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,EXPECTED_SNAPSHOT_DATE,WEATHER_ALIGNMENT_STATUS
0,FLD007,2023,1,7,2023-05-03,NO_PRIOR_WEATHER
45,FLD029,2023,1,10,2023-05-03,NO_PRIOR_WEATHER
69,FLD034,2023,1,12,2023-05-03,NO_PRIOR_WEATHER
81,FLD034,2025,1,7,2025-05-03,NO_PRIOR_WEATHER
87,FLD036,2023,1,8,2023-05-03,NO_PRIOR_WEATHER


In [122]:
weather_asof_aligned["FUTURE_WEATHER_FLAG"] = (
    weather_asof_aligned["SNAPSHOT_DATE"]
    > weather_asof_aligned["EXPECTED_SNAPSHOT_DATE"]
)

print(
    "Future Weather observations used:",
    weather_asof_aligned["FUTURE_WEATHER_FLAG"].sum()
)

print(
    "As-of aligned Weather observations:",
    (
        weather_asof_aligned["SNAPSHOT_DATE"].notna()
        &
        ~weather_asof_aligned["FUTURE_WEATHER_FLAG"]
    ).sum()
)

print(
    "Missing Weather checkpoints:",
    weather_asof_aligned["SNAPSHOT_DATE"].isna().sum()
)

if weather_asof_aligned["FUTURE_WEATHER_FLAG"].any():
    print("\nERROR — future Weather observations detected:")
    display(
        weather_asof_aligned[
            weather_asof_aligned["FUTURE_WEATHER_FLAG"]
        ][
            [
                "FIELD_ID",
                "SEASON_YEAR",
                "WEEK_INDEX",
                "EXPECTED_SNAPSHOT_DATE",
                "SNAPSHOT_DATE"
            ]
        ]
    )
else:
    print("\nPASS — no future Weather observations are used.")

Future Weather observations used: 0
As-of aligned Weather observations: 106
Missing Weather checkpoints: 5

PASS — no future Weather observations are used.


In [123]:
ndvi_for_forecast = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY,
        NDVI_VALUE,
        EXPECTED_NDVI,
        NDVI_DEVIATION,
        NDVI_CHANGE,
        NDVI_AVAILABLE,
        NDVI_RANGE_VALID,
        NDVI_CHANGE_VALID
    FROM YIELD_FORECAST_DB.POD_VERTEX.NDVI_FEATURE_SNAPSHOT
    """,
    conn
)

ndvi_for_forecast["SEASON_YEAR"] = (
    ndvi_for_forecast["SEASON_YEAR"].astype(int)
)

ndvi_for_forecast["WEEK_INDEX"] = (
    ndvi_for_forecast["WEEK_INDEX"].astype(int)
)

ndvi_checkpoint_coverage = forecast_timeline[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY"
    ]
].merge(
    ndvi_for_forecast,
    on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY"
    ],
    how="left",
    validate="one_to_one"
)

print("Forecast checkpoint rows:", len(ndvi_checkpoint_coverage))

print(
    "NDVI observations available:",
    ndvi_checkpoint_coverage["NDVI_VALUE"].notna().sum()
)

print(
    "NDVI observations missing:",
    ndvi_checkpoint_coverage["NDVI_VALUE"].isna().sum()
)

print("\nNDVI availability:")
display(
    ndvi_checkpoint_coverage["NDVI_AVAILABLE"]
    .value_counts(dropna=False)
)

/tmp/ipykernel_75164/154925563.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ndvi_for_forecast = pd.read_sql(


Forecast checkpoint rows: 111
NDVI observations available: 0
NDVI observations missing: 111

NDVI availability:


NDVI_AVAILABLE
NaN    87
0.0    24
Name: count, dtype: int64

In [124]:
print("NDVI source rows:", len(ndvi_for_forecast))

print("\nNDVI WEEK_INDEX coverage:")
display(
    ndvi_for_forecast["WEEK_INDEX"]
    .value_counts()
    .sort_index()
)

print("\nNDVI available flag:")
display(
    ndvi_for_forecast["NDVI_AVAILABLE"]
    .value_counts(dropna=False)
)

print("\nNDVI values available:")
print(
    ndvi_for_forecast["NDVI_VALUE"].notna().sum()
)

print("\nNDVI field-season coverage:")
print(
    ndvi_for_forecast[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print("\nSample NDVI rows:")
display(
    ndvi_for_forecast[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "NDVI_VALUE",
            "NDVI_AVAILABLE"
        ]
    ].head(20)
)

NDVI source rows: 1439

NDVI WEEK_INDEX coverage:


WEEK_INDEX
0     111
1     111
2     107
3     113
4     111
5     111
6     105
7     113
8     113
9     111
10    111
11    112
12    110
Name: count, dtype: int64


NDVI available flag:


NDVI_AVAILABLE
1    728
0    711
Name: count, dtype: int64


NDVI values available:
728

NDVI field-season coverage:
114

Sample NDVI rows:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,NDVI_VALUE,NDVI_AVAILABLE
0,FLD001,2026,0,3,0.61,1
1,FLD001,2026,1,10,NaN,0
2,FLD001,2026,10,73,0.77,1
3,FLD001,2026,11,80,NaN,0
4,FLD001,2026,12,87,0.81,1
5,FLD001,2026,2,17,0.62,1
6,FLD001,2026,3,24,NaN,0
7,FLD001,2026,4,31,0.68,1
8,FLD001,2026,5,38,NaN,0
9,FLD001,2026,6,45,0.70,1


In [125]:
print("\nNDVI available flag:")
display(
    ndvi_for_forecast["NDVI_AVAILABLE"]
    .value_counts(dropna=False)
)

print("\nNDVI values available:")
print(
    ndvi_for_forecast["NDVI_VALUE"].notna().sum()
)

print("\nNDVI field-season coverage:")
print(
    ndvi_for_forecast[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print("\nSample NDVI rows:")
display(
    ndvi_for_forecast[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "NDVI_VALUE",
            "NDVI_AVAILABLE"
        ]
    ].head(20)
)


NDVI available flag:


NDVI_AVAILABLE
1    728
0    711
Name: count, dtype: int64


NDVI values available:
728

NDVI field-season coverage:
114

Sample NDVI rows:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,NDVI_VALUE,NDVI_AVAILABLE
0,FLD001,2026,0,3,0.61,1
1,FLD001,2026,1,10,NaN,0
2,FLD001,2026,10,73,0.77,1
3,FLD001,2026,11,80,NaN,0
4,FLD001,2026,12,87,0.81,1
5,FLD001,2026,2,17,0.62,1
6,FLD001,2026,3,24,NaN,0
7,FLD001,2026,4,31,0.68,1
8,FLD001,2026,5,38,NaN,0
9,FLD001,2026,6,45,0.70,1


In [126]:
ndvi_available = ndvi_for_forecast[
    ndvi_for_forecast["NDVI_AVAILABLE"] == 1
].copy()

ndvi_available["SEASON_DAY"] = ndvi_available["SEASON_DAY"].astype(int)

ndvi_checkpoint_candidates = []

for _, checkpoint in forecast_timeline.iterrows():

    field_id = checkpoint["FIELD_ID"]
    season_year = checkpoint["SEASON_YEAR"]
    checkpoint_day = checkpoint["SEASON_DAY"]

    candidates = ndvi_available[
        (ndvi_available["FIELD_ID"] == field_id) &
        (ndvi_available["SEASON_YEAR"] == season_year) &
        (ndvi_available["SEASON_DAY"] <= checkpoint_day)
    ].sort_values("SEASON_DAY")

    if candidates.empty:
        ndvi_checkpoint_candidates.append({
            "FIELD_ID": field_id,
            "SEASON_YEAR": season_year,
            "WEEK_INDEX": checkpoint["WEEK_INDEX"],
            "CHECKPOINT_SEASON_DAY": checkpoint_day,
            "NDVI_SEASON_DAY": np.nan,
            "NDVI_VALUE": np.nan,
            "NDVI_LAG_DAYS": np.nan,
            "NDVI_STATUS": "NO_PRIOR_NDVI"
        })
    else:
        selected = candidates.iloc[-1]

        ndvi_checkpoint_candidates.append({
            "FIELD_ID": field_id,
            "SEASON_YEAR": season_year,
            "WEEK_INDEX": checkpoint["WEEK_INDEX"],
            "CHECKPOINT_SEASON_DAY": checkpoint_day,
            "NDVI_SEASON_DAY": selected["SEASON_DAY"],
            "NDVI_VALUE": selected["NDVI_VALUE"],
            "NDVI_LAG_DAYS": checkpoint_day - selected["SEASON_DAY"],
            "NDVI_STATUS": "PRIOR_NDVI_AVAILABLE"
        })

ndvi_checkpoint_asof = pd.DataFrame(ndvi_checkpoint_candidates)

print("Forecast checkpoints:", len(ndvi_checkpoint_asof))

print("\nNDVI status:")
display(
    ndvi_checkpoint_asof["NDVI_STATUS"].value_counts()
)

print("\nNDVI lag distribution:")
display(
    ndvi_checkpoint_asof["NDVI_LAG_DAYS"].describe()
)

print("\nCheckpoint-level NDVI alignment:")
display(
    ndvi_checkpoint_asof.head(20)
)

Forecast checkpoints: 111

NDVI status:


NDVI_STATUS
PRIOR_NDVI_AVAILABLE    110
NO_PRIOR_NDVI             1
Name: count, dtype: int64


NDVI lag distribution:


count    110.000000
mean       7.772727
std        5.444733
min        4.000000
25%        5.000000
50%        7.000000
75%        9.000000
max       37.000000
Name: NDVI_LAG_DAYS, dtype: float64


Checkpoint-level NDVI alignment:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,CHECKPOINT_SEASON_DAY,NDVI_SEASON_DAY,NDVI_VALUE,NDVI_LAG_DAYS,NDVI_STATUS
0,FLD007,2023,1,7,3.0,0.26,4.0,PRIOR_NDVI_AVAILABLE
1,FLD007,2023,3,21,17.0,0.38,4.0,PRIOR_NDVI_AVAILABLE
2,FLD007,2023,5,35,31.0,0.40,4.0,PRIOR_NDVI_AVAILABLE
3,FLD007,2023,7,49,45.0,0.48,4.0,PRIOR_NDVI_AVAILABLE
4,FLD007,2023,9,63,59.0,0.57,4.0,PRIOR_NDVI_AVAILABLE
5,FLD007,2023,11,77,73.0,0.66,4.0,PRIOR_NDVI_AVAILABLE
6,FLD007,2024,1,10,3.0,0.34,7.0,PRIOR_NDVI_AVAILABLE
7,FLD007,2024,3,24,17.0,0.39,7.0,PRIOR_NDVI_AVAILABLE
8,FLD007,2024,5,38,31.0,0.51,7.0,PRIOR_NDVI_AVAILABLE
9,FLD007,2024,7,52,45.0,0.57,7.0,PRIOR_NDVI_AVAILABLE


In [127]:
ndvi_long_lag = ndvi_checkpoint_asof[
    ndvi_checkpoint_asof["NDVI_LAG_DAYS"] > 14
].copy()

print(
    "NDVI checkpoints with lag > 14 days:",
    len(ndvi_long_lag)
)

display(
    ndvi_long_lag[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "CHECKPOINT_SEASON_DAY",
            "NDVI_SEASON_DAY",
            "NDVI_VALUE",
            "NDVI_LAG_DAYS",
            "NDVI_STATUS"
        ]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    )
)


NDVI checkpoints with lag > 14 days: 6


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,CHECKPOINT_SEASON_DAY,NDVI_SEASON_DAY,NDVI_VALUE,NDVI_LAG_DAYS,NDVI_STATUS
41,FLD014,2026,5,35,17.0,0.67,18.0,PRIOR_NDVI_AVAILABLE
42,FLD014,2026,7,49,17.0,0.67,32.0,PRIOR_NDVI_AVAILABLE
66,FLD029,2026,7,54,31.0,0.68,23.0,PRIOR_NDVI_AVAILABLE
67,FLD029,2026,9,68,31.0,0.68,37.0,PRIOR_NDVI_AVAILABLE
107,FLD036,2026,5,38,17.0,2.20,21.0,PRIOR_NDVI_AVAILABLE
108,FLD036,2026,7,52,17.0,2.20,35.0,PRIOR_NDVI_AVAILABLE


In [128]:
ndvi_2026_checkpoints = ndvi_checkpoint_asof[
    ndvi_checkpoint_asof["SEASON_YEAR"] == 2026
].copy()

display(
    ndvi_2026_checkpoints[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "CHECKPOINT_SEASON_DAY",
            "NDVI_SEASON_DAY",
            "NDVI_VALUE",
            "NDVI_LAG_DAYS",
            "NDVI_STATUS"
        ]
    ].sort_values(
        ["FIELD_ID", "WEEK_INDEX"]
    ).to_string(index=False)
)

'FIELD_ID  SEASON_YEAR  WEEK_INDEX  CHECKPOINT_SEASON_DAY  NDVI_SEASON_DAY  NDVI_VALUE  NDVI_LAG_DAYS          NDVI_STATUS\n  FLD007         2026           1                      9              NaN         NaN            NaN        NO_PRIOR_NDVI\n  FLD007         2026           3                     23             17.0        0.73            6.0 PRIOR_NDVI_AVAILABLE\n  FLD007         2026           7                     51             45.0        0.67            6.0 PRIOR_NDVI_AVAILABLE\n  FLD007         2026           9                     65             59.0        0.73            6.0 PRIOR_NDVI_AVAILABLE\n  FLD007         2026          11                     79             73.0        0.72            6.0 PRIOR_NDVI_AVAILABLE\n  FLD014         2026           1                      7              3.0        0.64            4.0 PRIOR_NDVI_AVAILABLE\n  FLD014         2026           5                     35             17.0        0.67           18.0 PRIOR_NDVI_AVAILABLE\n  FLD014       

In [129]:
ndvi_quality_check = ndvi_for_forecast[
    ndvi_for_forecast["FIELD_ID"].isin(
        ["FLD007", "FLD014", "FLD029", "FLD036"]
    ) &
    (ndvi_for_forecast["SEASON_YEAR"] == 2026)
].copy()

display(
    ndvi_quality_check[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "NDVI_VALUE",
            "EXPECTED_NDVI",
            "NDVI_DEVIATION",
            "NDVI_CHANGE",
            "NDVI_AVAILABLE",
            "NDVI_RANGE_VALID",
            "NDVI_CHANGE_VALID"
        ]
    ].sort_values(
        ["FIELD_ID", "WEEK_INDEX"]
    ).to_string(index=False)
)

'FIELD_ID  SEASON_YEAR  WEEK_INDEX  SEASON_DAY  NDVI_VALUE  EXPECTED_NDVI  NDVI_DEVIATION  NDVI_CHANGE  NDVI_AVAILABLE  NDVI_RANGE_VALID  NDVI_CHANGE_VALID\n  FLD007         2026           0           3         NaN          0.311             NaN          NaN               0               NaN                NaN\n  FLD007         2026           1          10         NaN            NaN             NaN          NaN               0               NaN                NaN\n  FLD007         2026           2          17        0.73          0.394             NaN          NaN               1               1.0                NaN\n  FLD007         2026           3          24         NaN            NaN             NaN          NaN               0               NaN                NaN\n  FLD007         2026           4          31        0.64          0.471           0.169          NaN               1               1.0                NaN\n  FLD007         2026           6          45        0.67      

In [130]:
# Start from the checkpoint-level as-of NDVI alignment
ndvi_forecast_input = ndvi_checkpoint_asof.copy()

# Bring upstream NDVI quality information for the selected observation
ndvi_quality = ndvi_for_forecast[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY",
        "NDVI_VALUE",
        "NDVI_AVAILABLE",
        "NDVI_RANGE_VALID",
        "NDVI_CHANGE_VALID"
    ]
].copy()

# Match the selected NDVI observation by its actual seasonal observation day
ndvi_forecast_input = ndvi_forecast_input.merge(
    ndvi_quality.rename(
        columns={
            "WEEK_INDEX": "NDVI_OBS_WEEK_INDEX",
            "SEASON_DAY": "NDVI_SEASON_DAY_JOIN",
            "NDVI_VALUE": "NDVI_SOURCE_VALUE"
        }
    ),
    left_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "NDVI_SEASON_DAY"
    ],
    right_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "NDVI_SEASON_DAY_JOIN"
    ],
    how="left",
    validate="one_to_one"
)

# A selected NDVI is usable only when:
# - a real observation exists
# - NDVI_AVAILABLE = 1
# - the source marks it as range-valid
ndvi_forecast_input["NDVI_VALID_FOR_FORECAST"] = (
    ndvi_forecast_input["NDVI_SOURCE_VALUE"].notna()
    & (ndvi_forecast_input["NDVI_AVAILABLE"] == 1)
    & (ndvi_forecast_input["NDVI_RANGE_VALID"] == 1)
)

# Do NOT carry stale/invalid values as current NDVI.
ndvi_forecast_input["NDVI_FORECAST_VALUE"] = np.where(
    ndvi_forecast_input["NDVI_VALID_FOR_FORECAST"],
    ndvi_forecast_input["NDVI_SOURCE_VALUE"],
    np.nan
)

# Explicitly distinguish freshness.
ndvi_forecast_input["NDVI_FRESHNESS_STATUS"] = np.select(
    [
        ndvi_forecast_input["NDVI_FORECAST_VALUE"].isna()
        & ndvi_forecast_input["NDVI_SOURCE_VALUE"].notna()
        & (ndvi_forecast_input["NDVI_RANGE_VALID"] == 0),

        ndvi_forecast_input["NDVI_FORECAST_VALUE"].notna()
        & (ndvi_forecast_input["NDVI_LAG_DAYS"] <= 14),

        ndvi_forecast_input["NDVI_FORECAST_VALUE"].notna()
        & (ndvi_forecast_input["NDVI_LAG_DAYS"] > 14),

        ndvi_forecast_input["NDVI_SOURCE_VALUE"].isna()
    ],
    [
        "INVALID_SOURCE_NDVI",
        "VALID_RECENT_NDVI",
        "VALID_STALE_NDVI",
        "NO_PRIOR_NDVI"
    ],
    default="REVIEW"
)

print("Forecast checkpoints:", len(ndvi_forecast_input))

print("\nNDVI freshness status:")
display(
    ndvi_forecast_input["NDVI_FRESHNESS_STATUS"].value_counts()
)

print("\nInvalid source NDVI:")
display(
    ndvi_forecast_input[
        ndvi_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "INVALID_SOURCE_NDVI"
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "NDVI_SEASON_DAY",
            "NDVI_SOURCE_VALUE",
            "NDVI_RANGE_VALID",
            "NDVI_CHANGE_VALID",
            "NDVI_LAG_DAYS"
        ]
    ]
)

print("\nStale valid NDVI:")
display(
    ndvi_forecast_input[
        ndvi_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "VALID_STALE_NDVI"
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "CHECKPOINT_SEASON_DAY",
            "NDVI_SEASON_DAY",
            "NDVI_SOURCE_VALUE",
            "NDVI_LAG_DAYS"
        ]
    ]
)

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge

In [131]:
# Start from the checkpoint-level as-of NDVI alignment
ndvi_forecast_input = ndvi_checkpoint_asof.copy()

# Select the source-quality fields from the NDVI feature table
ndvi_quality = ndvi_for_forecast[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "SEASON_DAY",
        "NDVI_VALUE",
        "NDVI_AVAILABLE",
        "NDVI_RANGE_VALID",
        "NDVI_CHANGE_VALID"
    ]
].copy()

# Rename source columns so they do not collide with checkpoint columns
ndvi_quality = ndvi_quality.rename(
    columns={
        "NDVI_VALUE": "NDVI_SOURCE_VALUE",
        "NDVI_AVAILABLE": "NDVI_SOURCE_AVAILABLE",
        "NDVI_RANGE_VALID": "NDVI_SOURCE_RANGE_VALID",
        "NDVI_CHANGE_VALID": "NDVI_SOURCE_CHANGE_VALID"
    }
)

# Match the selected prior NDVI observation
ndvi_forecast_input = ndvi_forecast_input.merge(
    ndvi_quality,
    left_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "NDVI_SEASON_DAY"
    ],
    right_on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "SEASON_DAY"
    ],
    how="left"
)

# A source observation is valid only when it is actually available
# and the upstream range check marks it as valid.
ndvi_forecast_input["NDVI_VALID_FOR_FORECAST"] = (
    ndvi_forecast_input["NDVI_SOURCE_VALUE"].notna()
    & (ndvi_forecast_input["NDVI_SOURCE_AVAILABLE"] == 1)
    & (ndvi_forecast_input["NDVI_SOURCE_RANGE_VALID"] == 1)
)

# Do not use invalid NDVI values as forecast features.
ndvi_forecast_input["NDVI_FORECAST_VALUE"] = np.where(
    ndvi_forecast_input["NDVI_VALID_FOR_FORECAST"],
    ndvi_forecast_input["NDVI_SOURCE_VALUE"],
    np.nan
)

# Classify freshness / availability.
ndvi_forecast_input["NDVI_FRESHNESS_STATUS"] = np.select(
    [
        (
            ndvi_forecast_input["NDVI_SOURCE_VALUE"].notna()
            & (ndvi_forecast_input["NDVI_SOURCE_RANGE_VALID"] == 0)
        ),

        (
            ndvi_forecast_input["NDVI_FORECAST_VALUE"].notna()
            & (ndvi_forecast_input["NDVI_LAG_DAYS"] <= 14)
        ),

        (
            ndvi_forecast_input["NDVI_FORECAST_VALUE"].notna()
            & (ndvi_forecast_input["NDVI_LAG_DAYS"] > 14)
        ),

        ndvi_forecast_input["NDVI_SOURCE_VALUE"].isna()
    ],
    [
        "INVALID_SOURCE_NDVI",
        "VALID_RECENT_NDVI",
        "VALID_STALE_NDVI",
        "NO_PRIOR_NDVI"
    ],
    default="REVIEW"
)

print("Forecast checkpoints:", len(ndvi_forecast_input))

print("\nNDVI freshness status:")
display(
    ndvi_forecast_input["NDVI_FRESHNESS_STATUS"].value_counts()
)

print("\nInvalid source NDVI:")
display(
    ndvi_forecast_input[
        ndvi_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "INVALID_SOURCE_NDVI"
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "NDVI_SEASON_DAY",
            "NDVI_SOURCE_VALUE",
            "NDVI_SOURCE_RANGE_VALID",
            "NDVI_SOURCE_CHANGE_VALID",
            "NDVI_LAG_DAYS"
        ]
    ]
)

print("\nStale valid NDVI:")
display(
    ndvi_forecast_input[
        ndvi_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "VALID_STALE_NDVI"
    ][
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "CHECKPOINT_SEASON_DAY",
            "NDVI_SEASON_DAY",
            "NDVI_SOURCE_VALUE",
            "NDVI_LAG_DAYS"
        ]
    ]
)

Forecast checkpoints: 111

NDVI freshness status:


NDVI_FRESHNESS_STATUS
VALID_RECENT_NDVI      102
VALID_STALE_NDVI         4
INVALID_SOURCE_NDVI      4
NO_PRIOR_NDVI            1
Name: count, dtype: int64


Invalid source NDVI:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,NDVI_SEASON_DAY,NDVI_SOURCE_VALUE,NDVI_SOURCE_RANGE_VALID,NDVI_SOURCE_CHANGE_VALID,NDVI_LAG_DAYS
106,FLD036,2026,3,17.0,2.2,0.0,0.0,7.0
107,FLD036,2026,5,17.0,2.2,0.0,0.0,21.0
108,FLD036,2026,7,17.0,2.2,0.0,0.0,35.0
109,FLD036,2026,9,59.0,9.9,0.0,0.0,7.0



Stale valid NDVI:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,CHECKPOINT_SEASON_DAY,NDVI_SEASON_DAY,NDVI_SOURCE_VALUE,NDVI_LAG_DAYS
41,FLD014,2026,5,35,17.0,0.67,18.0
42,FLD014,2026,7,49,17.0,0.67,32.0
66,FLD029,2026,7,54,31.0,0.68,23.0
67,FLD029,2026,9,68,31.0,0.68,37.0


In [132]:
# Final NDVI feature columns for the forecast input

ndvi_final = ndvi_forecast_input[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "CHECKPOINT_SEASON_DAY",
        "NDVI_FORECAST_VALUE",
        "NDVI_LAG_DAYS",
        "NDVI_FRESHNESS_STATUS",
        "NDVI_VALID_FOR_FORECAST"
    ]
].copy()

# Rename checkpoint season day to the standard forecast-input name
ndvi_final = ndvi_final.rename(
    columns={
        "CHECKPOINT_SEASON_DAY": "SEASON_DAY",
        "NDVI_FORECAST_VALUE": "NDVI_VALUE"
    }
)

print("NDVI forecast rows:", len(ndvi_final))
print(
    "Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX:",
    ndvi_final[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    ].drop_duplicates().shape[0]
)

print("\nMissing NDVI_VALUE:")
print(
    ndvi_final["NDVI_VALUE"].isna().sum()
)

print("\nNDVI status:")
display(
    ndvi_final["NDVI_FRESHNESS_STATUS"].value_counts()
)

print("\nFinal NDVI sample:")
display(
    ndvi_final.head(20)
)

NDVI forecast rows: 111
Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX: 111

Missing NDVI_VALUE:
5

NDVI status:


NDVI_FRESHNESS_STATUS
VALID_RECENT_NDVI      102
VALID_STALE_NDVI         4
INVALID_SOURCE_NDVI      4
NO_PRIOR_NDVI            1
Name: count, dtype: int64


Final NDVI sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,NDVI_VALUE,NDVI_LAG_DAYS,NDVI_FRESHNESS_STATUS,NDVI_VALID_FOR_FORECAST
0,FLD007,2023,1,7,0.26,4.0,VALID_RECENT_NDVI,True
1,FLD007,2023,3,21,0.38,4.0,VALID_RECENT_NDVI,True
2,FLD007,2023,5,35,0.40,4.0,VALID_RECENT_NDVI,True
3,FLD007,2023,7,49,0.48,4.0,VALID_RECENT_NDVI,True
4,FLD007,2023,9,63,0.57,4.0,VALID_RECENT_NDVI,True
5,FLD007,2023,11,77,0.66,4.0,VALID_RECENT_NDVI,True
6,FLD007,2024,1,10,0.34,7.0,VALID_RECENT_NDVI,True
7,FLD007,2024,3,24,0.39,7.0,VALID_RECENT_NDVI,True
8,FLD007,2024,5,38,0.51,7.0,VALID_RECENT_NDVI,True
9,FLD007,2024,7,52,0.57,7.0,VALID_RECENT_NDVI,True


In [133]:
sde_for_forecast = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY,
        DEVIATION_STATE,
        DEVIATION_LEVEL,
        DEVIATION_DIRECTION,
        DOMINANT_DEVIATION_DRIVER,
        WEATHER_DEVIATION,
        GDD_DEVIATION,
        NDVI_DEVIATION,
        PEST_DEVIATION,
        DISEASE_DEVIATION,
        SIGNALS_ABNORMAL_COUNT,
        SIGNALS_CONFIRMING_COUNT,
        MULTI_SIGNAL_CONFIRMED,
        DEVIATION_CONFIDENCE,
        MISSING_SIGNAL_COUNT,
        MODEL_VERSION
    FROM YIELD_FORECAST_DB.POD_VERTEX.SEASON_DEVIATION_RESULT
    """,
    conn
)

sde_for_forecast["SEASON_YEAR"] = (
    sde_for_forecast["SEASON_YEAR"].astype(int)
)

sde_for_forecast["WEEK_INDEX"] = (
    sde_for_forecast["WEEK_INDEX"].astype(int)
)

print("SDE rows:", len(sde_for_forecast))

print(
    "Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX:",
    sde_for_forecast[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    ].drop_duplicates().shape[0]
)

print("\nDeviation state distribution:")
display(
    sde_for_forecast["DEVIATION_STATE"].value_counts()
)

print("\nMissing values:")
display(
    sde_for_forecast.isna().sum()
)

print("\nSDE sample:")
display(
    sde_for_forecast.head(10)
)

/tmp/ipykernel_75164/1647597097.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sde_for_forecast = pd.read_sql(


SDE rows: 111
Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX: 111

Deviation state distribution:


DEVIATION_STATE
WATCH        47
DEVIATING    37
CONFIRMED    27
Name: count, dtype: int64


Missing values:


FIELD_ID                     0
SEASON_YEAR                  0
WEEK_INDEX                   0
SEASON_DAY                   0
DEVIATION_STATE              0
DEVIATION_LEVEL              0
DEVIATION_DIRECTION          0
DOMINANT_DEVIATION_DRIVER    0
WEATHER_DEVIATION            0
GDD_DEVIATION                0
NDVI_DEVIATION               0
PEST_DEVIATION               0
DISEASE_DEVIATION            0
SIGNALS_ABNORMAL_COUNT       0
SIGNALS_CONFIRMING_COUNT     0
MULTI_SIGNAL_CONFIRMED       0
DEVIATION_CONFIDENCE         0
MISSING_SIGNAL_COUNT         0
MODEL_VERSION                0
dtype: int64


SDE sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,DEVIATION_STATE,DEVIATION_LEVEL,DEVIATION_DIRECTION,DOMINANT_DEVIATION_DRIVER,WEATHER_DEVIATION,GDD_DEVIATION,NDVI_DEVIATION,PEST_DEVIATION,DISEASE_DEVIATION,SIGNALS_ABNORMAL_COUNT,SIGNALS_CONFIRMING_COUNT,MULTI_SIGNAL_CONFIRMED,DEVIATION_CONFIDENCE,MISSING_SIGNAL_COUNT,MODEL_VERSION
0,FLD007,2023,1,7,WATCH,LOW,NONE,PEST,0.0,-40.9,0.000,-0.40,0.50,2,0,False,LOW,0,SDE_V1.0
1,FLD007,2023,3,21,WATCH,LOW,NONE,PEST,0.0,-30.2,-0.055,-0.20,-2.90,4,0,False,LOW,0,SDE_V1.0
2,FLD007,2023,5,35,DEVIATING,MEDIUM,NEGATIVE,NDVI,-0.6,-60.3,-0.045,0.20,0.60,4,1,False,MEDIUM,0,SDE_V1.0
3,FLD007,2023,7,49,CONFIRMED,MEDIUM,NEGATIVE,PEST,0.0,-47.6,-0.055,-4.35,0.05,3,2,True,MEDIUM,0,SDE_V1.0
4,FLD007,2023,9,63,DEVIATING,MEDIUM,NEGATIVE,PEST,-2.2,-71.7,-0.025,11.90,-0.70,3,1,False,MEDIUM,0,SDE_V1.0
5,FLD007,2023,11,77,DEVIATING,MEDIUM,NEGATIVE,PEST,-1.2,-95.7,-0.035,-5.65,3.65,3,1,False,MEDIUM,0,SDE_V1.0
6,FLD007,2024,1,10,WATCH,LOW,NONE,PEST,0.6,57.9,0.040,0.40,0.50,4,0,False,LOW,0,SDE_V1.0
7,FLD007,2024,3,24,WATCH,LOW,NONE,WEATHER,2.8,103.8,0.005,0.20,-2.90,4,0,False,LOW,0,SDE_V1.0
8,FLD007,2024,5,38,CONFIRMED,HIGH,POSITIVE,PEST,2.2,135.7,0.055,0.70,0.60,4,3,True,HIGH,0,SDE_V1.0
9,FLD007,2024,7,52,CONFIRMED,HIGH,POSITIVE,GDD,2.3,191.7,0.045,4.35,0.05,3,3,True,HIGH,0,SDE_V1.0


In [134]:
sde_features = sde_for_forecast[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY",
        "DEVIATION_STATE",
        "DEVIATION_LEVEL",
        "DEVIATION_DIRECTION",
        "DOMINANT_DEVIATION_DRIVER",
        "WEATHER_DEVIATION",
        "GDD_DEVIATION",
        "NDVI_DEVIATION",
        "PEST_DEVIATION",
        "DISEASE_DEVIATION",
        "SIGNALS_ABNORMAL_COUNT",
        "SIGNALS_CONFIRMING_COUNT",
        "MULTI_SIGNAL_CONFIRMED",
        "DEVIATION_CONFIDENCE",
        "MISSING_SIGNAL_COUNT",
        "MODEL_VERSION"
    ]
].copy()

# Merge SDE onto the established forecast checkpoint grain.
forecast_with_sde = forecast_timeline.merge(
    sde_features,
    on=[
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX",
        "SEASON_DAY"
    ],
    how="left",
    validate="one_to_one"
)

print("Forecast rows after SDE merge:", len(forecast_with_sde))

print(
    "Unique forecast grain:",
    forecast_with_sde[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    ].drop_duplicates().shape[0]
)

print("\nMissing SDE values:")
display(
    forecast_with_sde[
        sde_features.columns.difference(
            ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
        )
    ].isna().sum()
)

print("\nSDE state distribution after merge:")
display(
    forecast_with_sde["DEVIATION_STATE"].value_counts()
)

print("\nSample:")
display(
    forecast_with_sde.head(10)
)

Forecast rows after SDE merge: 111
Unique forecast grain: 111

Missing SDE values:


DEVIATION_CONFIDENCE         0
DEVIATION_DIRECTION          0
DEVIATION_LEVEL              0
DEVIATION_STATE              0
DISEASE_DEVIATION            0
DOMINANT_DEVIATION_DRIVER    0
GDD_DEVIATION                0
MISSING_SIGNAL_COUNT         0
MODEL_VERSION                0
MULTI_SIGNAL_CONFIRMED       0
NDVI_DEVIATION               0
PEST_DEVIATION               0
SIGNALS_ABNORMAL_COUNT       0
SIGNALS_CONFIRMING_COUNT     0
WEATHER_DEVIATION            0
dtype: int64


SDE state distribution after merge:


DEVIATION_STATE
WATCH        47
DEVIATING    37
CONFIRMED    27
Name: count, dtype: int64


Sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,DEVIATION_STATE,DEVIATION_LEVEL,DEVIATION_DIRECTION,DOMINANT_DEVIATION_DRIVER,WEATHER_DEVIATION,GDD_DEVIATION,NDVI_DEVIATION,PEST_DEVIATION,DISEASE_DEVIATION,SIGNALS_ABNORMAL_COUNT,SIGNALS_CONFIRMING_COUNT,MULTI_SIGNAL_CONFIRMED,DEVIATION_CONFIDENCE,MISSING_SIGNAL_COUNT,MODEL_VERSION
0,FLD007,2023,1,7,WATCH,LOW,NONE,PEST,0.0,-40.9,0.000,-0.40,0.50,2,0,False,LOW,0,SDE_V1.0
1,FLD007,2023,3,21,WATCH,LOW,NONE,PEST,0.0,-30.2,-0.055,-0.20,-2.90,4,0,False,LOW,0,SDE_V1.0
2,FLD007,2023,5,35,DEVIATING,MEDIUM,NEGATIVE,NDVI,-0.6,-60.3,-0.045,0.20,0.60,4,1,False,MEDIUM,0,SDE_V1.0
3,FLD007,2023,7,49,CONFIRMED,MEDIUM,NEGATIVE,PEST,0.0,-47.6,-0.055,-4.35,0.05,3,2,True,MEDIUM,0,SDE_V1.0
4,FLD007,2023,9,63,DEVIATING,MEDIUM,NEGATIVE,PEST,-2.2,-71.7,-0.025,11.90,-0.70,3,1,False,MEDIUM,0,SDE_V1.0
5,FLD007,2023,11,77,DEVIATING,MEDIUM,NEGATIVE,PEST,-1.2,-95.7,-0.035,-5.65,3.65,3,1,False,MEDIUM,0,SDE_V1.0
6,FLD007,2024,1,10,WATCH,LOW,NONE,PEST,0.6,57.9,0.040,0.40,0.50,4,0,False,LOW,0,SDE_V1.0
7,FLD007,2024,3,24,WATCH,LOW,NONE,WEATHER,2.8,103.8,0.005,0.20,-2.90,4,0,False,LOW,0,SDE_V1.0
8,FLD007,2024,5,38,CONFIRMED,HIGH,POSITIVE,PEST,2.2,135.7,0.055,0.70,0.60,4,3,True,HIGH,0,SDE_V1.0
9,FLD007,2024,7,52,CONFIRMED,HIGH,POSITIVE,GDD,2.3,191.7,0.045,4.35,0.05,3,3,True,HIGH,0,SDE_V1.0


In [135]:
risk_for_forecast = pd.read_sql(
    """
    SELECT
        FIELD_ID,
        SEASON_YEAR,
        WEEK_INDEX,
        SEASON_DAY,
        PEST_RISK_SCORE,
        GDD_RISK_SCORE,
        WEATHER_DISEASE_RISK_SCORE,
        NDVI_STRESS_SCORE,
        HISTORICAL_DEVIATION_RISK_SCORE,
        OVERALL_RISK_SCORE,
        OVERALL_RISK_LOWER_BOUND,
        OVERALL_RISK_UPPER_BOUND,
        RISK_LEVEL,
        IS_PROVISIONAL,
        AVAILABLE_WEIGHT_PCT,
        DOMINANT_RISK_DRIVER,
        CONFLICTING_SIGNALS_FLAG,
        CONF_DATA_COMPLETENESS,
        CONF_PEST_FRESHNESS,
        CONF_NDVI_QUALITY,
        CONF_DISEASE_CONFIDENCE,
        CONFIDENCE_SCORE,
        CONFIDENCE_LEVEL,
        AVAILABLE_SIGNAL_COUNT,
        MISSING_SIGNAL_COUNT,
        MISSING_SIGNALS,
        DEGRADED_SIGNALS
    FROM YIELD_FORECAST_DB.POD_VERTEX.FIELD_RISK_CONFIDENCE_SCORECARD_PB
    """,
    conn
)

risk_for_forecast["SEASON_YEAR"] = (
    risk_for_forecast["SEASON_YEAR"].astype(int)
)

risk_for_forecast["WEEK_INDEX"] = (
    risk_for_forecast["WEEK_INDEX"].astype(int)
)

print("Risk rows:", len(risk_for_forecast))

print(
    "Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX:",
    risk_for_forecast[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"]
    ].drop_duplicates().shape[0]
)

print("\nMissing values:")
display(
    risk_for_forecast.isna().sum()
)

print("\nRisk level distribution:")
display(
    risk_for_forecast["RISK_LEVEL"].value_counts(dropna=False)
)

print("\nConfidence level distribution:")
display(
    risk_for_forecast["CONFIDENCE_LEVEL"].value_counts(dropna=False)
)

print("\nRisk sample:")
display(
    risk_for_forecast.head(10)
)


/tmp/ipykernel_75164/3762733952.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  risk_for_forecast = pd.read_sql(


Risk rows: 1475
Unique FIELD_ID + SEASON_YEAR + WEEK_INDEX: 1475

Missing values:


FIELD_ID                              0
SEASON_YEAR                           0
WEEK_INDEX                            0
SEASON_DAY                            0
PEST_RISK_SCORE                      71
GDD_RISK_SCORE                      217
WEATHER_DISEASE_RISK_SCORE           41
NDVI_STRESS_SCORE                    81
HISTORICAL_DEVIATION_RISK_SCORE      50
OVERALL_RISK_SCORE                    0
OVERALL_RISK_LOWER_BOUND              0
OVERALL_RISK_UPPER_BOUND              0
RISK_LEVEL                            0
IS_PROVISIONAL                        0
AVAILABLE_WEIGHT_PCT                  0
DOMINANT_RISK_DRIVER                  0
CONFLICTING_SIGNALS_FLAG              0
CONF_DATA_COMPLETENESS                0
CONF_PEST_FRESHNESS                   0
CONF_NDVI_QUALITY                     0
CONF_DISEASE_CONFIDENCE               0
CONFIDENCE_SCORE                      0
CONFIDENCE_LEVEL                      0
AVAILABLE_SIGNAL_COUNT                0
MISSING_SIGNAL_COUNT                  0



Risk level distribution:


RISK_LEVEL
MODERATE             827
LOW                  552
HIGH                  71
INSUFFICIENT_DATA     25
Name: count, dtype: int64


Confidence level distribution:


CONFIDENCE_LEVEL
MEDIUM    730
HIGH      679
LOW        66
Name: count, dtype: int64


Risk sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,PEST_RISK_SCORE,GDD_RISK_SCORE,WEATHER_DISEASE_RISK_SCORE,NDVI_STRESS_SCORE,HISTORICAL_DEVIATION_RISK_SCORE,OVERALL_RISK_SCORE,...,CONF_DATA_COMPLETENESS,CONF_PEST_FRESHNESS,CONF_NDVI_QUALITY,CONF_DISEASE_CONFIDENCE,CONFIDENCE_SCORE,CONFIDENCE_LEVEL,AVAILABLE_SIGNAL_COUNT,MISSING_SIGNAL_COUNT,MISSING_SIGNALS,DEGRADED_SIGNALS
0,FLD001,2026,0,3,0.0000,47.1079,37.3167,0.0,NaN,21.3781,...,90.0,100.0000,56.1429,0.0000,75.4214,HIGH,4,1,HISTORICAL(MISSING),WEATHER_DISEASE(PARTIAL_NO_DISEASE)
1,FLD001,2026,1,10,0.0000,69.6409,19.6000,0.0,NaN,23.7003,...,90.0,57.1429,47.7214,70.0000,76.0868,HIGH,4,1,HISTORICAL(MISSING),None
2,FLD001,2026,2,17,10.8614,60.5474,17.6800,0.0,NaN,24.3681,...,90.0,100.0000,24.5714,50.0000,78.1857,HIGH,4,1,HISTORICAL(MISSING),None
3,FLD001,2026,3,24,10.8614,61.8998,29.5000,0.0,NaN,27.3704,...,90.0,57.1429,20.8857,80.0000,73.5614,MEDIUM,4,1,HISTORICAL(MISSING),None
4,FLD001,2026,4,31,11.2241,49.7125,26.9333,0.0,NaN,23.5356,...,90.0,100.0000,98.0000,57.1429,90.2714,HIGH,4,1,HISTORICAL(MISSING),None
5,FLD001,2026,5,38,11.2241,44.1246,23.7467,0.0,NaN,21.2752,...,90.0,57.1429,83.3000,60.0000,79.9236,HIGH,4,1,HISTORICAL(MISSING),None
6,FLD001,2026,6,45,11.2241,33.7331,20.0267,0.0,NaN,17.5620,...,90.0,7.1429,58.1429,42.8571,63.5786,MEDIUM,4,1,HISTORICAL(MISSING),None
7,FLD001,2026,7,52,NaN,29.8265,22.2533,0.0,NaN,19.8455,...,60.0,0.0000,49.4214,52.0000,53.2132,MEDIUM,3,2,"PEST(STALE), HISTORICAL(MISSING)",None
8,FLD001,2026,8,59,60.3208,NaN,19.7961,60.0,NaN,47.7776,...,65.0,100.0000,33.7143,37.1429,70.1286,MEDIUM,3,2,"GDD(MISSING), HISTORICAL(MISSING)",None
9,FLD001,2026,9,66,60.3208,NaN,26.0000,60.0,NaN,49.6865,...,65.0,57.1429,28.6571,80.0000,67.2271,MEDIUM,3,2,"GDD(MISSING), HISTORICAL(MISSING)",None


In [136]:
# Compare the forecast checkpoint grain with available risk weeks

forecast_keys = forecast_with_sde[
    ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
].copy()

risk_key_summary = (
    risk_for_forecast[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ]
    .sort_values(["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX"])
)

print("Forecast checkpoints:", len(forecast_keys))
print("Risk rows:", len(risk_key_summary))

print("\nForecast WEEK_INDEX values:")
print(sorted(forecast_keys["WEEK_INDEX"].unique()))

print("\nRisk WEEK_INDEX values:")
print(sorted(risk_key_summary["WEEK_INDEX"].unique()))

print("\nRisk coverage for forecast grain:")

risk_coverage = forecast_keys.merge(
    risk_key_summary,
    on=["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"],
    how="left",
    indicator=True
)

print(
    risk_coverage["_merge"].value_counts()
)

print("\nMissing exact risk checkpoints:")
display(
    risk_coverage[
        risk_coverage["_merge"] == "left_only"
    ].head(20)
)

Forecast checkpoints: 111
Risk rows: 1475

Forecast WEEK_INDEX values:
[1, 3, 5, 7, 9, 11]

Risk WEEK_INDEX values:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Risk coverage for forecast grain:
_merge
left_only     87
both          24
right_only     0
Name: count, dtype: int64

Missing exact risk checkpoints:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,_merge
0,FLD007,2023,1,7,left_only
1,FLD007,2023,3,21,left_only
2,FLD007,2023,5,35,left_only
3,FLD007,2023,7,49,left_only
4,FLD007,2023,9,63,left_only
5,FLD007,2023,11,77,left_only
12,FLD007,2025,1,9,left_only
13,FLD007,2025,3,23,left_only
14,FLD007,2025,5,37,left_only
15,FLD007,2025,7,51,left_only


In [137]:
# Leakage-safe as-of Risk alignment
# For each forecast checkpoint, use only Risk information available
# at or before that forecast SEASON_DAY.

risk_asof_results = []

for _, forecast_row in forecast_with_sde[
    ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
].iterrows():

    field_id = forecast_row["FIELD_ID"]
    season_year = forecast_row["SEASON_YEAR"]
    forecast_day = forecast_row["SEASON_DAY"]

    risk_rows = risk_for_forecast[
        (risk_for_forecast["FIELD_ID"] == field_id)
        & (risk_for_forecast["SEASON_YEAR"] == season_year)
        & (risk_for_forecast["SEASON_DAY"] <= forecast_day)
    ].sort_values("SEASON_DAY")

    if len(risk_rows) == 0:
        risk_asof_results.append({
            "FIELD_ID": field_id,
            "SEASON_YEAR": season_year,
            "WEEK_INDEX": forecast_row["WEEK_INDEX"],
            "SEASON_DAY": forecast_day,
            "RISK_SOURCE_WEEK_INDEX": np.nan,
            "RISK_SOURCE_SEASON_DAY": np.nan,
            "RISK_LAG_DAYS": np.nan
        })
    else:
        selected = risk_rows.iloc[-1]

        risk_asof_results.append({
            "FIELD_ID": field_id,
            "SEASON_YEAR": season_year,
            "WEEK_INDEX": forecast_row["WEEK_INDEX"],
            "SEASON_DAY": forecast_day,
            "RISK_SOURCE_WEEK_INDEX": selected["WEEK_INDEX"],
            "RISK_SOURCE_SEASON_DAY": selected["SEASON_DAY"],
            "RISK_LAG_DAYS": (
                forecast_day - selected["SEASON_DAY"]
            )
        })

risk_asof_check = pd.DataFrame(risk_asof_results)

print("Forecast checkpoints:", len(risk_asof_check))
print(
    "Checkpoints with prior Risk:",
    risk_asof_check["RISK_SOURCE_SEASON_DAY"].notna().sum()
)
print(
    "Checkpoints without prior Risk:",
    risk_asof_check["RISK_SOURCE_SEASON_DAY"].isna().sum()
)

print("\nRisk lag distribution:")
display(
    risk_asof_check["RISK_LAG_DAYS"].describe()
)

print("\nRisk source week distribution:")
display(
    risk_asof_check["RISK_SOURCE_WEEK_INDEX"].value_counts(
        dropna=False
    ).sort_index()
)

print("\nPotential future-risk leakage check:")
print(
    (
        risk_asof_check["RISK_SOURCE_SEASON_DAY"]
        > risk_asof_check["SEASON_DAY"]
    ).sum()
)

print("\nSample:")
display(risk_asof_check.head(15))

Forecast checkpoints: 111
Checkpoints with prior Risk: 111
Checkpoints without prior Risk: 0

Risk lag distribution:


count    111.000000
mean       2.837838
std        2.025028
min        0.000000
25%        1.000000
50%        3.000000
75%        4.500000
max        6.000000
Name: RISK_LAG_DAYS, dtype: float64


Risk source week distribution:


RISK_SOURCE_WEEK_INDEX
0      9
1     10
2      8
3     10
4      8
5     10
6      9
7     10
8      9
9     10
10     8
11    10
Name: count, dtype: int64


Potential future-risk leakage check:
0

Sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,RISK_SOURCE_WEEK_INDEX,RISK_SOURCE_SEASON_DAY,RISK_LAG_DAYS
0,FLD007,2023,1,7,0,3,4
1,FLD007,2023,3,21,2,17,4
2,FLD007,2023,5,35,4,31,4
3,FLD007,2023,7,49,6,45,4
4,FLD007,2023,9,63,8,59,4
5,FLD007,2023,11,77,10,73,4
6,FLD007,2024,1,10,1,10,0
7,FLD007,2024,3,24,3,24,0
8,FLD007,2024,5,38,5,38,0
9,FLD007,2024,7,52,7,52,0


In [138]:
# Attach the leakage-safe Risk record to each forecast checkpoint

RISK_FEATURES = [
    "PEST_RISK_SCORE",
    "GDD_RISK_SCORE",
    "WEATHER_DISEASE_RISK_SCORE",
    "NDVI_STRESS_SCORE",
    "HISTORICAL_DEVIATION_RISK_SCORE",
    "OVERALL_RISK_SCORE",
    "OVERALL_RISK_LOWER_BOUND",
    "OVERALL_RISK_UPPER_BOUND",
    "RISK_LEVEL",
    "IS_PROVISIONAL",
    "AVAILABLE_WEIGHT_PCT",
    "DOMINANT_RISK_DRIVER",
    "CONFLICTING_SIGNALS_FLAG",
    "CONF_DATA_COMPLETENESS",
    "CONF_PEST_FRESHNESS",
    "CONF_NDVI_QUALITY",
    "CONF_DISEASE_CONFIDENCE",
    "CONFIDENCE_SCORE",
    "CONFIDENCE_LEVEL",
    "AVAILABLE_SIGNAL_COUNT",
    "MISSING_SIGNAL_COUNT",
    "MISSING_SIGNALS",
    "DEGRADED_SIGNALS"
]

risk_asof_full = []

for _, forecast_row in forecast_with_sde[
    ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
].iterrows():

    field_id = forecast_row["FIELD_ID"]
    season_year = forecast_row["SEASON_YEAR"]
    forecast_day = forecast_row["SEASON_DAY"]

    eligible = risk_for_forecast[
        (risk_for_forecast["FIELD_ID"] == field_id)
        & (risk_for_forecast["SEASON_YEAR"] == season_year)
        & (risk_for_forecast["SEASON_DAY"] <= forecast_day)
    ].sort_values("SEASON_DAY")

    selected = eligible.iloc[-1]

    row = forecast_row.to_dict()

    for col in RISK_FEATURES:
        row[col] = selected[col]

    row["RISK_SOURCE_WEEK_INDEX"] = selected["WEEK_INDEX"]
    row["RISK_SOURCE_SEASON_DAY"] = selected["SEASON_DAY"]
    row["RISK_LAG_DAYS"] = forecast_day - selected["SEASON_DAY"]

    risk_asof_full.append(row)

forecast_with_risk = pd.DataFrame(risk_asof_full)

print("Forecast rows after Risk merge:", len(forecast_with_risk))

print(
    "Unique forecast grain:",
    forecast_with_risk[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ].drop_duplicates().shape[0]
)

print("\nMissing Risk values:")
display(
    forecast_with_risk[RISK_FEATURES].isna().sum()
)

print("\nRisk level distribution:")
display(
    forecast_with_risk["RISK_LEVEL"].value_counts(dropna=False)
)

print("\nConfidence distribution:")
display(
    forecast_with_risk["CONFIDENCE_LEVEL"].value_counts(dropna=False)
)

print("\nRisk lag:")
display(
    forecast_with_risk["RISK_LAG_DAYS"].describe()
)

print("\nSample:")
display(
    forecast_with_risk[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "OVERALL_RISK_SCORE",
            "RISK_LEVEL",
            "CONFIDENCE_SCORE",
            "CONFIDENCE_LEVEL",
            "DOMINANT_RISK_DRIVER",
            "RISK_SOURCE_WEEK_INDEX",
            "RISK_LAG_DAYS"
        ]
    ].head(10)
)

Forecast rows after Risk merge: 111
Unique forecast grain: 111

Missing Risk values:


PEST_RISK_SCORE                     3
GDD_RISK_SCORE                     13
WEATHER_DISEASE_RISK_SCORE          1
NDVI_STRESS_SCORE                   4
HISTORICAL_DEVIATION_RISK_SCORE     1
OVERALL_RISK_SCORE                  0
OVERALL_RISK_LOWER_BOUND            0
OVERALL_RISK_UPPER_BOUND            0
RISK_LEVEL                          0
IS_PROVISIONAL                      0
AVAILABLE_WEIGHT_PCT                0
DOMINANT_RISK_DRIVER                0
CONFLICTING_SIGNALS_FLAG            0
CONF_DATA_COMPLETENESS              0
CONF_PEST_FRESHNESS                 0
CONF_NDVI_QUALITY                   0
CONF_DISEASE_CONFIDENCE             0
CONFIDENCE_SCORE                    0
CONFIDENCE_LEVEL                    0
AVAILABLE_SIGNAL_COUNT              0
MISSING_SIGNAL_COUNT                0
MISSING_SIGNALS                    94
DEGRADED_SIGNALS                   21
dtype: int64


Risk level distribution:


RISK_LEVEL
MODERATE             65
LOW                  42
INSUFFICIENT_DATA     2
HIGH                  2
Name: count, dtype: int64


Confidence distribution:


CONFIDENCE_LEVEL
MEDIUM    68
HIGH      41
LOW        2
Name: count, dtype: int64


Risk lag:


count    111.000000
mean       2.837838
std        2.025028
min        0.000000
25%        1.000000
50%        3.000000
75%        4.500000
max        6.000000
Name: RISK_LAG_DAYS, dtype: float64


Sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,OVERALL_RISK_SCORE,RISK_LEVEL,CONFIDENCE_SCORE,CONFIDENCE_LEVEL,DOMINANT_RISK_DRIVER,RISK_SOURCE_WEEK_INDEX,RISK_LAG_DAYS
0,FLD007,2023,1,7,35.9319,MODERATE,80.5000,HIGH,PEST,0,4
1,FLD007,2023,3,21,39.2903,MODERATE,77.9929,HIGH,PEST,2,4
2,FLD007,2023,5,35,46.9569,MODERATE,73.5357,MEDIUM,PEST,4,4
3,FLD007,2023,7,49,30.3088,MODERATE,77.5429,HIGH,PEST,6,4
4,FLD007,2023,9,63,46.2503,MODERATE,84.3357,HIGH,PEST,8,4
5,FLD007,2023,11,77,20.8315,LOW,85.0000,HIGH,PEST,10,4
6,FLD007,2024,1,10,20.3674,LOW,65.5450,MEDIUM,PEST,1,0
7,FLD007,2024,3,24,30.9275,MODERATE,70.5904,MEDIUM,PEST,3,0
8,FLD007,2024,5,38,29.9622,MODERATE,68.4957,MEDIUM,PEST,5,0
9,FLD007,2024,7,52,14.2344,LOW,64.7254,MEDIUM,PEST,7,0


In [139]:
# Profile valid historical yields available for historical-feature construction

valid_yield_history = historical_yield_target[
    historical_yield_target["YIELD_TARGET_STATUS"] == "VALID"
].copy()

print("Valid historical yield rows:", len(valid_yield_history))

print("\nUnique valid field-season combinations:")
print(
    valid_yield_history[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print("\nValid yield coverage by season:")
display(
    valid_yield_history.groupby("SEASON_YEAR")["FIELD_ID"]
    .nunique()
    .sort_index()
)

print("\nValid historical seasons per field:")
field_history_counts = (
    valid_yield_history
    .groupby("FIELD_ID")["SEASON_YEAR"]
    .nunique()
    .describe()
)

display(field_history_counts)

print("\nFields with only one valid historical season:")
one_season_fields = (
    valid_yield_history
    .groupby("FIELD_ID")["SEASON_YEAR"]
    .nunique()
)

display(
    one_season_fields[one_season_fields == 1]
)

print("\nValid yield sample:")
display(
    valid_yield_history[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "HARVEST_DATE"
        ]
    ].sort_values(
        ["FIELD_ID", "SEASON_YEAR"]
    ).head(20)
)

Valid historical yield rows: 237

Unique valid field-season combinations:
237

Valid yield coverage by season:


SEASON_YEAR
2021    39
2022    40
2023    40
2024    40
2025    40
2026    38
Name: FIELD_ID, dtype: int64


Valid historical seasons per field:


count    40.000000
mean      5.925000
std       0.349908
min       4.000000
25%       6.000000
50%       6.000000
75%       6.000000
max       6.000000
Name: SEASON_YEAR, dtype: float64


Fields with only one valid historical season:


Series([], Name: SEASON_YEAR, dtype: int64)


Valid yield sample:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,HARVEST_DATE
0,FLD001,2021,Soybean,N/A,NaT
1,FLD001,2022,Soybean,N/A,NaT
2,FLD001,2023,Soybean,N/A,NaT
3,FLD001,2024,Soybean,N/A,NaT
4,FLD001,2025,Soybean,N/A,NaT
5,FLD001,2026,Soybean,N/A,NaT
6,FLD002,2021,Corn,N/A,NaT
7,FLD002,2022,Corn,N/A,NaT
8,FLD002,2023,Corn,N/A,NaT
9,FLD002,2024,Corn,N/A,NaT


In [140]:
print("ACTUAL_YIELD_BU_ACRE dtype:")
print(valid_yield_history["ACTUAL_YIELD_BU_ACRE"].dtype)

print("\nSample raw values:")
display(
    valid_yield_history[
        ["FIELD_ID", "SEASON_YEAR", "ACTUAL_YIELD_BU_ACRE"]
    ].head(20)
)

print("\nNumeric conversion check:")

yield_numeric_check = pd.to_numeric(
    valid_yield_history["ACTUAL_YIELD_BU_ACRE"],
    errors="coerce"
)

print("Total valid rows:", len(yield_numeric_check))
print("Numeric values:", yield_numeric_check.notna().sum())
print("Non-numeric values:", yield_numeric_check.isna().sum())

print("\nNon-numeric examples:")
display(
    valid_yield_history.loc[
        yield_numeric_check.isna(),
        ["FIELD_ID", "SEASON_YEAR", "ACTUAL_YIELD_BU_ACRE"]
    ].head(20)
)

print("\nNumeric yield statistics:")
display(
    yield_numeric_check.describe()
)

ACTUAL_YIELD_BU_ACRE dtype:
object

Sample raw values:


,FIELD_ID,SEASON_YEAR,ACTUAL_YIELD_BU_ACRE
0,FLD001,2021,N/A
1,FLD001,2022,N/A
2,FLD001,2023,N/A
3,FLD001,2024,N/A
4,FLD001,2025,N/A
5,FLD001,2026,N/A
6,FLD002,2021,N/A
7,FLD002,2022,N/A
8,FLD002,2023,N/A
9,FLD002,2024,N/A



Numeric conversion check:
Total valid rows: 237
Numeric values: 171
Non-numeric values: 66

Non-numeric examples:


,FIELD_ID,SEASON_YEAR,ACTUAL_YIELD_BU_ACRE
0,FLD001,2021,N/A
1,FLD001,2022,N/A
2,FLD001,2023,N/A
3,FLD001,2024,N/A
4,FLD001,2025,N/A
5,FLD001,2026,N/A
6,FLD002,2021,N/A
7,FLD002,2022,N/A
8,FLD002,2023,N/A
9,FLD002,2024,N/A



Numeric yield statistics:


count    171.000000
mean     129.605263
std       72.676463
min       47.000000
25%       60.500000
50%       66.000000
75%      204.000000
max      219.000000
Name: ACTUAL_YIELD_BU_ACRE, dtype: float64

In [141]:
# Prepare numeric historical yields for feature engineering.
# Only rows with a genuinely numeric ACTUAL_YIELD are usable
# for previous-season, mean, variability, and trend calculations.

yield_history_numeric = valid_yield_history.copy()

yield_history_numeric["ACTUAL_YIELD_NUMERIC"] = pd.to_numeric(
    yield_history_numeric["ACTUAL_YIELD_BU_ACRE"],
    errors="coerce"
)

print("Rows in validated yield history:", len(yield_history_numeric))

print(
    "Rows with numeric yield:",
    yield_history_numeric["ACTUAL_YIELD_NUMERIC"].notna().sum()
)

print(
    "Rows without numeric yield:",
    yield_history_numeric["ACTUAL_YIELD_NUMERIC"].isna().sum()
)

print("\nNumeric yield range:")
print(
    yield_history_numeric["ACTUAL_YIELD_NUMERIC"].min(),
    "to",
    yield_history_numeric["ACTUAL_YIELD_NUMERIC"].max()
)

print("\nNumeric yield coverage by crop:")
display(
    yield_history_numeric[
        yield_history_numeric["ACTUAL_YIELD_NUMERIC"].notna()
    ]
    .groupby("CROP_TYPE")["ACTUAL_YIELD_NUMERIC"]
    .agg(["count", "min", "max", "mean", "std"])
)

print("\nFields with numeric historical yield:")
print(
    yield_history_numeric.loc[
        yield_history_numeric["ACTUAL_YIELD_NUMERIC"].notna(),
        "FIELD_ID"
    ].nunique()
)

Rows in validated yield history: 237
Rows with numeric yield: 171
Rows without numeric yield: 66

Numeric yield range:
47.0 to 219.0

Numeric yield coverage by crop:


,count,min,max,mean,std
CROP_TYPE,,,,,
Corn,84,176.0,219.0,202.845238,10.907601
Soybean,87,47.0,66.0,58.890805,5.296129



Fields with numeric historical yield:
29


In [142]:
# Leakage-safe historical yield features
#
# For each FIELD_ID + SEASON_YEAR:
# - Previous-season yield uses only the immediately previous available season.
# - Historical mean/std/count use only earlier seasons.
# - Trend is calculated only from earlier numeric seasons.
# - Current/future yields are never used.

yield_hist = yield_history_numeric[
    yield_history_numeric["ACTUAL_YIELD_NUMERIC"].notna()
].copy()

yield_hist = yield_hist[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "CROP_TYPE",
        "ACTUAL_YIELD_NUMERIC"
    ]
].copy()

yield_hist["SEASON_YEAR"] = yield_hist["SEASON_YEAR"].astype(int)

yield_hist = yield_hist.sort_values(
    ["FIELD_ID", "SEASON_YEAR"]
).reset_index(drop=True)


def build_historical_features(group):
    group = group.sort_values("SEASON_YEAR").copy()

    previous_yields = []
    historical_means = []
    historical_stds = []
    historical_counts = []
    historical_trends = []

    for i in range(len(group)):
        prior = group.iloc[:i]

        previous_yields.append(
            prior.iloc[-1]["ACTUAL_YIELD_NUMERIC"]
            if len(prior) > 0
            else np.nan
        )

        historical_means.append(
            prior["ACTUAL_YIELD_NUMERIC"].mean()
            if len(prior) > 0
            else np.nan
        )

        historical_stds.append(
            prior["ACTUAL_YIELD_NUMERIC"].std()
            if len(prior) >= 2
            else np.nan
        )

        historical_counts.append(len(prior))

        # Trend requires at least two prior seasons.
        if len(prior) >= 2:
            x = prior["SEASON_YEAR"].astype(float).to_numpy()
            y = prior["ACTUAL_YIELD_NUMERIC"].astype(float).to_numpy()

            historical_trends.append(
                np.polyfit(x, y, 1)[0]
            )
        else:
            historical_trends.append(np.nan)

    group["PREVIOUS_SEASON_YIELD"] = previous_yields
    group["HISTORICAL_MEAN_YIELD"] = historical_means
    group["HISTORICAL_YIELD_STD"] = historical_stds
    group["HISTORICAL_YIELD_COUNT"] = historical_counts
    group["HISTORICAL_YIELD_TREND"] = historical_trends

    return group


historical_yield_features = (
    yield_hist
    .groupby("FIELD_ID", group_keys=False)
    .apply(build_historical_features)
    .reset_index(drop=True)
)

print("Historical feature rows:", len(historical_yield_features))

print("\nUnique field-season grain:")
print(
    historical_yield_features[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

print("\nMissing historical features:")
display(
    historical_yield_features[
        [
            "PREVIOUS_SEASON_YIELD",
            "HISTORICAL_MEAN_YIELD",
            "HISTORICAL_YIELD_STD",
            "HISTORICAL_YIELD_COUNT",
            "HISTORICAL_YIELD_TREND"
        ]
    ].isna().sum()
)

print("\nSample:")
display(
    historical_yield_features.head(20)
)

Historical feature rows: 171

Unique field-season grain:
171

Missing historical features:


PREVIOUS_SEASON_YIELD     29
HISTORICAL_MEAN_YIELD     29
HISTORICAL_YIELD_STD      58
HISTORICAL_YIELD_COUNT     0
HISTORICAL_YIELD_TREND    58
dtype: int64


Sample:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,ACTUAL_YIELD_NUMERIC,PREVIOUS_SEASON_YIELD,HISTORICAL_MEAN_YIELD,HISTORICAL_YIELD_STD,HISTORICAL_YIELD_COUNT,HISTORICAL_YIELD_TREND
0,FLD003,2021,Soybean,62.0,NaN,NaN,NaN,0,NaN
1,FLD003,2022,Soybean,63.0,62.0,62.00,NaN,1,NaN
2,FLD003,2023,Soybean,64.0,63.0,62.50,0.707107,2,1.0
3,FLD003,2024,Soybean,66.0,64.0,63.00,1.000000,3,1.0
4,FLD003,2025,Soybean,62.5,66.0,63.75,1.707825,4,1.3
5,FLD004,2021,Corn,205.0,NaN,NaN,NaN,0,NaN
6,FLD004,2022,Corn,200.0,205.0,205.00,NaN,1,NaN
7,FLD004,2023,Corn,201.0,200.0,202.50,3.535534,2,-5.0
8,FLD004,2024,Corn,213.0,201.0,202.00,2.645751,3,-2.0
9,FLD004,2025,Corn,210.0,213.0,204.75,5.909033,4,2.5


In [143]:
# Create one historical-feature record per FIELD_ID + SEASON_YEAR
# for joining into the forecast input.

historical_yield_feature_table = (
    historical_yield_features[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            "PREVIOUS_SEASON_YIELD",
            "HISTORICAL_MEAN_YIELD",
            "HISTORICAL_YIELD_STD",
            "HISTORICAL_YIELD_COUNT",
            "HISTORICAL_YIELD_TREND"
        ]
    ]
    .drop_duplicates(
        ["FIELD_ID", "SEASON_YEAR"]
    )
    .copy()
)

print(
    "Historical feature field-seasons:",
    len(historical_yield_feature_table)
)

print(
    "Unique historical feature grain:",
    historical_yield_feature_table[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().shape[0]
)

# Check coverage against the forecast field-seasons.
forecast_field_seasons = forecast_with_risk[
    ["FIELD_ID", "SEASON_YEAR"]
].drop_duplicates()

historical_feature_coverage = forecast_field_seasons.merge(
    historical_yield_feature_table,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    indicator=True
)

print("\nForecast field-seasons:", len(forecast_field_seasons))

print("\nHistorical feature coverage:")
display(
    historical_feature_coverage["_merge"]
    .value_counts()
)

print("\nForecast field-seasons without historical yield features:")
display(
    historical_feature_coverage[
        historical_feature_coverage["_merge"] == "left_only"
    ]
)

print("\nHistorical feature sample:")
display(
    historical_yield_feature_table.head(15)
)

Historical feature field-seasons: 171
Unique historical feature grain: 171

Forecast field-seasons: 19

Historical feature coverage:


_merge
both          19
left_only      0
right_only     0
Name: count, dtype: int64


Forecast field-seasons without historical yield features:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,PREVIOUS_SEASON_YIELD,HISTORICAL_MEAN_YIELD,HISTORICAL_YIELD_STD,HISTORICAL_YIELD_COUNT,HISTORICAL_YIELD_TREND,_merge



Historical feature sample:


,FIELD_ID,SEASON_YEAR,CROP_TYPE,PREVIOUS_SEASON_YIELD,HISTORICAL_MEAN_YIELD,HISTORICAL_YIELD_STD,HISTORICAL_YIELD_COUNT,HISTORICAL_YIELD_TREND
0,FLD003,2021,Soybean,NaN,NaN,NaN,0,NaN
1,FLD003,2022,Soybean,62.0,62.00,NaN,1,NaN
2,FLD003,2023,Soybean,63.0,62.50,0.707107,2,1.0
3,FLD003,2024,Soybean,64.0,63.00,1.000000,3,1.0
4,FLD003,2025,Soybean,66.0,63.75,1.707825,4,1.3
5,FLD004,2021,Corn,NaN,NaN,NaN,0,NaN
6,FLD004,2022,Corn,205.0,205.00,NaN,1,NaN
7,FLD004,2023,Corn,200.0,202.50,3.535534,2,-5.0
8,FLD004,2024,Corn,201.0,202.00,2.645751,3,-2.0
9,FLD004,2025,Corn,213.0,204.75,5.909033,4,2.5


In [144]:
# Merge leakage-safe historical yield features into the forecast input

HISTORICAL_YIELD_FEATURES = [
    "PREVIOUS_SEASON_YIELD",
    "HISTORICAL_MEAN_YIELD",
    "HISTORICAL_YIELD_STD",
    "HISTORICAL_YIELD_COUNT",
    "HISTORICAL_YIELD_TREND"
]

forecast_with_historical_yield = forecast_with_risk.merge(
    historical_yield_feature_table[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "CROP_TYPE",
            *HISTORICAL_YIELD_FEATURES
        ]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    validate="many_to_one"
)

print(
    "Forecast rows after historical-yield merge:",
    len(forecast_with_historical_yield)
)

print(
    "Unique forecast grain:",
    forecast_with_historical_yield[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ].drop_duplicates().shape[0]
)

print("\nMissing historical-yield features:")
display(
    forecast_with_historical_yield[
        HISTORICAL_YIELD_FEATURES
    ].isna().sum()
)

print("\nHistorical yield count distribution:")
display(
    forecast_with_historical_yield[
        "HISTORICAL_YIELD_COUNT"
    ].value_counts().sort_index()
)

print("\nSample:")
display(
    forecast_with_historical_yield[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "CROP_TYPE",
            "PREVIOUS_SEASON_YIELD",
            "HISTORICAL_MEAN_YIELD",
            "HISTORICAL_YIELD_STD",
            "HISTORICAL_YIELD_COUNT",
            "HISTORICAL_YIELD_TREND"
        ]
    ].head(15)
)

Forecast rows after historical-yield merge: 111
Unique forecast grain: 111

Missing historical-yield features:


PREVIOUS_SEASON_YIELD     0
HISTORICAL_MEAN_YIELD     0
HISTORICAL_YIELD_STD      0
HISTORICAL_YIELD_COUNT    0
HISTORICAL_YIELD_TREND    0
dtype: int64


Historical yield count distribution:


HISTORICAL_YIELD_COUNT
2    30
3    29
4    30
5    22
Name: count, dtype: int64


Sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,CROP_TYPE,PREVIOUS_SEASON_YIELD,HISTORICAL_MEAN_YIELD,HISTORICAL_YIELD_STD,HISTORICAL_YIELD_COUNT,HISTORICAL_YIELD_TREND
0,FLD007,2023,1,7,Corn,214.0,203.00,15.556349,2,22.0
1,FLD007,2023,3,21,Corn,214.0,203.00,15.556349,2,22.0
2,FLD007,2023,5,35,Corn,214.0,203.00,15.556349,2,22.0
3,FLD007,2023,7,49,Corn,214.0,203.00,15.556349,2,22.0
4,FLD007,2023,9,63,Corn,214.0,203.00,15.556349,2,22.0
5,FLD007,2023,11,77,Corn,214.0,203.00,15.556349,2,22.0
6,FLD007,2024,1,10,Corn,209.0,205.00,11.532563,3,8.5
7,FLD007,2024,3,24,Corn,209.0,205.00,11.532563,3,8.5
8,FLD007,2024,5,38,Corn,209.0,205.00,11.532563,3,8.5
9,FLD007,2024,7,52,Corn,209.0,205.00,11.532563,3,8.5


In [145]:
# Attach the actual yield target separately from the historical feature variables

yield_target_for_forecast = historical_yield_target[
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "ACTUAL_YIELD_BU_ACRE",
        "YIELD_TARGET_STATUS"
    ]
].copy()

# Keep only one row per FIELD_ID + SEASON_YEAR
# Valid targets are already resolved; unresolved conflicts remain flagged.
yield_target_for_forecast = (
    yield_target_for_forecast
    .drop_duplicates(["FIELD_ID", "SEASON_YEAR"])
)

forecast_with_target = forecast_with_historical_yield.merge(
    yield_target_for_forecast,
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    validate="many_to_one"
)

print("Rows after target merge:", len(forecast_with_target))

print(
    "Unique forecast grain:",
    forecast_with_target[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ].drop_duplicates().shape[0]
)

print("\nTarget status distribution:")
display(
    forecast_with_target["YIELD_TARGET_STATUS"]
    .value_counts(dropna=False)
)

print("\nActual yield availability:")
print(
    "Available:",
    forecast_with_target["ACTUAL_YIELD_BU_ACRE"].notna().sum()
)

print(
    "Missing:",
    forecast_with_target["ACTUAL_YIELD_BU_ACRE"].isna().sum()
)

print("\nSample:")
display(
    forecast_with_target[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "YIELD_TARGET_STATUS"
        ]
    ].head(15)
)


Rows after target merge: 111
Unique forecast grain: 111

Target status distribution:


YIELD_TARGET_STATUS
NaN    111
Name: count, dtype: int64


Actual yield availability:
Available: 0
Missing: 111

Sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,YIELD_TARGET_STATUS
0,FLD007,2023,1,7,Corn,NaN,NaN
1,FLD007,2023,3,21,Corn,NaN,NaN
2,FLD007,2023,5,35,Corn,NaN,NaN
3,FLD007,2023,7,49,Corn,NaN,NaN
4,FLD007,2023,9,63,Corn,NaN,NaN
5,FLD007,2023,11,77,Corn,NaN,NaN
6,FLD007,2024,1,10,Corn,NaN,NaN
7,FLD007,2024,3,24,Corn,NaN,NaN
8,FLD007,2024,5,38,Corn,NaN,NaN
9,FLD007,2024,7,52,Corn,NaN,NaN


In [146]:
print("Forecast key dtypes:")
print(
    forecast_with_historical_yield[
        ["FIELD_ID", "SEASON_YEAR"]
    ].dtypes
)

print("\nYield target key dtypes:")
print(
    historical_yield_target[
        ["FIELD_ID", "SEASON_YEAR"]
    ].dtypes
)

print("\nForecast key sample:")
display(
    forecast_with_historical_yield[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().head(15)
)

print("\nYield target key sample:")
display(
    historical_yield_target[
        ["FIELD_ID", "SEASON_YEAR"]
    ].drop_duplicates().head(15)
)

Forecast key dtypes:
FIELD_ID       object
SEASON_YEAR     int64
dtype: object

Yield target key dtypes:
FIELD_ID       string[python]
SEASON_YEAR    string[python]
dtype: object

Forecast key sample:


,FIELD_ID,SEASON_YEAR
0,FLD007,2023
6,FLD007,2024
12,FLD007,2025
18,FLD007,2026
23,FLD014,2023
29,FLD014,2024
34,FLD014,2025
40,FLD014,2026
45,FLD029,2023
51,FLD029,2024



Yield target key sample:


,FIELD_ID,SEASON_YEAR
0,FLD001,2021
1,FLD001,2022
2,FLD001,2023
3,FLD001,2024
4,FLD001,2025
5,FLD001,2026
6,FLD002,2021
7,FLD002,2022
8,FLD002,2023
9,FLD002,2024


In [147]:
# Normalize merge keys before attaching the actual yield target

forecast_with_historical_yield["FIELD_ID"] = (
    forecast_with_historical_yield["FIELD_ID"]
    .astype("string")
    .str.strip()
)

forecast_with_historical_yield["SEASON_YEAR"] = (
    pd.to_numeric(
        forecast_with_historical_yield["SEASON_YEAR"],
        errors="coerce"
    )
    .astype("Int64")
)

yield_target_for_forecast["FIELD_ID"] = (
    yield_target_for_forecast["FIELD_ID"]
    .astype("string")
    .str.strip()
)

yield_target_for_forecast["SEASON_YEAR"] = (
    pd.to_numeric(
        yield_target_for_forecast["SEASON_YEAR"],
        errors="coerce"
    )
    .astype("Int64")
)

print("Forecast key dtypes:")
print(
    forecast_with_historical_yield[
        ["FIELD_ID", "SEASON_YEAR"]
    ].dtypes
)

print("\nTarget key dtypes:")
print(
    yield_target_for_forecast[
        ["FIELD_ID", "SEASON_YEAR"]
    ].dtypes
)

Forecast key dtypes:
FIELD_ID       string[python]
SEASON_YEAR             Int64
dtype: object

Target key dtypes:
FIELD_ID       string[python]
SEASON_YEAR             Int64
dtype: object


In [148]:
# Rebuild the forecast dataset with the normalized actual-yield target keys

forecast_with_target = forecast_with_historical_yield.merge(
    yield_target_for_forecast[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "ACTUAL_YIELD_BU_ACRE",
            "YIELD_TARGET_STATUS"
        ]
    ],
    on=["FIELD_ID", "SEASON_YEAR"],
    how="left",
    validate="many_to_one"
)

print("Rows after target merge:", len(forecast_with_target))

print(
    "Unique forecast grain:",
    forecast_with_target[
        ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ].drop_duplicates().shape[0]
)

print("\nTarget status distribution:")
display(
    forecast_with_target[
        "YIELD_TARGET_STATUS"
    ].value_counts(dropna=False)
)

print("\nActual yield availability:")
print(
    "Available:",
    forecast_with_target[
        "ACTUAL_YIELD_BU_ACRE"
    ].notna().sum()
)

print(
    "Missing:",
    forecast_with_target[
        "ACTUAL_YIELD_BU_ACRE"
    ].isna().sum()
)

print("\nSample:")
display(
    forecast_with_target[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "SEASON_DAY",
            "CROP_TYPE",
            "ACTUAL_YIELD_BU_ACRE",
            "YIELD_TARGET_STATUS"
        ]
    ].head(15)
)

Rows after target merge: 111
Unique forecast grain: 111

Target status distribution:


YIELD_TARGET_STATUS
VALID    111
Name: count, dtype: int64


Actual yield availability:
Available: 111
Missing: 0

Sample:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY,CROP_TYPE,ACTUAL_YIELD_BU_ACRE,YIELD_TARGET_STATUS
0,FLD007,2023,1,7,Corn,209,VALID
1,FLD007,2023,3,21,Corn,209,VALID
2,FLD007,2023,5,35,Corn,209,VALID
3,FLD007,2023,7,49,Corn,209,VALID
4,FLD007,2023,9,63,Corn,209,VALID
5,FLD007,2023,11,77,Corn,209,VALID
6,FLD007,2024,1,10,Corn,218,VALID
7,FLD007,2024,3,24,Corn,218,VALID
8,FLD007,2024,5,38,Corn,218,VALID
9,FLD007,2024,7,52,Corn,218,VALID


In [149]:
print("Final working dataset shape:")
print(forecast_with_target.shape)

print("\nAll columns:")
for i, col in enumerate(forecast_with_target.columns, start=1):
    print(f"{i:02d}. {col}")

Final working dataset shape:
(111, 38)

All columns:
01. FIELD_ID
02. SEASON_YEAR
03. WEEK_INDEX
04. SEASON_DAY
05. PEST_RISK_SCORE
06. GDD_RISK_SCORE
07. WEATHER_DISEASE_RISK_SCORE
08. NDVI_STRESS_SCORE
09. HISTORICAL_DEVIATION_RISK_SCORE
10. OVERALL_RISK_SCORE
11. OVERALL_RISK_LOWER_BOUND
12. OVERALL_RISK_UPPER_BOUND
13. RISK_LEVEL
14. IS_PROVISIONAL
15. AVAILABLE_WEIGHT_PCT
16. DOMINANT_RISK_DRIVER
17. CONFLICTING_SIGNALS_FLAG
18. CONF_DATA_COMPLETENESS
19. CONF_PEST_FRESHNESS
20. CONF_NDVI_QUALITY
21. CONF_DISEASE_CONFIDENCE
22. CONFIDENCE_SCORE
23. CONFIDENCE_LEVEL
24. AVAILABLE_SIGNAL_COUNT
25. MISSING_SIGNAL_COUNT
26. MISSING_SIGNALS
27. DEGRADED_SIGNALS
28. RISK_SOURCE_WEEK_INDEX
29. RISK_SOURCE_SEASON_DAY
30. RISK_LAG_DAYS
31. CROP_TYPE
32. PREVIOUS_SEASON_YIELD
33. HISTORICAL_MEAN_YIELD
34. HISTORICAL_YIELD_STD
35. HISTORICAL_YIELD_COUNT
36. HISTORICAL_YIELD_TREND
37. ACTUAL_YIELD_BU_ACRE
38. YIELD_TARGET_STATUS


In [150]:
print("Forecast working dataset:", forecast_with_target.shape)

print("\nAvailable dataframe variables:")
for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")

Forecast working dataset: (111, 38)

Available dataframe variables:


RuntimeError: dictionary changed size during iteration

In [151]:
print("Forecast working dataset:", forecast_with_target.shape)

print("\nAvailable dataframe variables:")

dataframes = [
    (name, obj)
    for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
]

for name, obj in dataframes:
    print(f"{name}: {obj.shape}")

Forecast working dataset: (111, 38)

Available dataframe variables:
historical_yield: (276, 7)
duplicate_yield: (238, 3)
duplicate_keys: (30, 3)
duplicate_records: (62, 7)
duplicate_analysis: (30, 6)
conflict_record: (0, 7)
yield_profile: (276, 7)
yield_dedup: (245, 7)
remaining_duplicates: (1, 3)
yield_conflict_check: (238, 5)
yield_conflicts: (1, 5)
missing_field_id: (6, 7)
field_context_sample: (10, 13)
missing_field_yield: (6, 7)
field_context: (101, 4)
matches: (6, 8)
context_for_matching: (101, 4)
duplicate_groups: (244, 3)
historical_yield_target: (245, 10)
fld003_2026: (3, 7)
season_weeks: (111, 3)
field_context_full: (101, 5)
weather_sample: (5, 20)
weather_check: (5, 23)
weather_all: (1412, 4)
context_keys: (101, 2)
fields_sample: (5, 12)
fields_all: (114, 5)
unmatched_weather: (24, 2)
coverage_check: (24, 6)
season_intel_sample: (5, 39)
df: (232, 3)
ndvi_week_structure: (13, 3)
ndvi_fld001: (13, 5)
ndvi_source_fld001: (7, 9)
ndvi_alignment: (1439, 6)
weather_fld001: (13, 3)


In [152]:
print("WEATHER AS-OF COLUMNS:")
print(list(weather_asof_aligned.columns))

print("\nNDVI FINAL COLUMNS:")
print(list(ndvi_final.columns))

print("\nSDE COLUMNS:")
print(list(forecast_with_sde.columns))

WEATHER AS-OF COLUMNS:
['FIELD_ID', 'SEASON_YEAR', 'WEEK_INDEX', 'SEASON_DAY', 'EXPECTED_SNAPSHOT_DATE', 'SNAPSHOT_DATE', 'MEAN_TEMPERATURE', 'TEMPERATURE_7D', 'TEMPERATURE_14D', 'TEMPERATURE_DEVIATION', 'RAINFALL_7D', 'RAINFALL_14D', 'RAINFALL_DEVIATION', 'HUMIDITY_7D', 'HUMIDITY_14D', 'HUMIDITY_DEVIATION', 'LEAFWETNESS_7D', 'LEAFWETNESS_DEVIATION', 'CUMULATIVE_GDD', 'HISTORICAL_CUMULATIVE_GDD', 'GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'WEATHER_DATA_AGE', 'WEATHER_DATA_LAG_DAYS', 'WEATHER_ALIGNMENT_STATUS', 'FUTURE_WEATHER_FLAG']

NDVI FINAL COLUMNS:
['FIELD_ID', 'SEASON_YEAR', 'WEEK_INDEX', 'SEASON_DAY', 'NDVI_VALUE', 'NDVI_LAG_DAYS', 'NDVI_FRESHNESS_STATUS', 'NDVI_VALID_FOR_FORECAST']

SDE COLUMNS:
['FIELD_ID', 'SEASON_YEAR', 'WEEK_INDEX', 'SEASON_DAY', 'DEVIATION_STATE', 'DEVIATION_LEVEL', 'DEVIATION_DIRECTION', 'DOMINANT_DEVIATION_DRIVER', 'WEATHER_DEVIATION', 'GDD_DEVIATION', 'NDVI_DEVIATION', 'PEST_DEVIATION', 'DISEASE_DEVIATION', 'SIGNALS_ABNORMAL_COUNT', 'SIGNALS_CONFIRMING_COUNT

In [153]:
# ---------------------------------------------------------
# Build the controlled feature blocks for Yield Forecasting
# ---------------------------------------------------------

forecast_keys = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "SEASON_DAY"
]

# Weather features available as-of each forecast checkpoint
weather_feature_cols = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "SEASON_DAY",
    "MEAN_TEMPERATURE",
    "TEMPERATURE_7D",
    "TEMPERATURE_14D",
    "TEMPERATURE_DEVIATION",
    "RAINFALL_7D",
    "RAINFALL_14D",
    "RAINFALL_DEVIATION",
    "HUMIDITY_7D",
    "HUMIDITY_14D",
    "HUMIDITY_DEVIATION",
    "LEAFWETNESS_7D",
    "LEAFWETNESS_DEVIATION",
    "CUMULATIVE_GDD",
    "HISTORICAL_CUMULATIVE_GDD",
    "GDD_DEVIATION",
    "GDD_DEVIATION_PCT",
    "WEATHER_DATA_AGE",
    "WEATHER_DATA_LAG_DAYS",
    "WEATHER_ALIGNMENT_STATUS"
]

# NDVI features
ndvi_feature_cols = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "SEASON_DAY",
    "NDVI_VALUE",
    "NDVI_LAG_DAYS",
    "NDVI_FRESHNESS_STATUS",
    "NDVI_VALID_FOR_FORECAST"
]

# SDE features
sde_feature_cols = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "SEASON_DAY",
    "DEVIATION_STATE",
    "DEVIATION_LEVEL",
    "DEVIATION_DIRECTION",
    "DOMINANT_DEVIATION_DRIVER",
    "WEATHER_DEVIATION",
    "GDD_DEVIATION",
    "NDVI_DEVIATION",
    "PEST_DEVIATION",
    "DISEASE_DEVIATION",
    "SIGNALS_ABNORMAL_COUNT",
    "SIGNALS_CONFIRMING_COUNT",
    "MULTI_SIGNAL_CONFIRMED",
    "DEVIATION_CONFIDENCE",
    "MISSING_SIGNAL_COUNT",
    "MODEL_VERSION"
]

weather_features_final = weather_asof_aligned[
    weather_feature_cols
].copy()

ndvi_features_final = ndvi_final[
    ndvi_feature_cols
].copy()

sde_features_final = forecast_with_sde[
    sde_feature_cols
].copy()

print("Weather feature block:", weather_features_final.shape)
print("NDVI feature block:", ndvi_features_final.shape)
print("SDE feature block:", sde_features_final.shape)

print("\nWeather duplicate forecast keys:")
print(
    weather_features_final.duplicated(
        forecast_keys
    ).sum()
)

print("\nNDVI duplicate forecast keys:")
print(
    ndvi_features_final.duplicated(
        forecast_keys
    ).sum()
)

print("\nSDE duplicate forecast keys:")
print(
    sde_features_final.duplicated(
        forecast_keys
    ).sum()
)

Weather feature block: (111, 23)
NDVI feature block: (111, 8)
SDE feature block: (111, 19)

Weather duplicate forecast keys:
0

NDVI duplicate forecast keys:
0

SDE duplicate forecast keys:
0


In [154]:
# ---------------------------------------------------------
# Merge Weather + NDVI + SDE feature blocks
# ---------------------------------------------------------

forecast_environmental = weather_features_final.merge(
    ndvi_features_final,
    on=forecast_keys,
    how="left",
    validate="one_to_one"
)

forecast_environmental = forecast_environmental.merge(
    sde_features_final,
    on=forecast_keys,
    how="left",
    validate="one_to_one"
)

print("Combined environmental/SDE dataset:")
print(forecast_environmental.shape)

print(
    "\nUnique forecast grain:",
    forecast_environmental[
        forecast_keys
    ].drop_duplicates().shape[0]
)

print("\nMissing values by feature group:")

print("\nWeather:")
display(
    forecast_environmental[
        weather_feature_cols[4:]
    ].isna().sum()
)

print("\nNDVI:")
display(
    forecast_environmental[
        ndvi_feature_cols[4:]
    ].isna().sum()
)

print("\nSDE:")
display(
    forecast_environmental[
        sde_feature_cols[4:]
    ].isna().sum()
)

Combined environmental/SDE dataset:
(111, 42)

Unique forecast grain: 111

Missing values by feature group:

Weather:


KeyError: "['GDD_DEVIATION'] not in index"

In [155]:
print("Combined dataset shape:", forecast_environmental.shape)

print("\nColumns containing GDD_DEVIATION:")
display(
    [
        col
        for col in forecast_environmental.columns
        if "GDD_DEVIATION" in col
    ]
)

print("\nColumns containing WEATHER_DEVIATION:")
display(
    [
        col
        for col in forecast_environmental.columns
        if "WEATHER_DEVIATION" in col
    ]
)

print("\nColumns containing NDVI_DEVIATION:")
display(
    [
        col
        for col in forecast_environmental.columns
        if "NDVI_DEVIATION" in col
    ]
)

Combined dataset shape: (111, 42)

Columns containing GDD_DEVIATION:


['GDD_DEVIATION_x', 'GDD_DEVIATION_PCT', 'GDD_DEVIATION_y']


Columns containing WEATHER_DEVIATION:


['WEATHER_DEVIATION']


Columns containing NDVI_DEVIATION:


['NDVI_DEVIATION']

In [156]:
# Make the two GDD deviation signals explicit
# so their source and meaning are clear in the final dataset.

forecast_environmental = forecast_environmental.rename(
    columns={
        "GDD_DEVIATION_x": "WEATHER_GDD_DEVIATION",
        "GDD_DEVIATION_y": "SDE_GDD_DEVIATION"
    }
)

print("GDD deviation columns after rename:")
display(
    [
        col
        for col in forecast_environmental.columns
        if "GDD_DEVIATION" in col
    ]
)

print("\nDataset shape:", forecast_environmental.shape)

print(
    "Unique forecast grain:",
    forecast_environmental[
        forecast_keys
    ].drop_duplicates().shape[0]
)

GDD deviation columns after rename:


['WEATHER_GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'SDE_GDD_DEVIATION']


Dataset shape: (111, 42)
Unique forecast grain: 111


In [157]:
# ---------------------------------------------------------
# Final feature assembly
# Weather + NDVI + SDE + Risk + Historical Yield + Target
# ---------------------------------------------------------

yield_forecast_input = forecast_environmental.merge(
    forecast_with_target,
    on=forecast_keys,
    how="left",
    validate="one_to_one",
    suffixes=("", "_duplicate")
)

print("YIELD_FORECAST_INPUT shape:")
print(yield_forecast_input.shape)

print(
    "\nUnique forecast grain:",
    yield_forecast_input[
        forecast_keys
    ].drop_duplicates().shape[0]
)

print("\nDuplicate columns created by merge:")
display(
    [
        col
        for col in yield_forecast_input.columns
        if col.endswith("_duplicate")
    ]
)

YIELD_FORECAST_INPUT shape:
(111, 76)

Unique forecast grain: 111

Duplicate columns created by merge:


['MISSING_SIGNAL_COUNT_duplicate']

In [158]:
print("Original MISSING_SIGNAL_COUNT:")
display(
    yield_forecast_input[
        ["MISSING_SIGNAL_COUNT", "MISSING_SIGNAL_COUNT_duplicate"]
    ].head(15)
)

print("\nDifference count:")
print(
    (
        yield_forecast_input["MISSING_SIGNAL_COUNT"]
        != yield_forecast_input["MISSING_SIGNAL_COUNT_duplicate"]
    ).sum()
)

print("\nMissing values:")
print(
    yield_forecast_input[
        ["MISSING_SIGNAL_COUNT", "MISSING_SIGNAL_COUNT_duplicate"]
    ].isna().sum()
)

Original MISSING_SIGNAL_COUNT:


,MISSING_SIGNAL_COUNT,MISSING_SIGNAL_COUNT_duplicate
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0



Difference count:
17

Missing values:
MISSING_SIGNAL_COUNT              0
MISSING_SIGNAL_COUNT_duplicate    0
dtype: int64


In [159]:
yield_forecast_input = yield_forecast_input.rename(
    columns={
        "MISSING_SIGNAL_COUNT": "RISK_MISSING_SIGNAL_COUNT",
        "MISSING_SIGNAL_COUNT_duplicate": "SDE_MISSING_SIGNAL_COUNT"
    }
)

print("Renamed columns:")
print(
    [
        col for col in yield_forecast_input.columns
        if "MISSING_SIGNAL_COUNT" in col
    ]
)

print("\nShape:")
print(yield_forecast_input.shape)

print("\nUnique forecast grain:")
print(
    yield_forecast_input[forecast_keys]
    .drop_duplicates()
    .shape[0]
)

Renamed columns:
['RISK_MISSING_SIGNAL_COUNT', 'SDE_MISSING_SIGNAL_COUNT']

Shape:
(111, 76)

Unique forecast grain:
111


In [160]:
print("Total columns:", len(yield_forecast_input.columns))

for i, col in enumerate(yield_forecast_input.columns, start=1):
    print(f"{i:02d}. {col}")

Total columns: 76
01. FIELD_ID
02. SEASON_YEAR
03. WEEK_INDEX
04. SEASON_DAY
05. MEAN_TEMPERATURE
06. TEMPERATURE_7D
07. TEMPERATURE_14D
08. TEMPERATURE_DEVIATION
09. RAINFALL_7D
10. RAINFALL_14D
11. RAINFALL_DEVIATION
12. HUMIDITY_7D
13. HUMIDITY_14D
14. HUMIDITY_DEVIATION
15. LEAFWETNESS_7D
16. LEAFWETNESS_DEVIATION
17. CUMULATIVE_GDD
18. HISTORICAL_CUMULATIVE_GDD
19. WEATHER_GDD_DEVIATION
20. GDD_DEVIATION_PCT
21. WEATHER_DATA_AGE
22. WEATHER_DATA_LAG_DAYS
23. WEATHER_ALIGNMENT_STATUS
24. NDVI_VALUE
25. NDVI_LAG_DAYS
26. NDVI_FRESHNESS_STATUS
27. NDVI_VALID_FOR_FORECAST
28. DEVIATION_STATE
29. DEVIATION_LEVEL
30. DEVIATION_DIRECTION
31. DOMINANT_DEVIATION_DRIVER
32. WEATHER_DEVIATION
33. SDE_GDD_DEVIATION
34. NDVI_DEVIATION
35. PEST_DEVIATION
36. DISEASE_DEVIATION
37. SIGNALS_ABNORMAL_COUNT
38. SIGNALS_CONFIRMING_COUNT
39. MULTI_SIGNAL_CONFIRMED
40. DEVIATION_CONFIDENCE
41. RISK_MISSING_SIGNAL_COUNT
42. MODEL_VERSION
43. PEST_RISK_SCORE
44. GDD_RISK_SCORE
45. WEATHER_DISEASE_RISK_SC

In [161]:
print("\nTarget columns:")
display(
    yield_forecast_input[
        [
            "ACTUAL_YIELD_BU_ACRE",
            "YIELD_TARGET_STATUS"
        ]
    ].head()
)

print("\nForecast grain:")
display(
    yield_forecast_input[
        forecast_keys
    ].head()
)


Target columns:


,ACTUAL_YIELD_BU_ACRE,YIELD_TARGET_STATUS
0,209,VALID
1,209,VALID
2,209,VALID
3,209,VALID
4,209,VALID



Forecast grain:


,FIELD_ID,SEASON_YEAR,WEEK_INDEX,SEASON_DAY
0,FLD007,2023,1,7
1,FLD007,2023,3,21
2,FLD007,2023,5,35
3,FLD007,2023,7,49
4,FLD007,2023,9,63


In [162]:
for i, col in enumerate(yield_forecast_input.columns, start=1):
    print(f"{i:02d}. {col}")

01. FIELD_ID
02. SEASON_YEAR
03. WEEK_INDEX
04. SEASON_DAY
05. MEAN_TEMPERATURE
06. TEMPERATURE_7D
07. TEMPERATURE_14D
08. TEMPERATURE_DEVIATION
09. RAINFALL_7D
10. RAINFALL_14D
11. RAINFALL_DEVIATION
12. HUMIDITY_7D
13. HUMIDITY_14D
14. HUMIDITY_DEVIATION
15. LEAFWETNESS_7D
16. LEAFWETNESS_DEVIATION
17. CUMULATIVE_GDD
18. HISTORICAL_CUMULATIVE_GDD
19. WEATHER_GDD_DEVIATION
20. GDD_DEVIATION_PCT
21. WEATHER_DATA_AGE
22. WEATHER_DATA_LAG_DAYS
23. WEATHER_ALIGNMENT_STATUS
24. NDVI_VALUE
25. NDVI_LAG_DAYS
26. NDVI_FRESHNESS_STATUS
27. NDVI_VALID_FOR_FORECAST
28. DEVIATION_STATE
29. DEVIATION_LEVEL
30. DEVIATION_DIRECTION
31. DOMINANT_DEVIATION_DRIVER
32. WEATHER_DEVIATION
33. SDE_GDD_DEVIATION
34. NDVI_DEVIATION
35. PEST_DEVIATION
36. DISEASE_DEVIATION
37. SIGNALS_ABNORMAL_COUNT
38. SIGNALS_CONFIRMING_COUNT
39. MULTI_SIGNAL_CONFIRMED
40. DEVIATION_CONFIDENCE
41. RISK_MISSING_SIGNAL_COUNT
42. MODEL_VERSION
43. PEST_RISK_SCORE
44. GDD_RISK_SCORE
45. WEATHER_DISEASE_RISK_SCORE
46. NDVI_STRES

In [163]:
forecast_keys = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "SEASON_DAY"
]

target_columns = [
    "ACTUAL_YIELD_BU_ACRE",
    "YIELD_TARGET_STATUS"
]

audit_columns = [
    "WEATHER_DATA_AGE",
    "WEATHER_DATA_LAG_DAYS",
    "WEATHER_ALIGNMENT_STATUS",
    "NDVI_LAG_DAYS",
    "NDVI_FRESHNESS_STATUS",
    "NDVI_VALID_FOR_FORECAST",
    "RISK_SOURCE_WEEK_INDEX",
    "RISK_SOURCE_SEASON_DAY",
    "RISK_LAG_DAYS",
    "YIELD_TARGET_STATUS"
]

print("Forecast keys:", forecast_keys)
print("\nTarget columns:", target_columns)
print("\nAudit / provenance columns:", audit_columns)

print("\nTotal columns:", len(yield_forecast_input.columns))
print("Unclassified columns:")

classified = set(forecast_keys + target_columns + audit_columns)

print(
    [
        col
        for col in yield_forecast_input.columns
        if col not in classified
    ]
)

Forecast keys: ['FIELD_ID', 'SEASON_YEAR', 'WEEK_INDEX', 'SEASON_DAY']

Target columns: ['ACTUAL_YIELD_BU_ACRE', 'YIELD_TARGET_STATUS']

Audit / provenance columns: ['WEATHER_DATA_AGE', 'WEATHER_DATA_LAG_DAYS', 'WEATHER_ALIGNMENT_STATUS', 'NDVI_LAG_DAYS', 'NDVI_FRESHNESS_STATUS', 'NDVI_VALID_FOR_FORECAST', 'RISK_SOURCE_WEEK_INDEX', 'RISK_SOURCE_SEASON_DAY', 'RISK_LAG_DAYS', 'YIELD_TARGET_STATUS']

Total columns: 76
Unclassified columns:
['MEAN_TEMPERATURE', 'TEMPERATURE_7D', 'TEMPERATURE_14D', 'TEMPERATURE_DEVIATION', 'RAINFALL_7D', 'RAINFALL_14D', 'RAINFALL_DEVIATION', 'HUMIDITY_7D', 'HUMIDITY_14D', 'HUMIDITY_DEVIATION', 'LEAFWETNESS_7D', 'LEAFWETNESS_DEVIATION', 'CUMULATIVE_GDD', 'HISTORICAL_CUMULATIVE_GDD', 'WEATHER_GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'NDVI_VALUE', 'DEVIATION_STATE', 'DEVIATION_LEVEL', 'DEVIATION_DIRECTION', 'DOMINANT_DEVIATION_DRIVER', 'WEATHER_DEVIATION', 'SDE_GDD_DEVIATION', 'NDVI_DEVIATION', 'PEST_DEVIATION', 'DISEASE_DEVIATION', 'SIGNALS_ABNORMAL_COUNT', 'SIGN

In [164]:
print("=== TARGET LEAKAGE CHECK ===")

# 1. Target presence
print("\nTarget columns:")
print(
    yield_forecast_input[
        ["ACTUAL_YIELD_BU_ACRE", "YIELD_TARGET_STATUS"]
    ].isna().sum()
)

# 2. Target status
print("\nTarget status:")
print(
    yield_forecast_input["YIELD_TARGET_STATUS"]
    .value_counts(dropna=False)
)

# 3. Check that historical yield features are not identical to current target
historical_feature_cols = [
    "PREVIOUS_SEASON_YIELD",
    "HISTORICAL_MEAN_YIELD",
    "HISTORICAL_YIELD_STD",
    "HISTORICAL_YIELD_COUNT",
    "HISTORICAL_YIELD_TREND"
]

print("\nHistorical feature vs current target:")
for col in historical_feature_cols:
    same = (
        yield_forecast_input[col].notna()
        & yield_forecast_input["ACTUAL_YIELD_BU_ACRE"].notna()
        & (
            yield_forecast_input[col]
            == yield_forecast_input["ACTUAL_YIELD_BU_ACRE"]
        )
    ).sum()

    print(f"{col}: {same} rows equal to current target")

# 4. Check risk as-of alignment
print("\nRisk source timing:")
print(
    yield_forecast_input[
        [
            "SEASON_DAY",
            "RISK_SOURCE_SEASON_DAY",
            "RISK_LAG_DAYS"
        ]
    ].describe()
)

future_risk_rows = (
    yield_forecast_input["RISK_SOURCE_SEASON_DAY"]
    > yield_forecast_input["SEASON_DAY"]
).sum()

print("\nFuture risk rows:", future_risk_rows)

# 5. Check weather alignment status
print("\nWeather alignment status:")
print(
    yield_forecast_input["WEATHER_ALIGNMENT_STATUS"]
    .value_counts(dropna=False)
)

# 6. Check NDVI validity status
print("\nNDVI freshness status:")
print(
    yield_forecast_input["NDVI_FRESHNESS_STATUS"]
    .value_counts(dropna=False)
)

=== TARGET LEAKAGE CHECK ===

Target columns:
ACTUAL_YIELD_BU_ACRE    0
YIELD_TARGET_STATUS     0
dtype: int64

Target status:
YIELD_TARGET_STATUS
VALID    111
Name: count, dtype: int64

Historical feature vs current target:
PREVIOUS_SEASON_YIELD: 0 rows equal to current target
HISTORICAL_MEAN_YIELD: 0 rows equal to current target
HISTORICAL_YIELD_STD: 0 rows equal to current target
HISTORICAL_YIELD_COUNT: 0 rows equal to current target
HISTORICAL_YIELD_TREND: 0 rows equal to current target

Risk source timing:
       SEASON_DAY  RISK_SOURCE_SEASON_DAY  RISK_LAG_DAYS
count  111.000000              111.000000     111.000000
mean    44.558559               41.720721       2.837838
std     24.097410               24.282119       2.025028
min      7.000000                3.000000       0.000000
25%     24.000000               24.000000       1.000000
50%     49.000000               45.000000       3.000000
75%     65.500000               62.500000       4.500000
max     83.000000          

In [165]:
print("=== FINAL DATA QUALITY AUDIT ===")

print("\nShape:")
print(yield_forecast_input.shape)

print("\nDuplicate forecast grain rows:")
print(
    yield_forecast_input.duplicated(
        subset=forecast_keys
    ).sum()
)

print("\nMissing values by column:")
missing_summary = (
    yield_forecast_input.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ].to_frame("MISSING_COUNT")
)

print("\nData types:")
display(
    yield_forecast_input.dtypes.to_frame("DTYPE")
)

=== FINAL DATA QUALITY AUDIT ===

Shape:
(111, 76)

Duplicate forecast grain rows:
0

Missing values by column:


,MISSING_COUNT
MISSING_SIGNALS,94
DEGRADED_SIGNALS,21
GDD_RISK_SCORE,13
TEMPERATURE_14D,5
TEMPERATURE_7D,5
MEAN_TEMPERATURE,5
RAINFALL_7D,5
TEMPERATURE_DEVIATION,5
HUMIDITY_14D,5
HUMIDITY_DEVIATION,5



Data types:


,DTYPE
FIELD_ID,object
SEASON_YEAR,int64
WEEK_INDEX,int64
SEASON_DAY,int64
MEAN_TEMPERATURE,float64
TEMPERATURE_7D,float64
TEMPERATURE_14D,float64
TEMPERATURE_DEVIATION,float64
RAINFALL_7D,float64
RAINFALL_14D,float64


In [166]:
yield_forecast_input["ACTUAL_YIELD_NUMERIC"] = pd.to_numeric(
    yield_forecast_input["ACTUAL_YIELD_BU_ACRE"],
    errors="coerce"
)

print("Target dtype:")
print(yield_forecast_input["ACTUAL_YIELD_NUMERIC"].dtype)

print("\nMissing numeric targets:")
print(yield_forecast_input["ACTUAL_YIELD_NUMERIC"].isna().sum())

print("\nTarget summary:")
display(
    yield_forecast_input["ACTUAL_YIELD_NUMERIC"].describe()
)

Target dtype:
float64

Missing numeric targets:
0

Target summary:


count    111.000000
mean     151.315315
std       69.034969
min       58.000000
25%       63.500000
50%      182.000000
75%      209.000000
max      218.000000
Name: ACTUAL_YIELD_NUMERIC, dtype: float64

In [167]:
target_comparison = pd.DataFrame({
    "ORIGINAL_TARGET": pd.to_numeric(
        yield_forecast_input["ACTUAL_YIELD_BU_ACRE"],
        errors="coerce"
    ),
    "NUMERIC_TARGET": yield_forecast_input["ACTUAL_YIELD_NUMERIC"]
})

target_comparison["MATCH"] = (
    target_comparison["ORIGINAL_TARGET"]
    == target_comparison["NUMERIC_TARGET"]
)

print("Target mismatches:")
print((~target_comparison["MATCH"]).sum())

print("\nTarget missing values:")
print(target_comparison.isna().sum())

print("\nTarget match rate:")
print(target_comparison["MATCH"].mean() * 100, "%")

Target mismatches:
0

Target missing values:
ORIGINAL_TARGET    0
NUMERIC_TARGET     0
MATCH              0
dtype: int64

Target match rate:
100.0 %


In [168]:
target_columns = [
    "ACTUAL_YIELD_BU_ACRE",
    "ACTUAL_YIELD_NUMERIC",
    "YIELD_TARGET_STATUS"
]

audit_columns = [
    "WEATHER_DATA_AGE",
    "WEATHER_DATA_LAG_DAYS",
    "WEATHER_ALIGNMENT_STATUS",
    "NDVI_LAG_DAYS",
    "NDVI_FRESHNESS_STATUS",
    "NDVI_VALID_FOR_FORECAST",
    "RISK_SOURCE_WEEK_INDEX",
    "RISK_SOURCE_SEASON_DAY",
    "RISK_LAG_DAYS"
]

key_columns = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "SEASON_DAY"
]

feature_columns = [
    col
    for col in yield_forecast_input.columns
    if col not in (
        target_columns
        + audit_columns
        + key_columns
    )
]

print("Feature columns:", len(feature_columns))

for i, col in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {col}")

print("\nTarget columns:")
print(target_columns)

print("\nAudit columns:")
print(audit_columns)

print("\nKey columns:")
print(key_columns)

Feature columns: 61
01. MEAN_TEMPERATURE
02. TEMPERATURE_7D
03. TEMPERATURE_14D
04. TEMPERATURE_DEVIATION
05. RAINFALL_7D
06. RAINFALL_14D
07. RAINFALL_DEVIATION
08. HUMIDITY_7D
09. HUMIDITY_14D
10. HUMIDITY_DEVIATION
11. LEAFWETNESS_7D
12. LEAFWETNESS_DEVIATION
13. CUMULATIVE_GDD
14. HISTORICAL_CUMULATIVE_GDD
15. WEATHER_GDD_DEVIATION
16. GDD_DEVIATION_PCT
17. NDVI_VALUE
18. DEVIATION_STATE
19. DEVIATION_LEVEL
20. DEVIATION_DIRECTION
21. DOMINANT_DEVIATION_DRIVER
22. WEATHER_DEVIATION
23. SDE_GDD_DEVIATION
24. NDVI_DEVIATION
25. PEST_DEVIATION
26. DISEASE_DEVIATION
27. SIGNALS_ABNORMAL_COUNT
28. SIGNALS_CONFIRMING_COUNT
29. MULTI_SIGNAL_CONFIRMED
30. DEVIATION_CONFIDENCE
31. RISK_MISSING_SIGNAL_COUNT
32. MODEL_VERSION
33. PEST_RISK_SCORE
34. GDD_RISK_SCORE
35. WEATHER_DISEASE_RISK_SCORE
36. NDVI_STRESS_SCORE
37. HISTORICAL_DEVIATION_RISK_SCORE
38. OVERALL_RISK_SCORE
39. OVERALL_RISK_LOWER_BOUND
40. OVERALL_RISK_UPPER_BOUND
41. RISK_LEVEL
42. IS_PROVISIONAL
43. AVAILABLE_WEIGHT_PCT
44.

In [169]:
feature_review = pd.DataFrame({
    "FEATURE": feature_columns
})

feature_review["MISSING_COUNT"] = (
    yield_forecast_input[feature_columns]
    .isna()
    .sum()
    .values
)

feature_review["DTYPE"] = (
    yield_forecast_input[feature_columns]
    .dtypes
    .astype(str)
    .values
)

feature_review["UNIQUE_VALUES"] = [
    yield_forecast_input[col].nunique(dropna=True)
    for col in feature_columns
]

display(feature_review)

,FEATURE,MISSING_COUNT,DTYPE,UNIQUE_VALUES
0,MEAN_TEMPERATURE,5,float64,81
1,TEMPERATURE_7D,5,float64,81
2,TEMPERATURE_14D,5,float64,96
3,TEMPERATURE_DEVIATION,5,float64,90
4,RAINFALL_7D,5,float64,80
5,RAINFALL_14D,5,float64,92
6,RAINFALL_DEVIATION,5,float64,99
7,HUMIDITY_7D,5,float64,90
8,HUMIDITY_14D,5,float64,99
9,HUMIDITY_DEVIATION,5,float64,104


In [170]:
constant_features = [
    col
    for col in feature_columns
    if yield_forecast_input[col].nunique(dropna=True) <= 1
]

print("Constant candidate features:")
print(constant_features)

print("\nCount:", len(constant_features))

Constant candidate features:
['RISK_MISSING_SIGNAL_COUNT', 'MODEL_VERSION', 'DEGRADED_SIGNALS']

Count: 3


In [171]:
categorical_features = [
    col
    for col in feature_columns
    if (
        yield_forecast_input[col].dtype == "object"
        or str(yield_forecast_input[col].dtype) == "string"
        or str(yield_forecast_input[col].dtype) == "bool"
    )
]

print("Categorical / boolean candidate features:")
for col in categorical_features:
    print(f"- {col}")

print("\nCount:", len(categorical_features))

Categorical / boolean candidate features:
- DEVIATION_STATE
- DEVIATION_LEVEL
- DEVIATION_DIRECTION
- DOMINANT_DEVIATION_DRIVER
- MULTI_SIGNAL_CONFIRMED
- DEVIATION_CONFIDENCE
- MODEL_VERSION
- RISK_LEVEL
- IS_PROVISIONAL
- DOMINANT_RISK_DRIVER
- CONFLICTING_SIGNALS_FLAG
- CONFIDENCE_LEVEL
- MISSING_SIGNALS
- DEGRADED_SIGNALS
- CROP_TYPE

Count: 15


In [172]:
print("RISK_MISSING_SIGNAL_COUNT values:")
print(
    yield_forecast_input["RISK_MISSING_SIGNAL_COUNT"]
    .value_counts(dropna=False)
)

print("\nMODEL_VERSION values:")
print(
    yield_forecast_input["MODEL_VERSION"]
    .value_counts(dropna=False)
)

print("\nDEGRADED_SIGNALS values:")
print(
    yield_forecast_input["DEGRADED_SIGNALS"]
    .value_counts(dropna=False)
)

RISK_MISSING_SIGNAL_COUNT values:
RISK_MISSING_SIGNAL_COUNT
0    111
Name: count, dtype: int64

MODEL_VERSION values:
MODEL_VERSION
SDE_V1.0    111
Name: count, dtype: int64

DEGRADED_SIGNALS values:
DEGRADED_SIGNALS
WEATHER_DISEASE(PARTIAL_NO_DISEASE)    90
None                                   21
Name: count, dtype: int64


In [173]:
constant_columns = [
    "RISK_MISSING_SIGNAL_COUNT",
    "MODEL_VERSION"
]

provenance_columns = [
    "WEATHER_DATA_AGE",
    "WEATHER_DATA_LAG_DAYS",
    "WEATHER_ALIGNMENT_STATUS",
    "NDVI_LAG_DAYS",
    "NDVI_FRESHNESS_STATUS",
    "NDVI_VALID_FOR_FORECAST",
    "RISK_SOURCE_WEEK_INDEX",
    "RISK_SOURCE_SEASON_DAY",
    "RISK_LAG_DAYS",
    "MISSING_SIGNALS",
    "DEGRADED_SIGNALS"
]

model_candidate_features = [
    col
    for col in feature_columns
    if col not in constant_columns + [
        "MISSING_SIGNALS",
        "DEGRADED_SIGNALS"
    ]
]

print("Model candidate features:", len(model_candidate_features))

print("\nConstant / metadata:")
for col in constant_columns:
    print("-", col)

print("\nProvenance / quality fields:")
for col in provenance_columns:
    print("-", col)

print("\nModel candidate features:")
for i, col in enumerate(model_candidate_features, start=1):
    print(f"{i:02d}. {col}")

Model candidate features: 57

Constant / metadata:
- RISK_MISSING_SIGNAL_COUNT
- MODEL_VERSION

Provenance / quality fields:
- WEATHER_DATA_AGE
- WEATHER_DATA_LAG_DAYS
- WEATHER_ALIGNMENT_STATUS
- NDVI_LAG_DAYS
- NDVI_FRESHNESS_STATUS
- NDVI_VALID_FOR_FORECAST
- RISK_SOURCE_WEEK_INDEX
- RISK_SOURCE_SEASON_DAY
- RISK_LAG_DAYS
- MISSING_SIGNALS
- DEGRADED_SIGNALS

Model candidate features:
01. MEAN_TEMPERATURE
02. TEMPERATURE_7D
03. TEMPERATURE_14D
04. TEMPERATURE_DEVIATION
05. RAINFALL_7D
06. RAINFALL_14D
07. RAINFALL_DEVIATION
08. HUMIDITY_7D
09. HUMIDITY_14D
10. HUMIDITY_DEVIATION
11. LEAFWETNESS_7D
12. LEAFWETNESS_DEVIATION
13. CUMULATIVE_GDD
14. HISTORICAL_CUMULATIVE_GDD
15. WEATHER_GDD_DEVIATION
16. GDD_DEVIATION_PCT
17. NDVI_VALUE
18. DEVIATION_STATE
19. DEVIATION_LEVEL
20. DEVIATION_DIRECTION
21. DOMINANT_DEVIATION_DRIVER
22. WEATHER_DEVIATION
23. SDE_GDD_DEVIATION
24. NDVI_DEVIATION
25. PEST_DEVIATION
26. DISEASE_DEVIATION
27. SIGNALS_ABNORMAL_COUNT
28. SIGNALS_CONFIRMING_COUNT


In [174]:
print("=== FINAL TEMPORAL LEAKAGE CHECK ===")

# Risk: source observation must not be after the forecast checkpoint
future_risk = (
    yield_forecast_input["RISK_SOURCE_SEASON_DAY"]
    > yield_forecast_input["SEASON_DAY"]
)

# Risk lag should therefore never be negative
negative_risk_lag = (
    yield_forecast_input["RISK_LAG_DAYS"] < 0
)

# Weather: explicit alignment status
invalid_weather_status = ~yield_forecast_input[
    "WEATHER_ALIGNMENT_STATUS"
].isin([
    "AS_OF_ALIGNED",
    "NO_PRIOR_WEATHER"
])

# NDVI: no future information can be represented by a negative lag
negative_ndvi_lag = (
    yield_forecast_input["NDVI_LAG_DAYS"].notna()
    & (yield_forecast_input["NDVI_LAG_DAYS"] < 0)
)

print("Future Risk rows:", future_risk.sum())
print("Negative Risk lag rows:", negative_risk_lag.sum())
print("Invalid Weather alignment statuses:", invalid_weather_status.sum())
print("Negative NDVI lag rows:", negative_ndvi_lag.sum())

print("\nTemporal leakage violations:")
print(
    int(
        future_risk.sum()
        + negative_risk_lag.sum()
        + invalid_weather_status.sum()
        + negative_ndvi_lag.sum()
    )
)

=== FINAL TEMPORAL LEAKAGE CHECK ===
Future Risk rows: 0
Negative Risk lag rows: 0
Invalid Weather alignment statuses: 0
Negative NDVI lag rows: 0

Temporal leakage violations:
0


In [175]:
feature_dictionary = pd.DataFrame({
    "COLUMN_NAME": yield_forecast_input.columns
})

feature_dictionary["DATA_TYPE"] = [
    str(yield_forecast_input[col].dtype)
    for col in yield_forecast_input.columns
]

feature_dictionary["MISSING_COUNT"] = [
    yield_forecast_input[col].isna().sum()
    for col in yield_forecast_input.columns
]

feature_dictionary["UNIQUE_VALUES"] = [
    yield_forecast_input[col].nunique(dropna=True)
    for col in yield_forecast_input.columns
]

def classify_column(col):
    if col in key_columns:
        return "FORECAST_KEY"
    elif col in target_columns:
        return "TARGET"
    elif col in constant_columns:
        return "CONSTANT_METADATA"
    elif col in provenance_columns:
        return "PROVENANCE_QUALITY"
    elif col in model_candidate_features:
        return "MODEL_CANDIDATE"
    else:
        return "REVIEW"

feature_dictionary["COLUMN_ROLE"] = [
    classify_column(col)
    for col in yield_forecast_input.columns
]

display(feature_dictionary)

,COLUMN_NAME,DATA_TYPE,MISSING_COUNT,UNIQUE_VALUES,COLUMN_ROLE
0,FIELD_ID,object,0,5,FORECAST_KEY
1,SEASON_YEAR,int64,0,4,FORECAST_KEY
2,WEEK_INDEX,int64,0,6,FORECAST_KEY
3,SEASON_DAY,int64,0,42,FORECAST_KEY
4,MEAN_TEMPERATURE,float64,5,81,MODEL_CANDIDATE
5,TEMPERATURE_7D,float64,5,81,MODEL_CANDIDATE
6,TEMPERATURE_14D,float64,5,96,MODEL_CANDIDATE
7,TEMPERATURE_DEVIATION,float64,5,90,MODEL_CANDIDATE
8,RAINFALL_7D,float64,5,80,MODEL_CANDIDATE
9,RAINFALL_14D,float64,5,92,MODEL_CANDIDATE


In [176]:
print("=== FEATURE DICTIONARY SUMMARY ===")

print(
    feature_dictionary["COLUMN_ROLE"]
    .value_counts()
)

print("\nTotal columns:", len(feature_dictionary))

=== FEATURE DICTIONARY SUMMARY ===
COLUMN_ROLE
MODEL_CANDIDATE       57
PROVENANCE_QUALITY    11
FORECAST_KEY           4
TARGET                 3
CONSTANT_METADATA      2
Name: count, dtype: int64

Total columns: 77


In [177]:

feature_dictionary["SOURCE"] = ""
feature_dictionary["DESCRIPTION"] = ""
feature_dictionary["FORECAST_AVAILABILITY"] = ""
feature_dictionary["MISSING_VALUE_HANDLING"] = ""
feature_dictionary["LEAKAGE_RULE"] = ""

In [178]:
key_documentation = {
    "FIELD_ID": {
        "SOURCE": "FIELD_SEASON_CONTEXT",
        "DESCRIPTION": "Unique field identifier for the forecast record.",
        "FORECAST_AVAILABILITY": "Available at forecast time.",
        "MISSING_VALUE_HANDLING": "Required; missing values are not permitted.",
        "LEAKAGE_RULE": "Identifier only; contains no future target information."
    },
    "SEASON_YEAR": {
        "SOURCE": "FIELD_SEASON_CONTEXT",
        "DESCRIPTION": "Crop season year associated with the field.",
        "FORECAST_AVAILABILITY": "Available at forecast time.",
        "MISSING_VALUE_HANDLING": "Required; missing values are not permitted.",
        "LEAKAGE_RULE": "Season identifier only."
    },
    "WEEK_INDEX": {
        "SOURCE": "SEASON_INTELLIGENCE_INPUT",
        "DESCRIPTION": "Canonical seasonal week index used for forecast checkpoints.",
        "FORECAST_AVAILABILITY": "Available at forecast time.",
        "MISSING_VALUE_HANDLING": "Required; missing values are not permitted.",
        "LEAKAGE_RULE": "Derived from the established seasonal timeline; does not use future observations."
    },
    "SEASON_DAY": {
        "SOURCE": "SEASON_INTELLIGENCE_INPUT",
        "DESCRIPTION": "Canonical seasonal day corresponding to the forecast checkpoint.",
        "FORECAST_AVAILABILITY": "Available at forecast time.",
        "MISSING_VALUE_HANDLING": "Required; missing values are not permitted.",
        "LEAKAGE_RULE": "Derived from the seasonal timeline only."
    }
}

for col, docs in key_documentation.items():
    for field, value in docs.items():
        feature_dictionary.loc[
            feature_dictionary["COLUMN_NAME"] == col, field.upper()
        ] = value

In [179]:
print(
    feature_dictionary[
        feature_dictionary["COLUMN_ROLE"] == "FORECAST_KEY"
    ].to_string(index=False)
)

COLUMN_NAME DATA_TYPE  MISSING_COUNT  UNIQUE_VALUES  COLUMN_ROLE                    SOURCE                                                      DESCRIPTION       FORECAST_AVAILABILITY                      MISSING_VALUE_HANDLING                                                                      LEAKAGE_RULE
   FIELD_ID    object              0              5 FORECAST_KEY      FIELD_SEASON_CONTEXT                 Unique field identifier for the forecast record. Available at forecast time. Required; missing values are not permitted.                           Identifier only; contains no future target information.
SEASON_YEAR     int64              0              4 FORECAST_KEY      FIELD_SEASON_CONTEXT                      Crop season year associated with the field. Available at forecast time. Required; missing values are not permitted.                                                           Season identifier only.
 WEEK_INDEX     int64              0              6 FORECAST_KEY SEASO

In [180]:
target_documentation = {
    "ACTUAL_YIELD_BU_ACRE": {
        "SOURCE": "CLEANED_HISTORICAL_YIELD",
        "DESCRIPTION": "Actual harvested crop yield in bushels per acre used as the supervised learning target.",
        "FORECAST_AVAILABILITY": "Historical target value associated with the completed field-season; not used as an input feature at forecast time.",
        "MISSING_VALUE_HANDLING": "Must be available and valid for the training target. Unresolved or conflicting historical yield records are excluded from definitive target use.",
        "LEAKAGE_RULE": "Target only; must never be included as a model input feature."
    },
    "ACTUAL_YIELD_NUMERIC": {
        "SOURCE": "CLEANED_HISTORICAL_YIELD",
        "DESCRIPTION": "Numeric representation of ACTUAL_YIELD_BU_ACRE used for supervised model training and evaluation.",
        "FORECAST_AVAILABILITY": "Available only as the historical training/evaluation target; not available for an active future forecast.",
        "MISSING_VALUE_HANDLING": "Converted using numeric parsing; invalid target values become missing and are not used as valid training targets.",
        "LEAKAGE_RULE": "Target only; must never be included as a model input feature."
    },
    "YIELD_TARGET_STATUS": {
        "SOURCE": "CLEANED_HISTORICAL_YIELD",
        "DESCRIPTION": "Data-quality status indicating whether the historical yield record is valid for use as a definitive target.",
        "FORECAST_AVAILABILITY": "Used for target validation and training-data filtering.",
        "MISSING_VALUE_HANDLING": "Records without a valid target status are not accepted as definitive training targets.",
        "LEAKAGE_RULE": "Target-quality metadata only; must not be used as a predictive feature."
    }
}

for col, docs in target_documentation.items():
    for field, value in docs.items():
        feature_dictionary.loc[
            feature_dictionary["COLUMN_NAME"] == col,
            field
        ] = value

In [181]:
print("=== TARGET DOCUMENTATION ===")

print(
    feature_dictionary[
        feature_dictionary["COLUMN_ROLE"] == "TARGET"
    ].to_string(index=False)
)

=== TARGET DOCUMENTATION ===
         COLUMN_NAME DATA_TYPE  MISSING_COUNT  UNIQUE_VALUES COLUMN_ROLE                   SOURCE                                                                                                 DESCRIPTION                                                                                              FORECAST_AVAILABILITY                                                                                                                           MISSING_VALUE_HANDLING                                                            LEAKAGE_RULE
ACTUAL_YIELD_BU_ACRE    object              0             11      TARGET CLEANED_HISTORICAL_YIELD                     Actual harvested crop yield in bushels per acre used as the supervised learning target. Historical target value associated with the completed field-season; not used as an input feature at forecast time. Must be available and valid for the training target. Unresolved or conflicting historical yield records are excl

In [182]:
print("=== FINAL FEATURE DICTIONARY ===")
display(feature_dictionary)

=== FINAL FEATURE DICTIONARY ===


,COLUMN_NAME,DATA_TYPE,MISSING_COUNT,UNIQUE_VALUES,COLUMN_ROLE,SOURCE,DESCRIPTION,FORECAST_AVAILABILITY,MISSING_VALUE_HANDLING,LEAKAGE_RULE
0,FIELD_ID,object,0,5,FORECAST_KEY,FIELD_SEASON_CONTEXT,Unique field identifier for the forecast record.,Available at forecast time.,Required; missing values are not permitted.,Identifier only; contains no future target inf...
1,SEASON_YEAR,int64,0,4,FORECAST_KEY,FIELD_SEASON_CONTEXT,Crop season year associated with the field.,Available at forecast time.,Required; missing values are not permitted.,Season identifier only.
2,WEEK_INDEX,int64,0,6,FORECAST_KEY,SEASON_INTELLIGENCE_INPUT,Canonical seasonal week index used for forecas...,Available at forecast time.,Required; missing values are not permitted.,Derived from the established seasonal timeline...
3,SEASON_DAY,int64,0,42,FORECAST_KEY,SEASON_INTELLIGENCE_INPUT,Canonical seasonal day corresponding to the fo...,Available at forecast time.,Required; missing values are not permitted.,Derived from the seasonal timeline only.
4,MEAN_TEMPERATURE,float64,5,81,MODEL_CANDIDATE,,,,,
5,TEMPERATURE_7D,float64,5,81,MODEL_CANDIDATE,,,,,
6,TEMPERATURE_14D,float64,5,96,MODEL_CANDIDATE,,,,,
7,TEMPERATURE_DEVIATION,float64,5,90,MODEL_CANDIDATE,,,,,
8,RAINFALL_7D,float64,5,80,MODEL_CANDIDATE,,,,,
9,RAINFALL_14D,float64,5,92,MODEL_CANDIDATE,,,,,


In [183]:
feature_dictionary.to_csv(
    "YIELD_FORECAST_FEATURE_DICTIONARY.csv",
    index=False
)

print("Feature Dictionary saved successfully.")

Feature Dictionary saved successfully.


In [184]:
data_quality_report = {
    "TOTAL_ROWS": len(yield_forecast_input),
    "TOTAL_COLUMNS": len(yield_forecast_input.columns),
    "DUPLICATE_FORECAST_GRAIN_ROWS": (
        yield_forecast_input.duplicated(
            subset=["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
        ).sum()
    ),
    "MISSING_TARGET_ROWS": (
        yield_forecast_input["ACTUAL_YIELD_NUMERIC"].isna().sum()
    ),
    "VALID_TARGET_ROWS": (
        yield_forecast_input["YIELD_TARGET_STATUS"].eq("VALID").sum()
    ),
    "FUTURE_RISK_ROWS": (
        yield_forecast_input["RISK_SOURCE_SEASON_DAY"]
        > yield_forecast_input["SEASON_DAY"]
    ).sum(),
    "NEGATIVE_RISK_LAG_ROWS": (
        yield_forecast_input["RISK_LAG_DAYS"] < 0
    ).sum(),
    "INVALID_WEATHER_ALIGNMENT_ROWS": (
        ~yield_forecast_input["WEATHER_ALIGNMENT_STATUS"].isin(
            ["AS_OF_ALIGNED", "NO_PRIOR_WEATHER"]
        )
    ).sum(),
    "NEGATIVE_NDVI_LAG_ROWS": (
        yield_forecast_input["NDVI_LAG_DAYS"].notna()
        & (yield_forecast_input["NDVI_LAG_DAYS"] < 0)
    ).sum(),
    "NO_PRIOR_WEATHER_ROWS": (
        yield_forecast_input["WEATHER_ALIGNMENT_STATUS"]
        == "NO_PRIOR_WEATHER"
    ).sum(),
    "NO_PRIOR_NDVI_ROWS": (
        yield_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "NO_PRIOR_NDVI"
    ).sum(),
    "INVALID_NDVI_SOURCE_ROWS": (
        yield_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "INVALID_SOURCE_NDVI"
    ).sum(),
    "STALE_NDVI_ROWS": (
        yield_forecast_input["NDVI_FRESHNESS_STATUS"]
        == "VALID_STALE_NDVI"
    ).sum()
}

data_quality_report_df = pd.DataFrame(
    list(data_quality_report.items()),
    columns=["CHECK", "VALUE"]
)

display(data_quality_report_df)

,CHECK,VALUE
0,TOTAL_ROWS,111
1,TOTAL_COLUMNS,77
2,DUPLICATE_FORECAST_GRAIN_ROWS,0
3,MISSING_TARGET_ROWS,0
4,VALID_TARGET_ROWS,111
5,FUTURE_RISK_ROWS,0
6,NEGATIVE_RISK_LAG_ROWS,0
7,INVALID_WEATHER_ALIGNMENT_ROWS,0
8,NEGATIVE_NDVI_LAG_ROWS,0
9,NO_PRIOR_WEATHER_ROWS,5


In [185]:
final_output_path = "YIELD_FORECAST_INPUT.csv"

yield_forecast_input.to_csv(
    final_output_path,
    index=False
)

print("Saved:", final_output_path)
print("Shape:", yield_forecast_input.shape)

Saved: YIELD_FORECAST_INPUT.csv
Shape: (111, 77)


In [186]:
check_df = pd.read_csv(final_output_path)

print("Reloaded shape:", check_df.shape)
print(
    "Duplicate forecast grains:",
    check_df.duplicated(
        subset=["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
    ).sum()
)

Reloaded shape: (111, 77)
Duplicate forecast grains: 0


In [187]:
from snowflake.connector.pandas_tools import write_pandas

success, num_chunks, num_rows, output = write_pandas(
    conn=conn,
    df=yield_forecast_input,
    table_name="YIELD_FORECAST_INPUT",
    database="YIELD_FORECAST_DB",
    schema="POD_VERTEX",
    auto_create_table=False,
    overwrite=True
)

print("Upload successful:", success)
print("Rows uploaded:", num_rows)
print("Chunks:", num_chunks)

Upload successful: True
Rows uploaded: 111
Chunks: 1


In [188]:
verify_sql = """
SELECT COUNT(*) AS ROW_COUNT
FROM YIELD_FORECAST_DB.POD_VERTEX.YIELD_FORECAST_INPUT
"""

verification = pd.read_sql(verify_sql, conn)

print(verification)

   ROW_COUNT
0        111


/tmp/ipykernel_75164/1413266135.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  verification = pd.read_sql(verify_sql, conn)
